In [ ]:
# ============================================================
# SETUP — 09.01 Baseline-v7 Graph Schema / Topology Audit
# Purpose:
#   Initialize Stage 09 using the validated IndexedRXGraph
#   loader and inspect the real node/edge/topology API before
#   implementing one-frame gap candidate detection.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: YES — Stage-09 graph schema evidence
# ============================================================

from pathlib import Path
import hashlib
import inspect
import sys

import numpy as np
import pandas as pd
import tracksdata as td


# ============================================================
# Project
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PRED_ROOT = (
    PROJECT
    / "predictions"
)

CV_PATH = (
    PROJECT
    / "configs/cv_manifest.csv"
)

WARM2_CKPT = (
    PROJECT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)


# ============================================================
# Frozen baseline-v7 identity
# ============================================================

EXPECTED_WARM2_SHA = (
    "31d209bfde1fd5bbf3b2a661cdb4a25b"
    "b72922b832d315ee5a75cc32f44656c1"
)


def sha256(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


assert WARM2_CKPT.exists()
assert (
    sha256(WARM2_CKPT)
    == EXPECTED_WARM2_SHA
)


# ============================================================
# Validated graph loader
# Exact implementation recovered from Stage 07
# ============================================================

def load_graph(path):
    loaded = (
        td.graph.IndexedRXGraph
        .from_geff(path)
    )

    return (
        loaded[0]
        if isinstance(
            loaded,
            tuple,
        )
        else loaded
    )


# ============================================================
# Fold0 metadata
# ============================================================

cv_meta = pd.read_csv(
    CV_PATH
)

fold0 = (
    cv_meta[
        cv_meta["fold"] == 0
    ]
    .copy()
)

assert len(fold0) == 40

dataset_names = sorted(
    fold0["dataset"]
)

prefix_map = dict(
    zip(
        fold0["dataset"],
        fold0["prefix"],
    )
)


# ============================================================
# Locate Warm2 det=.995
# ============================================================

METHOD = (
    "official_baseline_fold0_"
    "warm2_lr1e5_det0995"
)

candidates = []

for split_dir in (
    PRED_ROOT.rglob("split_0")
):

    if (
        split_dir.parent.name
        != METHOD
    ):
        continue

    geffs = list(
        split_dir.glob("*.geff")
    )

    if len(geffs) == 40:
        candidates.append(
            split_dir
        )

assert len(candidates) == 1, (
    candidates
)

WARM2_0995 = candidates[0]


# ============================================================
# Representative sample from each domain
# ============================================================

sample_44b6 = next(
    x
    for x in dataset_names
    if prefix_map[x] == "44b6"
)

sample_6bba = next(
    x
    for x in dataset_names
    if prefix_map[x] == "6bba"
)

audit_samples = [
    sample_44b6,
    sample_6bba,
]


# ============================================================
# Safe introspection helpers
# ============================================================

KEYWORDS = (
    "node",
    "edge",
    "neighbor",
    "success",
    "predecess",
    "parent",
    "child",
    "degree",
    "attr",
    "time",
)


def relevant_api_names(obj):

    names = []

    for name in dir(obj):

        if name.startswith("_"):
            continue

        if any(
            token in name.lower()
            for token in KEYWORDS
        ):
            names.append(name)

    return sorted(names)


def safe_signature(obj):

    try:
        return str(
            inspect.signature(obj)
        )

    except Exception:
        return "<signature unavailable>"


def describe_attribute(
    obj,
    name,
):

    try:
        value = getattr(
            obj,
            name,
        )

    except Exception as e:
        return (
            name,
            "ATTRIBUTE ERROR",
            repr(e),
        )

    if callable(value):

        return (
            name,
            "callable",
            safe_signature(value),
        )

    try:
        preview = repr(value)

        if len(preview) > 250:
            preview = (
                preview[:250]
                + " ..."
            )

    except Exception:
        preview = (
            "<repr unavailable>"
        )

    return (
        name,
        type(value).__name__,
        preview,
    )


# ============================================================
# Audit
# ============================================================

summary_rows = []

for name in audit_samples:

    path = (
        WARM2_0995
        / f"{name}.geff"
    )

    g = load_graph(
        path
    )

    print()
    print("=" * 72)

    print(
        f"DATASET: {name}"
        f" | prefix={prefix_map[name]}"
    )

    print("=" * 72)

    print(
        "Graph type:",
        type(g),
    )

    # --------------------------------------------------------
    # Generic representation
    # --------------------------------------------------------

    try:
        graph_len = len(g)
    except Exception:
        graph_len = None

    print(
        "len(graph):",
        graph_len,
    )

    try:
        graph_repr = repr(g)
    except Exception:
        graph_repr = (
            "<repr unavailable>"
        )

    print(
        "repr(graph):",
        graph_repr[:500],
    )

    # --------------------------------------------------------
    # Relevant public API
    # --------------------------------------------------------

    api_names = (
        relevant_api_names(g)
    )

    print()
    print(
        "Relevant public API names:"
    )

    for api_name in api_names:

        name_out, kind, detail = (
            describe_attribute(
                g,
                api_name,
            )
        )

        print(
            f"  {name_out:<30}"
            f" | {kind:<15}"
            f" | {detail}"
        )

    # --------------------------------------------------------
    # Also inspect wrapped rustworkx graph if exposed.
    # Do not assume its attribute name.
    # --------------------------------------------------------

    rx_candidates = []

    for attr_name in dir(g):

        if attr_name.startswith("__"):
            continue

        if not any(
            token in attr_name.lower()
            for token in (
                "graph",
                "rx",
            )
        ):
            continue

        try:
            value = getattr(
                g,
                attr_name,
            )
        except Exception:
            continue

        type_name = (
            type(value).__module__
            + "."
            + type(value).__name__
        )

        if (
            "rustworkx"
            in type_name.lower()
        ):
            rx_candidates.append(
                (
                    attr_name,
                    value,
                    type_name,
                )
            )

    print()
    print(
        "Embedded rustworkx-like objects:",
        len(rx_candidates),
    )

    for (
        attr_name,
        rx_obj,
        type_name,
    ) in rx_candidates:

        print(
            f"  {attr_name}: "
            f"{type_name}"
        )

        rx_api = [
            x
            for x in dir(rx_obj)
            if (
                not x.startswith("_")
                and any(
                    token in x.lower()
                    for token in KEYWORDS
                )
            )
        ]

        print(
            "  relevant API:",
            sorted(rx_api)[:80],
        )

    summary_rows.append({
        "dataset":
            name,

        "prefix":
            prefix_map[name],

        "graph_type":
            (
                type(g).__module__
                + "."
                + type(g).__name__
            ),

        "len_graph":
            graph_len,

        "relevant_api_count":
            len(api_names),

        "embedded_rx_objects":
            len(rx_candidates),
    })


# ============================================================
# Result
# ============================================================

audit_0901 = pd.DataFrame(
    summary_rows
)

print()
print("=" * 72)
print("09.01 RESULT")
print("=" * 72)

display(
    audit_0901
)

print()
print(
    "Warm2 SHA256:",
    sha256(WARM2_CKPT),
)

print(
    "Prediction root:",
    WARM2_0995,
)

print()
print(
    "09.01 graph schema / topology audit: COMPLETE"
)

In [ ]:
# ============================================================
# SETUP — 09.02 Node Attribute / Degree Semantics Audit
# Purpose:
#   Confirm the exact node attribute schema and bulk degree
#   semantics of IndexedRXGraph before implementing the
#   one-frame gap candidate detector.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: YES — Stage-09 graph API contract
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import tracksdata as td


# ============================================================
# Preconditions from 09.01
# ============================================================

assert "PROJECT" in globals()
assert "WARM2_0995" in globals()
assert "dataset_names" in globals()
assert "prefix_map" in globals()
assert "load_graph" in globals()


# ============================================================
# Representative sample
# ============================================================

name = next(
    x
    for x in dataset_names
    if prefix_map[x] == "44b6"
)

g = load_graph(
    WARM2_0995 / f"{name}.geff"
)

print("=== 09.02 NODE ATTRIBUTE / DEGREE AUDIT ===")
print()
print("Dataset:", name)
print("Prefix:", prefix_map[name])
print("Graph type:", type(g))
print("num_nodes():", g.num_nodes())
print("num_edges():", g.num_edges())


# ============================================================
# Attribute schema
# ============================================================

node_keys = g.node_attr_keys()

print()
print("Node attribute keys:")
print(node_keys)

attrs = g.node_attrs(
    unpack=True
)

print()
print("node_attrs(unpack=True) type:")
print(type(attrs))

print()
print("Columns:")
print(attrs.columns)

print()
print("Shape:")
print(attrs.shape)

print()
print("First 5 rows:")
print(attrs.head(5))


# ============================================================
# Node IDs
# ============================================================

node_ids = g.node_ids()

print()
print("node_ids type:", type(node_ids))
print("node_ids count:", len(node_ids))
print("First 10 node_ids:", node_ids[:10])

assert len(node_ids) == g.num_nodes()
assert attrs.height == g.num_nodes()


# ============================================================
# Degree semantics
# ============================================================

probe_ids = node_ids[:10]

in_deg = g.in_degree(
    probe_ids
)

out_deg = g.out_degree(
    probe_ids
)

print()
print("Probe node IDs:")
print(probe_ids)

print("in_degree(probe_ids):")
print(in_deg)

print("out_degree(probe_ids):")
print(out_deg)

assert len(in_deg) == len(probe_ids)
assert len(out_deg) == len(probe_ids)


# ============================================================
# Successor / predecessor semantics
# ============================================================

probe = probe_ids[0]

pred = g.predecessors(
    probe
)

succ = g.successors(
    probe
)

print()
print("Single-node topology probe:")
print("node_id:", probe)
print("in_degree:", g.in_degree(probe))
print("out_degree:", g.out_degree(probe))
print("predecessors:", pred)
print("successors:", succ)

assert len(pred) == g.in_degree(probe)
assert len(succ) == g.out_degree(probe)


# ============================================================
# Time structure
# ============================================================

times = g.time_points()

print()
print("time_points():")
print(times)

print(
    "Time range:",
    min(times),
    "→",
    max(times),
)

print(
    "Number of time points:",
    len(times),
)


# ============================================================
# Required coordinate schema gate
# ============================================================

required = {
    "t",
    "x",
    "y",
    "z",
}

available = set(
    str(x)
    for x in attrs.columns
)

missing = (
    required - available
)

print()
print("Required coordinate fields:", sorted(required))
print("Missing:", sorted(missing))

assert not missing, (
    f"Missing required node fields: {missing}"
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.02 RESULT")
print("=" * 72)

print(
    "Node schema:",
    list(attrs.columns),
)

print(
    "Nodes:",
    g.num_nodes(),
)

print(
    "Edges:",
    g.num_edges(),
)

print(
    "Frames:",
    len(times),
)

print(
    "Degree API:",
    "VERIFIED",
)

print(
    "Predecessor/successor API:",
    "VERIFIED",
)

print(
    "Coordinate schema:",
    "VERIFIED",
)

print()
print(
    "09.02 node attribute / degree audit: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.03 Existing-Node Single-Edge Gap Census
# Purpose:
#   Measure how many plausible track breaks can potentially be
#   repaired by adding exactly ONE edge between EXISTING nodes.
#
#   Candidate types:
#
#   LEFT_MISSING:
#       A(t)    B(t+1) -> C(t+2)
#         X---->
#
#   RIGHT_MISSING:
#       A(t) -> B(t+1)    C(t+2)
#                         ^
#                         X
#
#   Candidate geometry is evaluated under a constant-velocity
#   continuation model, but NO edge is added in this Cell.
#
# Pipeline position:
#   Warm2 det=.995
#     -> G2 p2=.80
#     -> confidence pruning K=9
#     -> [THIS GAP AUDIT]
#     -> frozen cleanup later
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: YES — Stage-09 candidate census evidence
# ============================================================

from pathlib import Path
import ast
import json
import sys

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from collections import defaultdict

# ============================================================
# Preconditions
# ============================================================

assert "PROJECT" in globals()
assert "WARM2_0995" in globals()
assert "dataset_names" in globals()
assert "prefix_map" in globals()
assert "load_graph" in globals()


# ============================================================
# Official source import
# ============================================================

OFFICIAL_SRC = (
    PROJECT
    / "external/official/src"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

for p in [
    OFFICIAL_SRC,
    OFFICIAL_SCRIPTS,
]:
    if str(p) not in sys.path:
        sys.path.insert(
            0,
            str(p),
        )

from tracking_cellmot.io import open_dataset


TRAIN_DIR = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)


# ============================================================
# Locate Warm2 det=.9975 high-confidence prediction
# ============================================================

METHOD_09975 = (
    "official_baseline_fold0_"
    "warm2_lr1e5_det09975"
)

high_candidates = []

for split_dir in (
    PRED_ROOT.rglob("split_0")
):
    if (
        split_dir.parent.name
        != METHOD_09975
    ):
        continue

    if len(
        list(
            split_dir.glob("*.geff")
        )
    ) == 40:
        high_candidates.append(
            split_dir
        )

assert len(high_candidates) == 1, (
    high_candidates
)

WARM2_09975 = (
    high_candidates[0]
)


# ============================================================
# Recover exact validated Stage-07 post-processing helpers
# ============================================================

NOTEBOOK_07 = (
    PROJECT
    / "notebooks/07_detection_threshold_sweep.ipynb"
)

with NOTEBOOK_07.open(
    "r",
    encoding="utf-8",
) as f:
    nb07 = json.load(f)


def import_functions_from_cell(
    cell_index,
    names,
):
    source = "".join(
        nb07["cells"][cell_index]["source"]
    )

    tree = ast.parse(source)

    selected = [
        node
        for node in tree.body
        if (
            isinstance(
                node,
                ast.FunctionDef,
            )
            and node.name in names
        )
    ]

    found = {
        node.name
        for node in selected
    }

    missing = (
        set(names) - found
    )

    assert not missing, (
        f"Missing functions in Stage-07 "
        f"Cell {cell_index}: {missing}"
    )

    module = ast.Module(
        body=selected,
        type_ignores=[],
    )

    ast.fix_missing_locations(
        module
    )

    exec(
        compile(
            module,
            filename=(
                f"07_detection_threshold_sweep.ipynb:"
                f"cell_{cell_index}"
            ),
            mode="exec",
        ),
        globals(),
    )


import_functions_from_cell(
    0,
    [
        "apply_frozen_g2",
    ],
)

import_functions_from_cell(
    3,
    [
        "component_sets",
    ],
)

import_functions_from_cell(
    16,
    [
        "node_coordinate_keys",
    ],
)

import_functions_from_cell(
    19,
    [
        "apply_low_conf_component_pruning",
    ],
)


# ============================================================
# Freeze baseline-v7 policy
# ============================================================

G2_P2_MIN = 0.80

G2_DAUGHTER_SEP_MIN_UM = 6.0
G2_ANGLE_MIN_DEG = 120.0
G2_MAX_PARENT_DAUGHTER_UM = 8.0
G2_DISTANCE_BALANCE_MAX = 0.50

CONF_COMPONENT_LIMIT = 9


# ============================================================
# Geometry helpers
# ============================================================

def turn_angle_deg(
    a,
    b,
    c,
):
    """
    0 degrees = perfectly straight continuation:
        A -> B -> C
    """

    v1 = b - a
    v2 = c - b

    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)

    if (
        n1 <= 1e-12
        or n2 <= 1e-12
    ):
        return np.nan

    cosine = np.dot(
        v1,
        v2,
    ) / (n1 * n2)

    cosine = np.clip(
        cosine,
        -1.0,
        1.0,
    )

    return float(
        np.degrees(
            np.arccos(cosine)
        )
    )


def step_balance(
    d1,
    d2,
):
    denom = max(
        d1,
        d2,
        1e-12,
    )

    return abs(
        d1 - d2
    ) / denom


# ============================================================
# Candidate extraction for one graph
# ============================================================

def census_single_edge_gaps(
    name,
):

    # --------------------------------------------------------
    # Load main + high-confidence graphs
    # --------------------------------------------------------

    g = load_graph(
        WARM2_0995
        / f"{name}.geff"
    )

    high_g = load_graph(
        WARM2_09975
        / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    # Expected physical-axis order is z, y, x.
    assert len(scale) == 3

    # --------------------------------------------------------
    # Baseline-v7 BEFORE final cleanup
    # --------------------------------------------------------

    apply_frozen_g2(
        g,
        scale,
    )

    high_keys = (
        node_coordinate_keys(
            high_g
        )
    )

    apply_low_conf_component_pruning(
        g,
        high_keys,
        CONF_COMPONENT_LIMIT,
    )

    # --------------------------------------------------------
    # Node table after G2 + K9
    # --------------------------------------------------------

    attrs = g.node_attrs(
        unpack=True
    )

    node_ids = (
        attrs["node_id"]
        .to_list()
    )

    t_arr = (
        attrs["t"]
        .to_numpy()
        .astype(int)
    )

    z_arr = (
        attrs["z"]
        .to_numpy()
        .astype(float)
    )

    y_arr = (
        attrs["y"]
        .to_numpy()
        .astype(float)
    )

    x_arr = (
        attrs["x"]
        .to_numpy()
        .astype(float)
    )

    # Physical coordinates in microns: z, y, x.
    pos = np.column_stack([
        z_arr * scale[0],
        y_arr * scale[1],
        x_arr * scale[2],
    ])

    id_to_i = {
        node_id: i
        for i, node_id
        in enumerate(node_ids)
    }

    in_deg = np.asarray(
        g.in_degree(
            node_ids
        ),
        dtype=int,
    )

    out_deg = np.asarray(
        g.out_degree(
            node_ids
        ),
        dtype=int,
    )

    # --------------------------------------------------------
    # Frame-indexed track endpoints / starts
    # --------------------------------------------------------

    ends_by_t = {}
    starts_by_t = {}

    for t in np.unique(
        t_arr
    ):

        frame_idx = np.flatnonzero(
            t_arr == t
        )

        ends = frame_idx[
            out_deg[frame_idx] == 0
        ]

        starts = frame_idx[
            in_deg[frame_idx] == 0
        ]

        ends_by_t[int(t)] = ends
        starts_by_t[int(t)] = starts

    # --------------------------------------------------------
    # KD trees
    # --------------------------------------------------------

    end_trees = {}
    start_trees = {}

    for t, idx in ends_by_t.items():

        if len(idx):
            end_trees[t] = (
                cKDTree(
                    pos[idx]
                ),
                idx,
            )

    for t, idx in starts_by_t.items():

        if len(idx):
            start_trees[t] = (
                cKDTree(
                    pos[idx]
                ),
                idx,
            )

    rows = []

    # ========================================================
    # TYPE L
    #
    # A(t)    B(t+1) -> C(t+2)
    #   X---->
    #
    # B is a track start with exactly one successor.
    # Predict A using constant velocity:
    #
    #   A_expected = 2B - C
    # ========================================================

    left_middle_idx = np.flatnonzero(
        (in_deg == 0)
        & (out_deg == 1)
        & (t_arr > t_arr.min())
    )

    for bi in left_middle_idx:

        b_id = node_ids[bi]
        tb = int(
            t_arr[bi]
        )

        succ = g.successors(
            b_id
        )

        if len(succ) != 1:
            continue

        c_id = succ[0]

        if c_id not in id_to_i:
            continue

        ci = id_to_i[
            c_id
        ]

        # Existing edge must be exactly B(t+1) -> C(t+2).
        if int(
            t_arr[ci]
        ) != tb + 1:
            continue

        candidate_frame = (
            tb - 1
        )

        if (
            candidate_frame
            not in end_trees
        ):
            continue

        tree, endpoint_idx = (
            end_trees[
                candidate_frame
            ]
        )

        expected_a = (
            2.0 * pos[bi]
            - pos[ci]
        )

        k = min(
            2,
            len(endpoint_idx),
        )

        distances, locs = tree.query(
            expected_a,
            k=k,
        )

        distances = np.atleast_1d(
            distances
        )

        locs = np.atleast_1d(
            locs
        )

        ai = endpoint_idx[
            int(locs[0])
        ]

        a_id = node_ids[
            ai
        ]

        # By construction B has no predecessor.
        assert not g.has_edge(
            a_id,
            b_id,
        )

        second = (
            float(distances[1])
            if len(distances) > 1
            else np.inf
        )

        d_ab = float(
            np.linalg.norm(
                pos[bi]
                - pos[ai]
            )
        )

        d_bc = float(
            np.linalg.norm(
                pos[ci]
                - pos[bi]
            )
        )

        rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "type":
                "LEFT_MISSING",

            "a_id":
                a_id,

            "b_id":
                b_id,

            "c_id":
                c_id,

            "t_mid":
                tb,

            "prediction_residual_um":
                float(
                    distances[0]
                ),

            "second_residual_um":
                second,

            "ambiguity_margin_um":
                (
                    second
                    - float(
                        distances[0]
                    )
                ),

            "d_ab_um":
                d_ab,

            "d_bc_um":
                d_bc,

            "max_step_um":
                max(
                    d_ab,
                    d_bc,
                ),

            "step_balance":
                step_balance(
                    d_ab,
                    d_bc,
                ),

            "turn_angle_deg":
                turn_angle_deg(
                    pos[ai],
                    pos[bi],
                    pos[ci],
                ),
        })

    # ========================================================
    # TYPE R
    #
    # A(t) -> B(t+1)    C(t+2)
    #                    ^
    #                    X
    #
    # B is a track end with exactly one predecessor.
    # Predict C:
    #
    #   C_expected = 2B - A
    # ========================================================

    right_middle_idx = np.flatnonzero(
        (in_deg == 1)
        & (out_deg == 0)
        & (t_arr < t_arr.max())
    )

    for bi in right_middle_idx:

        b_id = node_ids[bi]
        tb = int(
            t_arr[bi]
        )

        pred = g.predecessors(
            b_id
        )

        if len(pred) != 1:
            continue

        a_id = pred[0]

        if a_id not in id_to_i:
            continue

        ai = id_to_i[
            a_id
        ]

        if int(
            t_arr[ai]
        ) != tb - 1:
            continue

        candidate_frame = (
            tb + 1
        )

        if (
            candidate_frame
            not in start_trees
        ):
            continue

        tree, start_idx = (
            start_trees[
                candidate_frame
            ]
        )

        expected_c = (
            2.0 * pos[bi]
            - pos[ai]
        )

        k = min(
            2,
            len(start_idx),
        )

        distances, locs = tree.query(
            expected_c,
            k=k,
        )

        distances = np.atleast_1d(
            distances
        )

        locs = np.atleast_1d(
            locs
        )

        ci = start_idx[
            int(locs[0])
        ]

        c_id = node_ids[
            ci
        ]

        assert not g.has_edge(
            b_id,
            c_id,
        )

        second = (
            float(distances[1])
            if len(distances) > 1
            else np.inf
        )

        d_ab = float(
            np.linalg.norm(
                pos[bi]
                - pos[ai]
            )
        )

        d_bc = float(
            np.linalg.norm(
                pos[ci]
                - pos[bi]
            )
        )

        rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "type":
                "RIGHT_MISSING",

            "a_id":
                a_id,

            "b_id":
                b_id,

            "c_id":
                c_id,

            "t_mid":
                tb,

            "prediction_residual_um":
                float(
                    distances[0]
                ),

            "second_residual_um":
                second,

            "ambiguity_margin_um":
                (
                    second
                    - float(
                        distances[0]
                    )
                ),

            "d_ab_um":
                d_ab,

            "d_bc_um":
                d_bc,

            "max_step_um":
                max(
                    d_ab,
                    d_bc,
                ),

            "step_balance":
                step_balance(
                    d_ab,
                    d_bc,
                ),

            "turn_angle_deg":
                turn_angle_deg(
                    pos[ai],
                    pos[bi],
                    pos[ci],
                ),
        })

    return pd.DataFrame(
        rows
    )


# ============================================================
# Full Fold0 census
# ============================================================

all_rows = []

print(
    "=== 09.03 EXISTING-NODE SINGLE-EDGE GAP CENSUS ==="
)

for i, name in enumerate(
    dataset_names,
    start=1,
):

    df = census_single_edge_gaps(
        name
    )

    if len(df):
        all_rows.append(
            df
        )

    # Concise output only.
    if (
        i == 1
        or i % 10 == 0
        or i == len(dataset_names)
    ):
        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


assert all_rows

gap_candidates_0903 = (
    pd.concat(
        all_rows,
        ignore_index=True,
    )
)


# ============================================================
# Opportunity counts
# ============================================================

counts_0903 = (
    gap_candidates_0903
    .groupby(
        [
            "prefix",
            "type",
        ]
    )
    .size()
    .reset_index(
        name="candidate_opportunities"
    )
)


# ============================================================
# Geometry quantiles
# ============================================================

quantiles_0903 = (
    gap_candidates_0903
    .groupby(
        [
            "prefix",
            "type",
        ]
    )
    [
        [
            "prediction_residual_um",
            "max_step_um",
            "step_balance",
            "turn_angle_deg",
            "ambiguity_margin_um",
        ]
    ]
    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .round(4)
)


# ============================================================
# Residual-threshold census
#
# IMPORTANT:
# These are diagnostic thresholds only.
# Nothing is accepted/repaired here.
# ============================================================

threshold_rows = []

for prefix in [
    "44b6",
    "6bba",
]:

    for gap_type in [
        "LEFT_MISSING",
        "RIGHT_MISSING",
    ]:

        sub = (
            gap_candidates_0903[
                (
                    gap_candidates_0903[
                        "prefix"
                    ] == prefix
                )
                &
                (
                    gap_candidates_0903[
                        "type"
                    ] == gap_type
                )
            ]
        )

        if not len(sub):
            continue

        for radius in [
            1.0,
            2.0,
            3.0,
            4.0,
            5.0,
            7.0,
        ]:

            q = sub[
                sub[
                    "prediction_residual_um"
                ] <= radius
            ]

            threshold_rows.append({
                "prefix":
                    prefix,

                "type":
                    gap_type,

                "residual_radius_um":
                    radius,

                "n":
                    len(q),

                "fraction_of_opportunities":
                    len(q) / len(sub),

                "angle_le_30_rate":
                    (
                        float(
                            (
                                q[
                                    "turn_angle_deg"
                                ] <= 30
                            ).mean()
                        )
                        if len(q)
                        else np.nan
                    ),

                "angle_le_60_rate":
                    (
                        float(
                            (
                                q[
                                    "turn_angle_deg"
                                ] <= 60
                            ).mean()
                        )
                        if len(q)
                        else np.nan
                    ),

                "balance_le_0_5_rate":
                    (
                        float(
                            (
                                q[
                                    "step_balance"
                                ] <= 0.50
                            ).mean()
                        )
                        if len(q)
                        else np.nan
                    ),

                "max_step_le_8_rate":
                    (
                        float(
                            (
                                q[
                                    "max_step_um"
                                ] <= 8.0
                            ).mean()
                        )
                        if len(q)
                        else np.nan
                    ),

                "ambiguity_margin_ge_1_rate":
                    (
                        float(
                            (
                                q[
                                    "ambiguity_margin_um"
                                ] >= 1.0
                            ).mean()
                        )
                        if len(q)
                        else np.nan
                    ),
            })


threshold_census_0903 = (
    pd.DataFrame(
        threshold_rows
    )
)


# ============================================================
# Very conservative geometry VIEW
#
# Still NOT an acceptance policy.
# It only tells us whether a meaningful low-ambiguity population
# exists before touching GT or modifying graphs.
# ============================================================

conservative_view_0903 = (
    gap_candidates_0903[
        (
            gap_candidates_0903[
                "prediction_residual_um"
            ] <= 3.0
        )
        &
        (
            gap_candidates_0903[
                "max_step_um"
            ] <= 8.0
        )
        &
        (
            gap_candidates_0903[
                "step_balance"
            ] <= 0.50
        )
        &
        (
            gap_candidates_0903[
                "turn_angle_deg"
            ] <= 60.0
        )
        &
        (
            gap_candidates_0903[
                "ambiguity_margin_um"
            ] >= 1.0
        )
    ]
)


conservative_counts_0903 = (
    conservative_view_0903
    .groupby(
        [
            "prefix",
            "type",
        ]
    )
    .size()
    .reset_index(
        name="n_conservative_geometry"
    )
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.03 RESULT")
print("=" * 72)

print()
print("CANDIDATE OPPORTUNITIES")
display(
    counts_0903
)

print()
print("GEOMETRY QUANTILES")
display(
    quantiles_0903
)

print()
print("RESIDUAL-THRESHOLD CENSUS")
display(
    threshold_census_0903
)

print()
print(
    "CONSERVATIVE GEOMETRY VIEW "
    "(diagnostic only)"
)
display(
    conservative_counts_0903
)

print()
print(
    "Total opportunities:",
    len(
        gap_candidates_0903
    ),
)

print(
    "Conservative-view candidates:",
    len(
        conservative_view_0903
    ),
)

print()
print(
    "No graph was modified."
)

print(
    "No GT was used."
)

print()
print(
    "09.03 existing-node gap census: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.05 Existing-Node Gap Oracle Audit
# Purpose:
#   Use the locked official metric's prediction->GT node
#   matching to measure whether Stage-09 proposed missing edges
#   correspond to real observable GT edges.
#
#   IMPORTANT:
#   GT matching is used ONLY for validation/oracle analysis.
#   No GT-derived field will be used as a submission feature.
#
#   Primary scopes:
#     ALL
#     CONSERVATIVE
#     MUTUAL
#     MUTUAL_CONSERVATIVE
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 oracle analysis only
# Keep after running: YES — formal Stage-09 oracle evidence
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "gap_candidates_0903" in globals()
assert "WARM2_0995" in globals()
assert "WARM2_09975" in globals()
assert "TRAIN_DIR" in globals()
assert "dataset_names" in globals()
assert "prefix_map" in globals()

assert "load_graph" in globals()
assert "apply_frozen_g2" in globals()
assert "apply_low_conf_component_pruning" in globals()
assert "node_coordinate_keys" in globals()

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
)


# ============================================================
# Freeze baseline-v7 PRE-CLEANUP state
# ============================================================

G2_P2_MIN = 0.80
CONF_COMPONENT_LIMIT = 9


# ============================================================
# Convert Stage-09 triplets into proposed missing edges
# ============================================================

proposals = (
    gap_candidates_0903
    .copy()
)

assert set(
    proposals["type"].unique()
) <= {
    "LEFT_MISSING",
    "RIGHT_MISSING",
}


# LEFT:
#     A(t)    B(t+1) -> C(t+2)
#       missing A -> B
#
# RIGHT:
#     A(t) -> B(t+1)    C(t+2)
#                missing B -> C

proposals["source_id"] = np.where(
    proposals["type"]
    == "LEFT_MISSING",
    proposals["a_id"],
    proposals["b_id"],
).astype(int)

proposals["target_id"] = np.where(
    proposals["type"]
    == "LEFT_MISSING",
    proposals["b_id"],
    proposals["c_id"],
).astype(int)


# ============================================================
# Recreate 09.03 conservative geometry flag
# ============================================================

proposals[
    "conservative"
] = (
    (
        proposals[
            "prediction_residual_um"
        ] <= 3.0
    )
    &
    (
        proposals[
            "max_step_um"
        ] <= 8.0
    )
    &
    (
        proposals[
            "step_balance"
        ] <= 0.50
    )
    &
    (
        proposals[
            "turn_angle_deg"
        ] <= 60.0
    )
    &
    (
        proposals[
            "ambiguity_margin_um"
        ] >= 1.0
    )
)

proposals["is_left"] = (
    proposals["type"]
    == "LEFT_MISSING"
)

proposals["is_right"] = (
    proposals["type"]
    == "RIGHT_MISSING"
)

proposals["conservative_left"] = (
    proposals["conservative"]
    & proposals["is_left"]
)

proposals["conservative_right"] = (
    proposals["conservative"]
    & proposals["is_right"]
)


# ============================================================
# Collapse duplicate directional proposals onto unique edge
#
# A single missing edge may be independently proposed by:
#   LEFT evidence
#   RIGHT evidence
#   or both.
# ============================================================

edge_support_0905 = (
    proposals
    .groupby(
        [
            "dataset",
            "prefix",
            "source_id",
            "target_id",
        ],
        as_index=False,
    )
    .agg(
        support_rows=(
            "type",
            "size",
        ),

        left_support=(
            "is_left",
            "max",
        ),

        right_support=(
            "is_right",
            "max",
        ),

        conservative_support=(
            "conservative",
            "max",
        ),

        conservative_left=(
            "conservative_left",
            "max",
        ),

        conservative_right=(
            "conservative_right",
            "max",
        ),

        min_residual_um=(
            "prediction_residual_um",
            "min",
        ),

        max_residual_um=(
            "prediction_residual_um",
            "max",
        ),

        min_turn_angle_deg=(
            "turn_angle_deg",
            "min",
        ),

        max_turn_angle_deg=(
            "turn_angle_deg",
            "max",
        ),

        min_ambiguity_margin_um=(
            "ambiguity_margin_um",
            "min",
        ),
    )
)

edge_support_0905[
    "mutual"
] = (
    edge_support_0905[
        "left_support"
    ]
    &
    edge_support_0905[
        "right_support"
    ]
)

edge_support_0905[
    "mutual_conservative"
] = (
    edge_support_0905[
        "conservative_left"
    ]
    &
    edge_support_0905[
        "conservative_right"
    ]
)


# ============================================================
# Official metric node matching for all Fold0 datasets
# ============================================================

oracle_parts = []

print(
    "=== 09.05 EXISTING-NODE GAP ORACLE AUDIT ==="
)

for i, name in enumerate(
    dataset_names,
    start=1,
):

    candidate_edges = (
        edge_support_0905[
            edge_support_0905[
                "dataset"
            ] == name
        ]
        .copy()
    )

    if not len(candidate_edges):
        continue

    # --------------------------------------------------------
    # Fresh baseline-v7 PRE-CLEANUP graph
    # --------------------------------------------------------

    pred_g = load_graph(
        WARM2_0995
        / f"{name}.geff"
    )

    high_g = load_graph(
        WARM2_09975
        / f"{name}.geff"
    )

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    apply_frozen_g2(
        pred_g,
        scale,
    )

    high_keys = (
        node_coordinate_keys(
            high_g
        )
    )

    apply_low_conf_component_pruning(
        pred_g,
        high_keys,
        CONF_COMPONENT_LIMIT,
    )

    # --------------------------------------------------------
    # Locked official matching.
    #
    # This mutates pred_g by adding:
    #   match_node_id
    #   match_score
    #   matched_edge_mask
    # --------------------------------------------------------

    official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )

    matched_nodes = (
        pred_g
        .node_attrs(
            unpack=True
        )
        .select(
            [
                "node_id",
                "match_node_id",
                "match_score",
            ]
        )
        .to_pandas()
    )

    match_map = dict(
        zip(
            matched_nodes[
                "node_id"
            ].astype(int),

            matched_nodes[
                "match_node_id"
            ].astype(int),
        )
    )

    score_map = dict(
        zip(
            matched_nodes[
                "node_id"
            ].astype(int),

            matched_nodes[
                "match_score"
            ].astype(float),
        )
    )

    source_match = np.asarray([
        match_map.get(
            int(node_id),
            -1,
        )
        for node_id in candidate_edges[
            "source_id"
        ]
    ])

    target_match = np.asarray([
        match_map.get(
            int(node_id),
            -1,
        )
        for node_id in candidate_edges[
            "target_id"
        ]
    ])

    candidate_edges[
        "gt_source_id"
    ] = source_match

    candidate_edges[
        "gt_target_id"
    ] = target_match

    candidate_edges[
        "source_match_score"
    ] = [
        score_map.get(
            int(node_id),
            0.0,
        )
        for node_id in candidate_edges[
            "source_id"
        ]
    ]

    candidate_edges[
        "target_match_score"
    ] = [
        score_map.get(
            int(node_id),
            0.0,
        )
        for node_id in candidate_edges[
            "target_id"
        ]
    ]

    candidate_edges[
        "observable"
    ] = (
        (
            candidate_edges[
                "gt_source_id"
            ] >= 0
        )
        &
        (
            candidate_edges[
                "gt_target_id"
            ] >= 0
        )
    )

    # --------------------------------------------------------
    # Oracle truth:
    #
    # Only meaningful when BOTH proposed endpoints are
    # observable under the locked official node matching.
    # --------------------------------------------------------

    truth = []

    for row in candidate_edges.itertuples():

        if not row.observable:
            truth.append(
                False
            )
            continue

        truth.append(
            bool(
                gt_g.has_edge(
                    int(
                        row.gt_source_id
                    ),
                    int(
                        row.gt_target_id
                    ),
                )
            )
        )

    candidate_edges[
        "true_repair"
    ] = truth

    oracle_parts.append(
        candidate_edges
    )

    if (
        i == 1
        or i % 10 == 0
        or i == len(dataset_names)
    ):
        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


oracle_edges_0905 = (
    pd.concat(
        oracle_parts,
        ignore_index=True,
    )
)

assert len(
    oracle_edges_0905
) == len(
    edge_support_0905
)


# ============================================================
# Scope definitions
# ============================================================

SCOPES = {
    "ALL":
        np.ones(
            len(
                oracle_edges_0905
            ),
            dtype=bool,
        ),

    "CONSERVATIVE":
        oracle_edges_0905[
            "conservative_support"
        ].to_numpy(
            dtype=bool
        ),

    "MUTUAL":
        oracle_edges_0905[
            "mutual"
        ].to_numpy(
            dtype=bool
        ),

    "MUTUAL_CONSERVATIVE":
        oracle_edges_0905[
            "mutual_conservative"
        ].to_numpy(
            dtype=bool
        ),
}


# ============================================================
# Summary helper
# ============================================================

def summarize_scope(
    df,
    mask,
    scope,
    prefix="ALL",
):

    sub = df.loc[
        mask
    ].copy()

    if prefix != "ALL":
        sub = sub[
            sub["prefix"]
            == prefix
        ].copy()

    n = len(sub)

    observable = int(
        sub[
            "observable"
        ].sum()
    )

    true_repairs = int(
        sub[
            "true_repair"
        ].sum()
    )

    observable_precision = (
        true_repairs
        / observable
        if observable
        else np.nan
    )

    observable_rate = (
        observable / n
        if n
        else np.nan
    )

    # --------------------------------------------------------
    # Candidate collision structure
    # --------------------------------------------------------

    if n:

        source_counts = (
            sub
            .groupby(
                [
                    "dataset",
                    "source_id",
                ]
            )[
                "target_id"
            ]
            .transform(
                "nunique"
            )
        )

        target_counts = (
            sub
            .groupby(
                [
                    "dataset",
                    "target_id",
                ]
            )[
                "source_id"
            ]
            .transform(
                "nunique"
            )
        )

        conflict_free_rate = float(
            (
                (source_counts == 1)
                &
                (target_counts == 1)
            ).mean()
        )

    else:

        conflict_free_rate = (
            np.nan
        )

    return {
        "scope":
            scope,

        "prefix":
            prefix,

        "unique_proposed_edges":
            n,

        "observable_edges":
            observable,

        "observable_rate":
            observable_rate,

        "true_repairs":
            true_repairs,

        "observable_precision":
            observable_precision,

        "conflict_free_rate":
            conflict_free_rate,
    }


# ============================================================
# Overall + prefix oracle tables
# ============================================================

summary_rows = []

for scope, mask in (
    SCOPES.items()
):

    summary_rows.append(
        summarize_scope(
            oracle_edges_0905,
            mask,
            scope,
            "ALL",
        )
    )

    for prefix in [
        "44b6",
        "6bba",
    ]:

        summary_rows.append(
            summarize_scope(
                oracle_edges_0905,
                mask,
                scope,
                prefix,
            )
        )


oracle_summary_0905 = (
    pd.DataFrame(
        summary_rows
    )
)


# ============================================================
# Direction-specific observable precision
#
# This is edge-level evidence:
#   LEFT_ONLY
#   RIGHT_ONLY
#   MUTUAL
# ============================================================

direction_df = (
    oracle_edges_0905
    .copy()
)

direction_df[
    "support_class"
] = np.select(
    [
        direction_df[
            "mutual"
        ],

        (
            direction_df[
                "left_support"
            ]
            &
            ~direction_df[
                "right_support"
            ]
        ),
    ],
    [
        "MUTUAL",
        "LEFT_ONLY",
    ],
    default="RIGHT_ONLY",
)

direction_rows = []

for (
    prefix,
    support_class
), sub in (
    direction_df
    .groupby(
        [
            "prefix",
            "support_class",
        ]
    )
):

    observable = int(
        sub[
            "observable"
        ].sum()
    )

    true_repairs = int(
        sub[
            "true_repair"
        ].sum()
    )

    direction_rows.append({
        "prefix":
            prefix,

        "support_class":
            support_class,

        "n":
            len(sub),

        "observable":
            observable,

        "true_repairs":
            true_repairs,

        "observable_precision":
            (
                true_repairs
                / observable
                if observable
                else np.nan
            ),
    })


direction_summary_0905 = (
    pd.DataFrame(
        direction_rows
    )
    .sort_values(
        [
            "prefix",
            "support_class",
        ]
    )
)


# ============================================================
# Conservative mutual detail
# ============================================================

mutual_cons = (
    oracle_edges_0905[
        oracle_edges_0905[
            "mutual_conservative"
        ]
    ]
    .copy()
)

mutual_cons_observed = (
    mutual_cons[
        mutual_cons[
            "observable"
        ]
    ]
)

mutual_cons_true = (
    mutual_cons[
        mutual_cons[
            "true_repair"
        ]
    ]
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.05 RESULT")
print("=" * 72)

print()
print("ORACLE SUMMARY")
display(
    oracle_summary_0905
)

print()
print("SUPPORT-CLASS SUMMARY")
display(
    direction_summary_0905
)

print()
print(
    "Mutual-conservative unique edges:",
    len(mutual_cons),
)

print(
    "Mutual-conservative observable:",
    len(
        mutual_cons_observed
    ),
)

print(
    "Mutual-conservative true repairs:",
    len(
        mutual_cons_true
    ),
)

if len(
    mutual_cons_observed
):
    print(
        "Mutual-conservative observable precision:",
        f"{len(mutual_cons_true) / len(mutual_cons_observed):.6f}",
    )

print()
print(
    "No prediction edge was added."
)

print(
    "No prediction file was modified."
)

print(
    "GT matching was used only for oracle analysis."
)

print()
print(
    "09.05 gap oracle audit: COMPLETE"
)

In [ ]:
# ============================================================
# DEBUG — 09.06a Edge-Addition Schema / In-Memory Safety Gate
# Purpose:
#   Inspect the prediction edge schema and verify that one
#   MUTUAL_CONSERVATIVE candidate can be added safely to a
#   fresh in-memory graph using the real IndexedRXGraph API.
#
#   The original GEFF file is NEVER modified.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: NO — DELETE after edge-add API is resolved
# ============================================================

import numpy as np


# ============================================================
# Preconditions
# ============================================================

assert "edge_support_0905" in globals()
assert "WARM2_0995" in globals()
assert "load_graph" in globals()


# ============================================================
# Select one real policy candidate
#
# IMPORTANT:
# mutual_conservative was derived entirely from prediction
# topology + geometry. No GT/oracle field is used here.
# ============================================================

policy_candidates = (
    edge_support_0905[
        edge_support_0905[
            "mutual_conservative"
        ]
    ]
    .copy()
)

assert len(policy_candidates) > 0

row = policy_candidates.iloc[0]

name = row["dataset"]
source_id = int(row["source_id"])
target_id = int(row["target_id"])


# ============================================================
# Fresh in-memory graph
# ============================================================

g = load_graph(
    WARM2_0995 / f"{name}.geff"
)

print("=== 09.06a EDGE-ADD SCHEMA GATE ===")
print()

print("Dataset:", name)
print("Candidate:", source_id, "->", target_id)

print()
print("Edge attribute keys:")
print(g.edge_attr_keys())

print()
print("Edge table columns:")

edge_df = g.edge_attrs(
    unpack=True
)

print(edge_df.columns)

print()
print("First 5 existing edges:")
print(edge_df.head(5))


# ============================================================
# Candidate topology preconditions
# ============================================================

print()
print("PRE-ADD")

print(
    "has_edge:",
    g.has_edge(
        source_id,
        target_id,
    ),
)

print(
    "source out_degree:",
    g.out_degree(
        source_id
    ),
)

print(
    "target in_degree:",
    g.in_degree(
        target_id
    ),
)

assert not g.has_edge(
    source_id,
    target_id,
)


# ============================================================
# Try empty attrs first.
#
# This only mutates this fresh in-memory graph object.
# ============================================================

before_edges = g.num_edges()

try:

    new_edge_id = g.add_edge(
        source_id,
        target_id,
        {},
    )

    add_mode = "EMPTY_ATTRS"

except Exception as exc:

    print()
    print(
        "add_edge({}, attrs={}) failed:"
    )

    print(
        type(exc).__name__,
        str(exc),
    )

    new_edge_id = None
    add_mode = "FAILED"


# ============================================================
# Verification
# ============================================================

print()
print("POST-ADD")

print(
    "mode:",
    add_mode,
)

print(
    "new edge id:",
    new_edge_id,
)

print(
    "has_edge:",
    g.has_edge(
        source_id,
        target_id,
    ),
)

print(
    "edge count:",
    before_edges,
    "->",
    g.num_edges(),
)

print(
    "source out_degree:",
    g.out_degree(
        source_id
    ),
)

print(
    "target in_degree:",
    g.in_degree(
        target_id
    ),
)


if add_mode == "EMPTY_ATTRS":

    assert g.has_edge(
        source_id,
        target_id,
    )

    assert (
        g.num_edges()
        == before_edges + 1
    )

    print()
    print(
        "Empty-attribute edge insertion: VERIFIED"
    )


print()
print("=" * 72)
print("09.06a RESULT")
print("=" * 72)

print(
    "Edge keys:",
    g.edge_attr_keys(),
)

print(
    "Insertion mode:",
    add_mode,
)

print(
    "Original GEFF modified: NO"
)

print()
print(
    "09.06a edge-add schema gate: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.06 Mutual-Conservative Existing-Node Repair Ablation
# Purpose:
#   Test whether adding the prediction-only MUTUAL_CONSERVATIVE
#   missing-edge proposals improves the frozen baseline-v7.
#
# Repair policy:
#   - candidates derived ONLY from prediction topology/geometry
#   - LEFT + RIGHT must independently propose the same edge
#   - conservative geometry must pass in both directions
#   - add exactly one existing-node association edge
#   - no synthetic nodes
#   - edge_dist uses verified model-grid convention
#   - edge_prob = 0.0 sentinel (cleanup does not inspect value)
#
# Pipeline:
#   Warm2 det=.995
#     -> G2 p2=.80
#     -> confidence pruning K=9
#     -> mutual-conservative repair
#     -> frozen singleton / 2-node cleanup
#     -> locked official metric
#
# Changes files: NO — graph changes are in-memory only
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only; NOT used by repair policy
# Keep after running: YES — formal Stage-09 repair ablation
# ============================================================

from pathlib import Path
from collections import defaultdict
import ast
import json

import numpy as np
import pandas as pd

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)


# ============================================================
# Preconditions / leakage guard
# ============================================================

assert "edge_support_0905" in globals()
assert "WARM2_0995" in globals()
assert "WARM2_09975" in globals()
assert "TRAIN_DIR" in globals()
assert "dataset_names" in globals()
assert "prefix_map" in globals()

assert "load_graph" in globals()
assert "apply_frozen_g2" in globals()
assert "apply_low_conf_component_pruning" in globals()
assert "node_coordinate_keys" in globals()
assert "component_sets" in globals()

# edge_support_0905 was constructed BEFORE GT oracle annotation.
# Explicitly prevent accidental oracle leakage.
FORBIDDEN_ORACLE_COLUMNS = {
    "observable",
    "true_repair",
    "gt_source_id",
    "gt_target_id",
    "source_match_score",
    "target_match_score",
}

assert not (
    FORBIDDEN_ORACLE_COLUMNS
    & set(edge_support_0905.columns)
), (
    "Oracle-derived columns detected in repair policy table."
)


# ============================================================
# Recover exact frozen cleanup from Stage 07
# ============================================================

NOTEBOOK_07 = (
    PROJECT
    / "notebooks/07_detection_threshold_sweep.ipynb"
)

with NOTEBOOK_07.open(
    "r",
    encoding="utf-8",
) as f:
    nb07 = json.load(f)

cell19_source = "".join(
    nb07["cells"][19]["source"]
)

tree = ast.parse(
    cell19_source
)

selected = [
    node
    for node in tree.body
    if (
        isinstance(node, ast.FunctionDef)
        and node.name
        == "apply_frozen_component_cleanup"
    )
]

assert len(selected) == 1

module = ast.Module(
    body=selected,
    type_ignores=[],
)

ast.fix_missing_locations(
    module
)

exec(
    compile(
        module,
        filename=(
            "07_detection_threshold_sweep.ipynb:"
            "cell_19"
        ),
        mode="exec",
    ),
    globals(),
)


# ============================================================
# Freeze baseline-v7 policy
# ============================================================

G2_P2_MIN = 0.80

G2_DAUGHTER_SEP_MIN_UM = 6.0
G2_ANGLE_MIN_DEG = 120.0
G2_MAX_PARENT_DAUGHTER_UM = 8.0
G2_DISTANCE_BALANCE_MAX = 0.50

CONF_COMPONENT_LIMIT = 9

TWO_NODE_MAX_DIST_UM = 3.5

FROZEN_BASELINE_V7_SCORE = 0.767286


# ============================================================
# Estimated-node metadata
# ============================================================

if "estimated_map" not in globals():

    cv_meta_0906 = pd.read_csv(
        PROJECT / "configs/cv_manifest.csv"
    )

    fold0_0906 = (
        cv_meta_0906[
            cv_meta_0906["fold"] == 0
        ]
        .copy()
    )

    assert len(fold0_0906) == 40

    estimated_map = dict(
        zip(
            fold0_0906["dataset"],
            fold0_0906["estimated_nodes"],
        )
    )


# ============================================================
# Frozen prediction-only repair policy
# ============================================================

repair_policy_0906 = (
    edge_support_0905[
        edge_support_0905[
            "mutual_conservative"
        ]
    ][
        [
            "dataset",
            "prefix",
            "source_id",
            "target_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)

repair_policy_0906[
    "source_id"
] = repair_policy_0906[
    "source_id"
].astype(int)

repair_policy_0906[
    "target_id"
] = repair_policy_0906[
    "target_id"
].astype(int)

assert len(
    repair_policy_0906
) == 2785


# ============================================================
# Verified recovered-edge attribute convention
# ============================================================

MODEL_DOWNSAMPLE_ZYX = np.array(
    [
        1.0,
        4.0,
        4.0,
    ],
    dtype=float,
)


def recovered_edge_dist(
    node_table,
    source_id,
    target_id,
):

    a = node_table.loc[
        source_id
    ]

    b = node_table.loc[
        target_id
    ]

    delta_zyx = np.array(
        [
            float(
                b["z"]
                - a["z"]
            ),
            float(
                b["y"]
                - a["y"]
            ),
            float(
                b["x"]
                - a["x"]
            ),
        ],
        dtype=float,
    )

    return float(
        np.linalg.norm(
            delta_zyx
            / MODEL_DOWNSAMPLE_ZYX
        )
    )


# ============================================================
# Shared baseline-v7 pre-cleanup preparation
# ============================================================

def prepare_precleanup_graph(
    name,
):

    g = load_graph(
        WARM2_0995
        / f"{name}.geff"
    )

    high_g = load_graph(
        WARM2_09975
        / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    apply_frozen_g2(
        g,
        scale,
    )

    high_keys = (
        node_coordinate_keys(
            high_g
        )
    )

    apply_low_conf_component_pruning(
        g,
        high_keys,
        CONF_COMPONENT_LIMIT,
    )

    return (
        g,
        scale,
    )


# ============================================================
# Metric helper
# ============================================================

def evaluate_graph_0906(
    graph,
    gt_graph,
    scale,
    name,
):

    er = official_evaluate(
        graph,
        gt_graph,
        scale=tuple(scale),
        max_distance=7.0,
    )

    recall = node_recall(
        graph,
        gt_graph,
    )

    return per_sample_metrics(
        er,
        float(
            estimated_map[name]
        ),
        recall,
    )


# ============================================================
# Full Fold0 baseline + repair evaluation
# ============================================================

baseline_records = []
repair_records = []

dataset_rows = []

total_expected_repairs = 0
total_added_repairs = 0

baseline_cleanup_removed = 0
repair_cleanup_removed = 0


print(
    "=== 09.06 MUTUAL-CONSERVATIVE REPAIR ABLATION ==="
)


for i, name in enumerate(
    dataset_names,
    start=1,
):

    # ========================================================
    # BASELINE-V7
    # ========================================================

    base_g, scale = (
        prepare_precleanup_graph(
            name
        )
    )

    base_cleanup = (
        apply_frozen_component_cleanup(
            base_g,
            scale,
        )
    )

    baseline_cleanup_removed += int(
        base_cleanup[
            "nodes_removed"
        ]
    )

    gt_base = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    base_m = evaluate_graph_0906(
        base_g,
        gt_base,
        scale,
        name,
    )

    baseline_records.append(
        base_m
    )


    # ========================================================
    # REPAIR GRAPH
    # ========================================================

    repair_g, repair_scale = (
        prepare_precleanup_graph(
            name
        )
    )

    assert np.allclose(
        scale,
        repair_scale,
    )

    candidate_edges = (
        repair_policy_0906[
            repair_policy_0906[
                "dataset"
            ] == name
        ]
    )

    expected_this_dataset = (
        len(candidate_edges)
    )

    total_expected_repairs += (
        expected_this_dataset
    )

    node_table = (
        repair_g
        .node_attrs(
            unpack=True
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )

    added_this_dataset = 0


    for row in (
        candidate_edges
        .itertuples(
            index=False
        )
    ):

        source_id = int(
            row.source_id
        )

        target_id = int(
            row.target_id
        )

        # Candidates were generated from this exact
        # G2+K9 pre-cleanup state.
        assert repair_g.has_node(
            source_id
        )

        assert repair_g.has_node(
            target_id
        )

        assert not repair_g.has_edge(
            source_id,
            target_id,
        )

        # Critical structural gate:
        # this is a track-end -> track-start repair.
        assert (
            repair_g.out_degree(
                source_id
            )
            == 0
        )

        assert (
            repair_g.in_degree(
                target_id
            )
            == 0
        )

        # Temporal association must remain Δt=1.
        t_source = int(
            node_table.loc[
                source_id,
                "t",
            ]
        )

        t_target = int(
            node_table.loc[
                target_id,
                "t",
            ]
        )

        assert (
            t_target
            == t_source + 1
        )

        edge_dist = (
            recovered_edge_dist(
                node_table,
                source_id,
                target_id,
            )
        )

        repair_g.add_edge(
            source_id,
            target_id,
            {
                "edge_dist":
                    edge_dist,

                # Sentinel only.
                # Frozen cleanup does not inspect its value.
                "edge_prob":
                    0.0,
            },
        )

        added_this_dataset += 1


    assert (
        added_this_dataset
        == expected_this_dataset
    )

    total_added_repairs += (
        added_this_dataset
    )


    # --------------------------------------------------------
    # Frozen cleanup AFTER repair
    # --------------------------------------------------------

    repair_cleanup = (
        apply_frozen_component_cleanup(
            repair_g,
            repair_scale,
        )
    )

    repair_cleanup_removed += int(
        repair_cleanup[
            "nodes_removed"
        ]
    )


    # --------------------------------------------------------
    # Locked metric
    # --------------------------------------------------------

    gt_repair = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    repair_m = (
        evaluate_graph_0906(
            repair_g,
            gt_repair,
            repair_scale,
            name,
        )
    )

    repair_records.append(
        repair_m
    )


    # --------------------------------------------------------
    # Per-dataset paired evidence
    # --------------------------------------------------------

    dataset_rows.append({
        "dataset":
            name,

        "prefix":
            prefix_map[name],

        "repairs_added":
            added_this_dataset,

        "baseline_edge_jaccard":
            base_m[
                "edge_jaccard"
            ],

        "repair_edge_jaccard":
            repair_m[
                "edge_jaccard"
            ],

        "delta_edge_jaccard":
            (
                repair_m[
                    "edge_jaccard"
                ]
                - base_m[
                    "edge_jaccard"
                ]
            ),

        "baseline_adj_edge_jaccard":
            base_m[
                "adj_edge_jaccard"
            ],

        "repair_adj_edge_jaccard":
            repair_m[
                "adj_edge_jaccard"
            ],

        "delta_adj_edge_jaccard":
            (
                repair_m[
                    "adj_edge_jaccard"
                ]
                - base_m[
                    "adj_edge_jaccard"
                ]
            ),

        "baseline_recall":
            base_m[
                "node_recall"
            ],

        "repair_recall":
            repair_m[
                "node_recall"
            ],

        "delta_recall":
            (
                repair_m[
                    "node_recall"
                ]
                - base_m[
                    "node_recall"
                ]
            ),

        "baseline_node_ratio":
            base_m[
                "total_node_ratio"
            ],

        "repair_node_ratio":
            repair_m[
                "total_node_ratio"
            ],

        "delta_node_ratio":
            (
                repair_m[
                    "total_node_ratio"
                ]
                - base_m[
                    "total_node_ratio"
                ]
            ),
    })


    if (
        i == 1
        or i % 10 == 0
        or i == len(
            dataset_names
        )
    ):

        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


# ============================================================
# Aggregate
# ============================================================

baseline_summary = summarise(
    baseline_records
)

repair_summary = summarise(
    repair_records
)


def summary_row(
    label,
    summary,
    records,
):

    return {
        "policy":
            label,

        "edge_jaccard":
            summary[
                "edge_jaccard"
            ],

        "adj_edge_jaccard":
            summary[
                "adj_edge_jaccard"
            ],

        "division_jaccard":
            summary[
                "division_jaccard"
            ],

        "node_recall":
            summary[
                "node_recall"
            ],

        "median_node_ratio":
            float(
                np.median([
                    m[
                        "total_node_ratio"
                    ]
                    for m in records
                ])
            ),

        "score":
            summary[
                "score"
            ],
    }


overall_0906 = pd.DataFrame([
    summary_row(
        "baseline_v7",
        baseline_summary,
        baseline_records,
    ),

    summary_row(
        "baseline_v7_plus_gap_repair",
        repair_summary,
        repair_records,
    ),
])


# ============================================================
# Reproduction gate
# ============================================================

base_score = float(
    baseline_summary[
        "score"
    ]
)

repair_score = float(
    repair_summary[
        "score"
    ]
)

repro_delta = (
    base_score
    - FROZEN_BASELINE_V7_SCORE
)

assert abs(
    repro_delta
) < 5e-6, (
    f"Baseline-v7 reproduction failed: "
    f"{base_score:.6f}"
)


# ============================================================
# Paired dataset robustness
# ============================================================

paired_0906 = pd.DataFrame(
    dataset_rows
)


paired_summary_rows = []

for prefix in [
    "ALL",
    "44b6",
    "6bba",
]:

    if prefix == "ALL":

        sub = (
            paired_0906
        )

    else:

        sub = (
            paired_0906[
                paired_0906[
                    "prefix"
                ] == prefix
            ]
        )

    paired_summary_rows.append({
        "prefix":
            prefix,

        "datasets":
            len(sub),

        "repairs_added":
            int(
                sub[
                    "repairs_added"
                ].sum()
            ),

        "edgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "adjEdgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "mean_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                sub[
                    "delta_recall"
                ].mean()
            ),

        "mean_delta_node_ratio":
            float(
                sub[
                    "delta_node_ratio"
                ].mean()
            ),
    })


paired_summary_0906 = (
    pd.DataFrame(
        paired_summary_rows
    )
)


# ============================================================
# Prefix aggregate metric
# ============================================================

prefix_rows = []

for prefix in [
    "44b6",
    "6bba",
]:

    idx = [
        i
        for i, name in enumerate(
            dataset_names
        )
        if (
            prefix_map[name]
            == prefix
        )
    ]

    base_sub = [
        baseline_records[i]
        for i in idx
    ]

    repair_sub = [
        repair_records[i]
        for i in idx
    ]

    bs = summarise(
        base_sub
    )

    rs = summarise(
        repair_sub
    )

    for label, s, recs in [
        (
            "baseline_v7",
            bs,
            base_sub,
        ),
        (
            "gap_repair",
            rs,
            repair_sub,
        ),
    ]:

        prefix_rows.append({
            "prefix":
                prefix,

            "policy":
                label,

            "edge_jaccard":
                s[
                    "edge_jaccard"
                ],

            "adj_edge_jaccard":
                s[
                    "adj_edge_jaccard"
                ],

            "division_jaccard":
                s[
                    "division_jaccard"
                ],

            "node_recall":
                s[
                    "node_recall"
                ],

            "median_node_ratio":
                float(
                    np.median([
                        m[
                            "total_node_ratio"
                        ]
                        for m in recs
                    ])
                ),

            "score":
                s[
                    "score"
                ],
        })


prefix_0906 = pd.DataFrame(
    prefix_rows
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.06 RESULT")
print("=" * 72)

print()
print("OVERALL")

display(
    overall_0906
)

print(
    "Repair - baseline score:",
    f"{repair_score - base_score:+.6f}",
)

print()
print("BY PREFIX")

display(
    prefix_0906
)

print()
print("PAIRED ROBUSTNESS")

display(
    paired_summary_0906
)

print()
print("REPAIR / CLEANUP COUNTS")

print(
    "Expected repair edges:",
    total_expected_repairs,
)

print(
    "Successfully added:",
    total_added_repairs,
)

print(
    "Baseline cleanup nodes removed:",
    baseline_cleanup_removed,
)

print(
    "Repair cleanup nodes removed:",
    repair_cleanup_removed,
)

print(
    "Nodes rescued from cleanup:",
    (
        baseline_cleanup_removed
        - repair_cleanup_removed
    ),
)

print()
print(
    "Baseline reproduction:",
    f"{base_score:.6f}",
)

print(
    "Reproduction delta:",
    f"{repro_delta:+.6f}",
)

print()
print(
    "No GEFF file was modified."
)

print(
    "Repair policy used NO GT information."
)

print()
print(
    "09.06 mutual-conservative repair ablation: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.07 Gap-Repair Mechanism Decomposition
# Purpose:
#   Determine whether the Stage-09 gain comes primarily from:
#
#     (1) directly restoring missing association edges, or
#     (2) repairing before cleanup and thereby rescuing nodes
#         that baseline-v7 would otherwise remove.
#
# Policies:
#   BASELINE:
#       G2 -> K9 -> cleanup
#
#   PRE_CLEANUP_REPAIR:
#       G2 -> K9 -> repair -> cleanup
#
#   POST_CLEANUP_REPAIR:
#       G2 -> K9 -> cleanup -> repair surviving endpoints only
#
# Changes files: NO — in-memory graphs only
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only
# Keep after running: YES — formal Stage-09 mechanism evidence
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Preconditions from retained 09.06
# ============================================================

assert "repair_policy_0906" in globals()
assert "prepare_precleanup_graph" in globals()
assert "recovered_edge_dist" in globals()
assert "evaluate_graph_0906" in globals()
assert "apply_frozen_component_cleanup" in globals()

assert len(repair_policy_0906) == 2785


# ============================================================
# Helper: add currently valid repair edges
# ============================================================

def add_available_repairs_0907(
    graph,
    candidate_edges,
):
    """
    Add a repair only when both endpoints still exist and
    source/target remain a valid track-end -> track-start pair.
    """

    node_table = (
        graph
        .node_attrs(
            unpack=True
        )
        .to_pandas()
        .set_index("node_id")
    )

    stats = {
        "candidate_edges":
            len(candidate_edges),

        "added":
            0,

        "missing_endpoint":
            0,

        "degree_conflict":
            0,

        "already_present":
            0,
    }

    for row in candidate_edges.itertuples(
        index=False
    ):

        source_id = int(
            row.source_id
        )

        target_id = int(
            row.target_id
        )

        if (
            not graph.has_node(source_id)
            or not graph.has_node(target_id)
        ):
            stats[
                "missing_endpoint"
            ] += 1
            continue

        if graph.has_edge(
            source_id,
            target_id,
        ):
            stats[
                "already_present"
            ] += 1
            continue

        if (
            graph.out_degree(source_id)
            != 0
            or graph.in_degree(target_id)
            != 0
        ):
            stats[
                "degree_conflict"
            ] += 1
            continue

        t_source = int(
            node_table.loc[
                source_id,
                "t",
            ]
        )

        t_target = int(
            node_table.loc[
                target_id,
                "t",
            ]
        )

        assert (
            t_target
            == t_source + 1
        )

        edge_dist = (
            recovered_edge_dist(
                node_table,
                source_id,
                target_id,
            )
        )

        graph.add_edge(
            source_id,
            target_id,
            {
                "edge_dist":
                    edge_dist,

                "edge_prob":
                    0.0,
            },
        )

        stats["added"] += 1

    return stats


# ============================================================
# Fold0 evaluation
# ============================================================

baseline_records_0907 = []
pre_records_0907 = []
post_records_0907 = []

rows_0907 = []

pre_added_total = 0
post_added_total = 0
post_missing_total = 0

baseline_removed_total = 0
pre_removed_total = 0


print(
    "=== 09.07 GAP-REPAIR MECHANISM DECOMPOSITION ==="
)


for i, name in enumerate(
    dataset_names,
    start=1,
):

    candidates = (
        repair_policy_0906[
            repair_policy_0906[
                "dataset"
            ] == name
        ]
    )

    # ========================================================
    # A. BASELINE
    # ========================================================

    base_g, scale = (
        prepare_precleanup_graph(
            name
        )
    )

    base_cleanup = (
        apply_frozen_component_cleanup(
            base_g,
            scale,
        )
    )

    baseline_removed_total += int(
        base_cleanup[
            "nodes_removed"
        ]
    )

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    base_m = evaluate_graph_0906(
        base_g,
        gt_g,
        scale,
        name,
    )

    baseline_records_0907.append(
        base_m
    )


    # ========================================================
    # B. REPAIR BEFORE CLEANUP
    # ========================================================

    pre_g, pre_scale = (
        prepare_precleanup_graph(
            name
        )
    )

    pre_stats = (
        add_available_repairs_0907(
            pre_g,
            candidates,
        )
    )

    assert (
        pre_stats["added"]
        == len(candidates)
    )

    pre_added_total += (
        pre_stats["added"]
    )

    pre_cleanup = (
        apply_frozen_component_cleanup(
            pre_g,
            pre_scale,
        )
    )

    pre_removed_total += int(
        pre_cleanup[
            "nodes_removed"
        ]
    )

    gt_pre = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    pre_m = evaluate_graph_0906(
        pre_g,
        gt_pre,
        pre_scale,
        name,
    )

    pre_records_0907.append(
        pre_m
    )


    # ========================================================
    # C. CLEANUP BEFORE REPAIR
    # ========================================================

    post_g, post_scale = (
        prepare_precleanup_graph(
            name
        )
    )

    apply_frozen_component_cleanup(
        post_g,
        post_scale,
    )

    post_stats = (
        add_available_repairs_0907(
            post_g,
            candidates,
        )
    )

    post_added_total += (
        post_stats["added"]
    )

    post_missing_total += (
        post_stats[
            "missing_endpoint"
        ]
    )

    gt_post = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    post_m = evaluate_graph_0906(
        post_g,
        gt_post,
        post_scale,
        name,
    )

    post_records_0907.append(
        post_m
    )


    rows_0907.append({
        "dataset":
            name,

        "prefix":
            prefix_map[name],

        "repairs_requested":
            len(candidates),

        "pre_repairs_added":
            pre_stats["added"],

        "post_repairs_added":
            post_stats["added"],

        "post_missing_endpoint":
            post_stats[
                "missing_endpoint"
            ],

        "delta_pre_edgeJ":
            (
                pre_m["edge_jaccard"]
                - base_m["edge_jaccard"]
            ),

        "delta_post_edgeJ":
            (
                post_m["edge_jaccard"]
                - base_m["edge_jaccard"]
            ),

        "delta_pre_adjEdgeJ":
            (
                pre_m[
                    "adj_edge_jaccard"
                ]
                - base_m[
                    "adj_edge_jaccard"
                ]
            ),

        "delta_post_adjEdgeJ":
            (
                post_m[
                    "adj_edge_jaccard"
                ]
                - base_m[
                    "adj_edge_jaccard"
                ]
            ),

        "delta_pre_recall":
            (
                pre_m["node_recall"]
                - base_m["node_recall"]
            ),

        "delta_post_recall":
            (
                post_m["node_recall"]
                - base_m["node_recall"]
            ),
    })


    if (
        i == 1
        or i % 10 == 0
        or i == len(dataset_names)
    ):
        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


# ============================================================
# Aggregate
# ============================================================

base_s = summarise(
    baseline_records_0907
)

pre_s = summarise(
    pre_records_0907
)

post_s = summarise(
    post_records_0907
)


def row_from_summary_0907(
    policy,
    s,
    records,
):

    return {
        "policy":
            policy,

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    }


overall_0907 = pd.DataFrame([
    row_from_summary_0907(
        "baseline_v7",
        base_s,
        baseline_records_0907,
    ),

    row_from_summary_0907(
        "repair_before_cleanup",
        pre_s,
        pre_records_0907,
    ),

    row_from_summary_0907(
        "repair_after_cleanup",
        post_s,
        post_records_0907,
    ),
])


# ============================================================
# Reproduction gates
# ============================================================

assert abs(
    float(base_s["score"])
    - 0.767286
) < 5e-6

assert abs(
    float(pre_s["score"])
    - 0.769099
) < 5e-6


# ============================================================
# Prefix metrics
# ============================================================

prefix_rows_0907 = []

for prefix in [
    "44b6",
    "6bba",
]:

    idx = [
        j
        for j, name in enumerate(
            dataset_names
        )
        if prefix_map[name]
        == prefix
    ]

    variants = {
        "baseline_v7": [
            baseline_records_0907[j]
            for j in idx
        ],

        "repair_before_cleanup": [
            pre_records_0907[j]
            for j in idx
        ],

        "repair_after_cleanup": [
            post_records_0907[j]
            for j in idx
        ],
    }

    for policy, recs in (
        variants.items()
    ):

        s = summarise(
            recs
        )

        prefix_rows_0907.append({
            "prefix":
                prefix,

            "policy":
                policy,

            "edge_jaccard":
                s["edge_jaccard"],

            "adj_edge_jaccard":
                s["adj_edge_jaccard"],

            "node_recall":
                s["node_recall"],

            "score":
                s["score"],
        })


prefix_0907 = pd.DataFrame(
    prefix_rows_0907
)


# ============================================================
# Paired mechanism summary
# ============================================================

paired_0907 = pd.DataFrame(
    rows_0907
)

mechanism_rows = []

for prefix in [
    "ALL",
    "44b6",
    "6bba",
]:

    sub = (
        paired_0907
        if prefix == "ALL"
        else paired_0907[
            paired_0907[
                "prefix"
            ] == prefix
        ]
    )

    mechanism_rows.append({
        "prefix":
            prefix,

        "datasets":
            len(sub),

        "requested_repairs":
            int(
                sub[
                    "repairs_requested"
                ].sum()
            ),

        "pre_added":
            int(
                sub[
                    "pre_repairs_added"
                ].sum()
            ),

        "post_added":
            int(
                sub[
                    "post_repairs_added"
                ].sum()
            ),

        "post_lost_to_cleanup":
            int(
                sub[
                    "post_missing_endpoint"
                ].sum()
            ),

        "pre_edgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_pre_edgeJ"
                    ] > 0
                ).mean()
            ),

        "post_edgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_post_edgeJ"
                    ] > 0
                ).mean()
            ),

        "mean_pre_delta_edgeJ":
            float(
                sub[
                    "delta_pre_edgeJ"
                ].mean()
            ),

        "mean_post_delta_edgeJ":
            float(
                sub[
                    "delta_post_edgeJ"
                ].mean()
            ),

        "mean_pre_delta_recall":
            float(
                sub[
                    "delta_pre_recall"
                ].mean()
            ),

        "mean_post_delta_recall":
            float(
                sub[
                    "delta_post_recall"
                ].mean()
            ),
    })


mechanism_0907 = pd.DataFrame(
    mechanism_rows
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.07 RESULT")
print("=" * 72)

print()
print("OVERALL")
display(
    overall_0907
)

print()
print("BY PREFIX")
display(
    prefix_0907
)

print()
print("MECHANISM")
display(
    mechanism_0907
)

print()
print("GLOBAL COUNTS")

print(
    "Requested repair edges:",
    len(repair_policy_0906),
)

print(
    "Pre-cleanup repairs added:",
    pre_added_total,
)

print(
    "Post-cleanup repairs added:",
    post_added_total,
)

print(
    "Repair endpoints lost to baseline cleanup:",
    post_missing_total,
)

print()

print(
    "Baseline cleanup nodes removed:",
    baseline_removed_total,
)

print(
    "Repair-before-cleanup nodes removed:",
    pre_removed_total,
)

print(
    "Nodes rescued:",
    baseline_removed_total
    - pre_removed_total,
)

print()
print(
    "Direct-edge-only score gain:",
    f"{float(post_s['score']) - float(base_s['score']):+.6f}",
)

print(
    "Full repair score gain:",
    f"{float(pre_s['score']) - float(base_s['score']):+.6f}",
)

print(
    "Additional gain from pre-cleanup rescue:",
    f"{float(pre_s['score']) - float(post_s['score']):+.6f}",
)

print()
print(
    "No GEFF file was modified."
)

print()
print(
    "09.07 repair mechanism decomposition: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.08 Mutual-Repair Residual Radius Robustness
# Purpose:
#   Test whether the successful mutual-conservative existing-node
#   repair is robust to a small change in constant-velocity
#   residual radius.
#
#   Only residual threshold changes:
#       2.0 um  = stricter
#       3.0 um  = current Stage-09 policy
#       4.0 um  = slightly looser
#
#   Frozen for every candidate direction:
#       max_step <= 8 um
#       step_balance <= 0.50
#       turn_angle <= 60 deg
#       ambiguity_margin >= 1 um
#
#   An edge is accepted only when BOTH LEFT and RIGHT evidence
#   independently pass the same geometry gates.
#
# Pipeline:
#   Warm2 -> G2 p2=.80 -> K9
#   -> mutual repair
#   -> frozen cleanup
#   -> locked metric
#
# Changes files: NO — in-memory only
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only; repair policy uses NO GT
# Keep after running: YES — formal Stage-09 robustness evidence
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "gap_candidates_0903" in globals()
assert "prepare_precleanup_graph" in globals()
assert "add_available_repairs_0907" in globals()
assert "apply_frozen_component_cleanup" in globals()
assert "evaluate_graph_0906" in globals()
assert "baseline_records_0907" in globals()

RADIUS_CANDIDATES_0908 = [
    2.0,
    3.0,
    4.0,
]


# ============================================================
# Build a prediction-only mutual policy at a given radius
# ============================================================

def build_mutual_policy_0908(
    residual_radius_um,
):

    p = (
        gap_candidates_0903
        .copy()
    )

    p["source_id"] = np.where(
        p["type"] == "LEFT_MISSING",
        p["a_id"],
        p["b_id"],
    ).astype(int)

    p["target_id"] = np.where(
        p["type"] == "LEFT_MISSING",
        p["b_id"],
        p["c_id"],
    ).astype(int)

    # ----------------------------------------
    # Same conservative geometry as 09.06.
    # Only residual_radius_um changes.
    # ----------------------------------------

    p["geometry_pass"] = (
        (
            p["prediction_residual_um"]
            <= residual_radius_um
        )
        &
        (
            p["max_step_um"]
            <= 8.0
        )
        &
        (
            p["step_balance"]
            <= 0.50
        )
        &
        (
            p["turn_angle_deg"]
            <= 60.0
        )
        &
        (
            p["ambiguity_margin_um"]
            >= 1.0
        )
    )

    p["left_pass"] = (
        p["geometry_pass"]
        &
        (
            p["type"]
            == "LEFT_MISSING"
        )
    )

    p["right_pass"] = (
        p["geometry_pass"]
        &
        (
            p["type"]
            == "RIGHT_MISSING"
        )
    )

    edge_level = (
        p
        .groupby(
            [
                "dataset",
                "prefix",
                "source_id",
                "target_id",
            ],
            as_index=False,
        )
        .agg(
            left_pass=(
                "left_pass",
                "max",
            ),

            right_pass=(
                "right_pass",
                "max",
            ),
        )
    )

    policy = (
        edge_level[
            edge_level["left_pass"]
            &
            edge_level["right_pass"]
        ][
            [
                "dataset",
                "prefix",
                "source_id",
                "target_id",
            ]
        ]
        .copy()
    )

    return policy


# ============================================================
# Verify current r=3 policy reproduces 2785 candidates
# ============================================================

policy_r3_gate = (
    build_mutual_policy_0908(
        3.0
    )
)

assert len(policy_r3_gate) == 2785, (
    f"Current policy reproduction failed: "
    f"{len(policy_r3_gate)} != 2785"
)


# ============================================================
# Evaluate each radius
# ============================================================

radius_results = []
radius_dataset_rows = {}
radius_prefix_rows = []

print(
    "=== 09.08 MUTUAL REPAIR RESIDUAL ROBUSTNESS ==="
)


for radius in RADIUS_CANDIDATES_0908:

    policy = (
        build_mutual_policy_0908(
            radius
        )
    )

    records = []
    dataset_rows = []

    total_added = 0
    total_cleanup_removed = 0

    for i, name in enumerate(
        dataset_names
    ):

        g, scale = (
            prepare_precleanup_graph(
                name
            )
        )

        candidates = (
            policy[
                policy["dataset"]
                == name
            ]
        )

        stats = (
            add_available_repairs_0907(
                g,
                candidates,
            )
        )

        assert (
            stats["added"]
            == len(candidates)
        )

        total_added += (
            stats["added"]
        )

        cleanup = (
            apply_frozen_component_cleanup(
                g,
                scale,
            )
        )

        total_cleanup_removed += int(
            cleanup[
                "nodes_removed"
            ]
        )

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )

        m = evaluate_graph_0906(
            g,
            gt_g,
            scale,
            name,
        )

        records.append(m)

        base_m = (
            baseline_records_0907[i]
        )

        dataset_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "radius_um":
                radius,

            "repairs_added":
                stats["added"],

            "delta_edge_jaccard":
                (
                    m["edge_jaccard"]
                    - base_m["edge_jaccard"]
                ),

            "delta_adj_edge_jaccard":
                (
                    m["adj_edge_jaccard"]
                    - base_m[
                        "adj_edge_jaccard"
                    ]
                ),

            "delta_recall":
                (
                    m["node_recall"]
                    - base_m["node_recall"]
                ),
        })

    s = summarise(
        records
    )

    radius_results.append({
        "radius_um":
            radius,

        "repair_edges":
            len(policy),

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],

        "cleanup_nodes_removed":
            total_cleanup_removed,

        "nodes_rescued":
            (
                86571
                - total_cleanup_removed
            ),
    })

    df = pd.DataFrame(
        dataset_rows
    )

    radius_dataset_rows[
        radius
    ] = df

    # ----------------------------------------
    # Prefix aggregate
    # ----------------------------------------

    for prefix in [
        "44b6",
        "6bba",
    ]:

        idx = [
            j
            for j, name in enumerate(
                dataset_names
            )
            if prefix_map[name]
            == prefix
        ]

        recs = [
            records[j]
            for j in idx
        ]

        ps = summarise(
            recs
        )

        sub = df[
            df["prefix"] == prefix
        ]

        radius_prefix_rows.append({
            "radius_um":
                radius,

            "prefix":
                prefix,

            "repair_edges":
                int(
                    sub[
                        "repairs_added"
                    ].sum()
                ),

            "edge_jaccard":
                ps["edge_jaccard"],

            "adj_edge_jaccard":
                ps[
                    "adj_edge_jaccard"
                ],

            "node_recall":
                ps["node_recall"],

            "score":
                ps["score"],

            "edgeJ_improved_rate":
                float(
                    (
                        sub[
                            "delta_edge_jaccard"
                        ] > 0
                    ).mean()
                ),

            "adjEdgeJ_improved_rate":
                float(
                    (
                        sub[
                            "delta_adj_edge_jaccard"
                        ] > 0
                    ).mean()
                ),

            "median_delta_edgeJ":
                float(
                    sub[
                        "delta_edge_jaccard"
                    ].median()
                ),

            "median_delta_adjEdgeJ":
                float(
                    sub[
                        "delta_adj_edge_jaccard"
                    ].median()
                ),
        })

    print(
        f"radius={radius:.1f} um "
        f"| repairs={len(policy)} "
        f"| score={s['score']:.6f}"
    )


# ============================================================
# Tables
# ============================================================

overall_0908 = (
    pd.DataFrame(
        radius_results
    )
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

prefix_0908 = (
    pd.DataFrame(
        radius_prefix_rows
    )
    .sort_values(
        [
            "prefix",
            "radius_um",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# Reproduction gate for current r=3 policy
# ============================================================

r3 = (
    overall_0908[
        overall_0908[
            "radius_um"
        ] == 3.0
    ]
    .iloc[0]
)

assert abs(
    float(r3["score"])
    - 0.769099
) < 5e-6, (
    "09.06 r=3 reproduction failed."
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.08 RESULT")
print("=" * 72)

print()
print("OVERALL")
display(
    overall_0908
)

print()
print("BY PREFIX / ROBUSTNESS")
display(
    prefix_0908
)

best = (
    overall_0908
    .iloc[0]
)

print()
print(
    "Best radius:",
    f"{best['radius_um']:.1f} um",
)

print(
    "Best score:",
    f"{best['score']:.6f}",
)

print(
    "Delta vs current r=3:",
    f"{best['score'] - 0.769099:+.6f}",
)

print()
print(
    "09.08 residual-radius robustness: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.09 Mutual-Repair r=5um Boundary Check
# Purpose:
#   Test exactly one looser residual radius (5.0 um) because
#   the best 09.08 result occurred at the upper tested boundary
#   r=4.0 um.
#
#   This is NOT a new broad sweep.
#
# Frozen:
#   Warm2 checkpoint
#   det = .995
#   high confidence = .9975
#   G2 p2 = .80
#   confidence pruning K = 9
#
#   Mutual support required.
#
#   Per-direction geometry:
#       residual <= 5.0 um
#       max_step <= 8 um
#       step_balance <= .50
#       turn_angle <= 60 deg
#       ambiguity margin >= 1 um
#
#   Repair occurs BEFORE frozen cleanup.
#
# Changes files: NO — in-memory graphs only
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only; repair policy uses NO GT
# Keep after running: YES — Stage-09 boundary evidence
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "build_mutual_policy_0908" in globals()
assert "prepare_precleanup_graph" in globals()
assert "add_available_repairs_0907" in globals()
assert "apply_frozen_component_cleanup" in globals()
assert "evaluate_graph_0906" in globals()
assert "baseline_records_0907" in globals()

RADIUS_0909 = 5.0


# ============================================================
# Prediction-only policy
# ============================================================

policy_0909 = (
    build_mutual_policy_0908(
        RADIUS_0909
    )
)

print(
    "=== 09.09 r=5um BOUNDARY CHECK ==="
)

print(
    "Repair candidates:",
    len(policy_0909),
)


# ============================================================
# Fold0 evaluation
# ============================================================

records_0909 = []
rows_0909 = []

total_added_0909 = 0
total_cleanup_removed_0909 = 0


for i, name in enumerate(
    dataset_names
):

    g, scale = (
        prepare_precleanup_graph(
            name
        )
    )

    candidates = (
        policy_0909[
            policy_0909[
                "dataset"
            ] == name
        ]
    )

    stats = (
        add_available_repairs_0907(
            g,
            candidates,
        )
    )

    assert (
        stats["added"]
        == len(candidates)
    )

    total_added_0909 += (
        stats["added"]
    )

    cleanup = (
        apply_frozen_component_cleanup(
            g,
            scale,
        )
    )

    total_cleanup_removed_0909 += int(
        cleanup[
            "nodes_removed"
        ]
    )

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    m = evaluate_graph_0906(
        g,
        gt_g,
        scale,
        name,
    )

    records_0909.append(
        m
    )

    base_m = (
        baseline_records_0907[i]
    )

    rows_0909.append({
        "dataset":
            name,

        "prefix":
            prefix_map[name],

        "repairs_added":
            stats["added"],

        "delta_edge_jaccard":
            (
                m["edge_jaccard"]
                - base_m["edge_jaccard"]
            ),

        "delta_adj_edge_jaccard":
            (
                m["adj_edge_jaccard"]
                - base_m[
                    "adj_edge_jaccard"
                ]
            ),

        "delta_recall":
            (
                m["node_recall"]
                - base_m["node_recall"]
            ),
    })


# ============================================================
# Overall
# ============================================================

s0909 = summarise(
    records_0909
)

overall_0909 = pd.DataFrame([
    {
        "radius_um":
            3.0,

        "repair_edges":
            2785,

        "edge_jaccard":
            0.772120,

        "adj_edge_jaccard":
            0.765253,

        "division_jaccard":
            0.038462,

        "node_recall":
            0.953529,

        "score":
            0.769099,

        "nodes_rescued":
            992,
    },

    {
        "radius_um":
            4.0,

        "repair_edges":
            3945,

        "edge_jaccard":
            0.772746,

        "adj_edge_jaccard":
            0.765801,

        "division_jaccard":
            0.038462,

        "node_recall":
            0.954728,

        "score":
            0.769647,

        "nodes_rescued":
            1768,
    },

    {
        "radius_um":
            5.0,

        "repair_edges":
            len(policy_0909),

        "edge_jaccard":
            s0909[
                "edge_jaccard"
            ],

        "adj_edge_jaccard":
            s0909[
                "adj_edge_jaccard"
            ],

        "division_jaccard":
            s0909[
                "division_jaccard"
            ],

        "node_recall":
            s0909[
                "node_recall"
            ],

        "score":
            s0909[
                "score"
            ],

        "nodes_rescued":
            (
                86571
                - total_cleanup_removed_0909
            ),
    },
])


# ============================================================
# Prefix / paired robustness
# ============================================================

df0909 = pd.DataFrame(
    rows_0909
)

prefix_rows_0909 = []

for prefix in [
    "44b6",
    "6bba",
]:

    idx = [
        j
        for j, name in enumerate(
            dataset_names
        )
        if prefix_map[name]
        == prefix
    ]

    recs = [
        records_0909[j]
        for j in idx
    ]

    ps = summarise(
        recs
    )

    sub = (
        df0909[
            df0909[
                "prefix"
            ] == prefix
        ]
    )

    prefix_rows_0909.append({
        "prefix":
            prefix,

        "repair_edges":
            int(
                sub[
                    "repairs_added"
                ].sum()
            ),

        "edge_jaccard":
            ps["edge_jaccard"],

        "adj_edge_jaccard":
            ps[
                "adj_edge_jaccard"
            ],

        "node_recall":
            ps["node_recall"],

        "score":
            ps["score"],

        "edgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "adjEdgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "mean_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                sub[
                    "delta_recall"
                ].mean()
            ),
    })


prefix_0909 = pd.DataFrame(
    prefix_rows_0909
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.09 RESULT")
print("=" * 72)

print()
print("OVERALL: r=3 vs r=4 vs r=5")

display(
    overall_0909
)

print()
print("r=5 BY PREFIX / PAIRED ROBUSTNESS")

display(
    prefix_0909
)

score5 = float(
    s0909["score"]
)

print()
print(
    "r=5 - r=4 score:",
    f"{score5 - 0.769647:+.6f}",
)

print(
    "r=5 - baseline-v7 score:",
    f"{score5 - 0.767286:+.6f}",
)

print()
print(
    "Repair edges added:",
    total_added_0909,
)

print(
    "Nodes rescued:",
    86571
    - total_cleanup_removed_0909,
)

print()
print(
    "No GEFF file was modified."
)

print(
    "Repair policy used NO GT information."
)

print()
print(
    "09.09 r=5 boundary check: COMPLETE"
)

## 09.09 Decision — Residual Radius Frozen at 4.0 µm

The 5.0 µm boundary check produced a micro-improvement:

- r=4.0: 0.769647
- r=5.0: 0.769671
- delta: +0.000024

However, the additional 147 repaired edges produced essentially no
additional recall and no improvement on the 44b6 aggregate score.
The gain is too small to justify loosening the recovery rule.

**Decision:** retain `residual <= 4.0 µm` as the robust Stage-09 policy.

**baseline-v8 Fold0 score: 0.769647**

In [ ]:
# ============================================================
# RUN — 09.10 Existing-Node Two-Edge Bridge Census
# Purpose:
#   Measure plausible one-frame isolated-node bridge structures:
#
#       A(t)     B(t+1)     C(t+2)
#
#   where:
#       A is a track end
#       B is an isolated EXISTING prediction node
#       C is a track start
#
#   A candidate would eventually repair:
#
#       A -> B -> C
#
#   but this Cell ONLY performs prediction-only geometry census.
#   No edge is added.
#
# Pipeline state:
#   Warm2 det=.995
#     -> G2 p2=.80
#     -> confidence pruning K=9
#     -> [THIS AUDIT]
#     -> gap repair / cleanup later
#
# Geometry:
#   Candidate A and C must each be within 8 um of B.
#   For every isolated B, retain the A/C pair with minimum
#   constant-velocity residual:
#
#       || A + C - 2B ||
#
#   This is equivalent to the residual convention already used
#   for Stage-09 single-edge recovery.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: YES — Stage-09 bridge census evidence
# ============================================================

from collections import defaultdict

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from tracking_cellmot.io import open_dataset


# ============================================================
# Preconditions
# ============================================================

assert "PROJECT" in globals()
assert "WARM2_0995" in globals()
assert "WARM2_09975" in globals()
assert "TRAIN_DIR" in globals()
assert "dataset_names" in globals()
assert "prefix_map" in globals()

assert "load_graph" in globals()
assert "apply_frozen_g2" in globals()
assert "apply_low_conf_component_pruning" in globals()
assert "node_coordinate_keys" in globals()


# ============================================================
# Freeze baseline-v8 pre-cleanup state
# ============================================================

G2_P2_MIN = 0.80

G2_DAUGHTER_SEP_MIN_UM = 6.0
G2_ANGLE_MIN_DEG = 120.0
G2_MAX_PARENT_DAUGHTER_UM = 8.0
G2_DISTANCE_BALANCE_MAX = 0.50

CONF_COMPONENT_LIMIT = 9


# ============================================================
# Geometry helpers
# ============================================================

MAX_STEP_UM_0910 = 8.0


def turn_angle_deg_0910(
    a,
    b,
    c,
):
    """
    0 deg = straight continuation A -> B -> C.
    """

    v1 = b - a
    v2 = c - b

    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)

    if (
        n1 <= 1e-12
        or n2 <= 1e-12
    ):
        return np.nan

    cosine = np.dot(
        v1,
        v2,
    ) / (n1 * n2)

    cosine = np.clip(
        cosine,
        -1.0,
        1.0,
    )

    return float(
        np.degrees(
            np.arccos(cosine)
        )
    )


def step_balance_0910(
    d1,
    d2,
):
    return float(
        abs(d1 - d2)
        / max(
            d1,
            d2,
            1e-12,
        )
    )


# ============================================================
# One-dataset census
# ============================================================

def census_isolated_bridges_0910(
    name,
):

    # --------------------------------------------------------
    # Fresh prediction graphs
    # --------------------------------------------------------

    g = load_graph(
        WARM2_0995
        / f"{name}.geff"
    )

    high_g = load_graph(
        WARM2_09975
        / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    assert scale.shape == (3,)


    # --------------------------------------------------------
    # Frozen G2 + K9, BEFORE cleanup / gap repair
    # --------------------------------------------------------

    apply_frozen_g2(
        g,
        scale,
    )

    high_keys = (
        node_coordinate_keys(
            high_g
        )
    )

    apply_low_conf_component_pruning(
        g,
        high_keys,
        CONF_COMPONENT_LIMIT,
    )


    # --------------------------------------------------------
    # Node arrays
    # --------------------------------------------------------

    attrs = (
        g.node_attrs(
            unpack=True
        )
        .to_pandas()
    )

    node_ids = (
        attrs["node_id"]
        .to_numpy(
            dtype=int
        )
    )

    t_arr = (
        attrs["t"]
        .to_numpy(
            dtype=int
        )
    )

    z_arr = (
        attrs["z"]
        .to_numpy(
            dtype=float
        )
    )

    y_arr = (
        attrs["y"]
        .to_numpy(
            dtype=float
        )
    )

    x_arr = (
        attrs["x"]
        .to_numpy(
            dtype=float
        )
    )

    # Physical z,y,x coordinates in microns.
    pos = np.column_stack([
        z_arr * scale[0],
        y_arr * scale[1],
        x_arr * scale[2],
    ])

    in_deg = np.asarray(
        g.in_degree(
            node_ids.tolist()
        ),
        dtype=int,
    )

    out_deg = np.asarray(
        g.out_degree(
            node_ids.tolist()
        ),
        dtype=int,
    )


    # --------------------------------------------------------
    # Relevant topology classes
    # --------------------------------------------------------

    isolated_mask = (
        (in_deg == 0)
        &
        (out_deg == 0)
        &
        (t_arr > t_arr.min())
        &
        (t_arr < t_arr.max())
    )

    end_mask = (
        out_deg == 0
    )

    start_mask = (
        in_deg == 0
    )


    isolated_idx = np.flatnonzero(
        isolated_mask
    )


    # --------------------------------------------------------
    # Frame indexes / KD trees
    # --------------------------------------------------------

    ends_by_t = {}
    starts_by_t = {}

    for t in np.unique(
        t_arr
    ):

        frame = np.flatnonzero(
            t_arr == t
        )

        ends = frame[
            end_mask[frame]
        ]

        starts = frame[
            start_mask[frame]
        ]

        if len(ends):
            ends_by_t[int(t)] = (
                cKDTree(
                    pos[ends]
                ),
                ends,
            )

        if len(starts):
            starts_by_t[int(t)] = (
                cKDTree(
                    pos[starts]
                ),
                starts,
            )


    # --------------------------------------------------------
    # Best A-B-C geometry for every isolated B
    # --------------------------------------------------------

    rows = []

    isolated_with_pair = 0

    for bi in isolated_idx:

        tb = int(
            t_arr[bi]
        )

        # Need:
        #   A at tb-1
        #   C at tb+1
        if (
            tb - 1
            not in ends_by_t
            or tb + 1
            not in starts_by_t
        ):
            continue


        end_tree, end_idx = (
            ends_by_t[
                tb - 1
            ]
        )

        start_tree, start_idx = (
            starts_by_t[
                tb + 1
            ]
        )


        # ----------------------------------------
        # Candidate endpoints within 8 um of B
        # ----------------------------------------

        local_a = (
            end_tree.query_ball_point(
                pos[bi],
                r=MAX_STEP_UM_0910,
            )
        )

        local_c = (
            start_tree.query_ball_point(
                pos[bi],
                r=MAX_STEP_UM_0910,
            )
        )

        if (
            not local_a
            or not local_c
        ):
            continue

        isolated_with_pair += 1


        # ----------------------------------------
        # Enumerate local A/C combinations.
        #
        # Usually very small because the physical radius is only
        # 8 um. Keep exact enumeration rather than guessing one
        # nearest endpoint independently.
        # ----------------------------------------

        pair_rows = []

        for a_local in local_a:

            ai = end_idx[
                int(a_local)
            ]

            d_ab = float(
                np.linalg.norm(
                    pos[bi]
                    - pos[ai]
                )
            )

            for c_local in local_c:

                ci = start_idx[
                    int(c_local)
                ]

                d_bc = float(
                    np.linalg.norm(
                        pos[ci]
                        - pos[bi]
                    )
                )

                # Exact Stage-09 constant-velocity residual:
                #
                # A_expected = 2B - C
                #
                # residual = ||A - A_expected||
                #          = ||A + C - 2B||
                #
                # The same value is obtained from the opposite
                # direction.
                cv_residual = float(
                    np.linalg.norm(
                        pos[ai]
                        + pos[ci]
                        - 2.0 * pos[bi]
                    )
                )

                angle = (
                    turn_angle_deg_0910(
                        pos[ai],
                        pos[bi],
                        pos[ci],
                    )
                )

                balance = (
                    step_balance_0910(
                        d_ab,
                        d_bc,
                    )
                )

                pair_rows.append(
                    (
                        cv_residual,
                        max(
                            d_ab,
                            d_bc,
                        ),
                        angle,
                        balance,
                        ai,
                        ci,
                        d_ab,
                        d_bc,
                    )
                )


        # ----------------------------------------
        # Sort primarily by constant-velocity residual.
        # Tie-break using max step.
        # ----------------------------------------

        pair_rows.sort(
            key=lambda x: (
                x[0],
                x[1],
            )
        )

        best = pair_rows[0]

        second_residual = (
            float(
                pair_rows[1][0]
            )
            if len(pair_rows) > 1
            else np.inf
        )


        (
            best_residual,
            best_max_step,
            best_angle,
            best_balance,
            ai,
            ci,
            d_ab,
            d_bc,
        ) = best


        rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "a_id":
                int(
                    node_ids[ai]
                ),

            "b_id":
                int(
                    node_ids[bi]
                ),

            "c_id":
                int(
                    node_ids[ci]
                ),

            "t_mid":
                tb,

            "local_a_count":
                len(local_a),

            "local_c_count":
                len(local_c),

            "pair_count":
                len(pair_rows),

            "cv_residual_um":
                float(
                    best_residual
                ),

            "second_cv_residual_um":
                second_residual,

            "ambiguity_margin_um":
                (
                    second_residual
                    - float(
                        best_residual
                    )
                ),

            "d_ab_um":
                float(
                    d_ab
                ),

            "d_bc_um":
                float(
                    d_bc
                ),

            "max_step_um":
                float(
                    best_max_step
                ),

            "step_balance":
                float(
                    best_balance
                ),

            "turn_angle_deg":
                float(
                    best_angle
                ),

            "isolated_nodes_dataset":
                len(
                    isolated_idx
                ),
        })


    return (
        pd.DataFrame(
            rows
        ),
        {
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "isolated_nodes":
                len(
                    isolated_idx
                ),

            "isolated_with_local_pair":
                isolated_with_pair,
        },
    )


# ============================================================
# Full Fold0 census
# ============================================================

candidate_parts = []
dataset_stats = []

print(
    "=== 09.10 ISOLATED-NODE TWO-EDGE BRIDGE CENSUS ==="
)


for i, name in enumerate(
    dataset_names,
    start=1,
):

    df, stats = (
        census_isolated_bridges_0910(
            name
        )
    )

    if len(df):
        candidate_parts.append(
            df
        )

    dataset_stats.append(
        stats
    )

    if (
        i == 1
        or i % 10 == 0
        or i == len(
            dataset_names
        )
    ):
        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


assert candidate_parts

isolated_bridge_candidates_0910 = (
    pd.concat(
        candidate_parts,
        ignore_index=True,
    )
)

isolated_stats_0910 = (
    pd.DataFrame(
        dataset_stats
    )
)


# ============================================================
# Opportunity summary
# ============================================================

opportunity_0910 = (
    isolated_stats_0910
    .groupby(
        "prefix",
        as_index=False,
    )
    .agg(
        datasets=(
            "dataset",
            "size",
        ),

        isolated_nodes=(
            "isolated_nodes",
            "sum",
        ),

        isolated_with_local_pair=(
            "isolated_with_local_pair",
            "sum",
        ),
    )
)

opportunity_0910[
    "pair_opportunity_rate"
] = (
    opportunity_0910[
        "isolated_with_local_pair"
    ]
    /
    opportunity_0910[
        "isolated_nodes"
    ]
)


# ============================================================
# Geometry quantiles
# ============================================================

geometry_quantiles_0910 = (
    isolated_bridge_candidates_0910
    .groupby(
        "prefix"
    )
    [
        [
            "cv_residual_um",
            "max_step_um",
            "step_balance",
            "turn_angle_deg",
            "ambiguity_margin_um",
            "pair_count",
        ]
    ]
    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .round(4)
)


# ============================================================
# Residual diagnostic
#
# Same Stage-09 residual scale as single-edge recovery.
# No policy is frozen here.
# ============================================================

residual_rows = []

for prefix in [
    "44b6",
    "6bba",
]:

    sub = (
        isolated_bridge_candidates_0910[
            isolated_bridge_candidates_0910[
                "prefix"
            ] == prefix
        ]
    )

    for radius in [
        2.0,
        3.0,
        4.0,
        5.0,
    ]:

        q = sub[
            sub[
                "cv_residual_um"
            ] <= radius
        ]

        residual_rows.append({
            "prefix":
                prefix,

            "residual_radius_um":
                radius,

            "n":
                len(q),

            "fraction":
                (
                    len(q)
                    / len(sub)
                    if len(sub)
                    else np.nan
                ),

            "balance_le_0_5_rate":
                (
                    float(
                        (
                            q[
                                "step_balance"
                            ] <= 0.50
                        ).mean()
                    )
                    if len(q)
                    else np.nan
                ),

            "angle_le_60_rate":
                (
                    float(
                        (
                            q[
                                "turn_angle_deg"
                            ] <= 60.0
                        ).mean()
                    )
                    if len(q)
                    else np.nan
                ),

            "ambiguity_ge_1_rate":
                (
                    float(
                        (
                            q[
                                "ambiguity_margin_um"
                            ] >= 1.0
                        ).mean()
                    )
                    if len(q)
                    else np.nan
                ),

            "unique_pair_rate":
                (
                    float(
                        (
                            q[
                                "pair_count"
                            ] == 1
                        ).mean()
                    )
                    if len(q)
                    else np.nan
                ),
        })


residual_census_0910 = (
    pd.DataFrame(
        residual_rows
    )
)


# ============================================================
# Conservative diagnostic view
#
# Use current robust single-edge residual threshold r=4 um.
#
# IMPORTANT:
# This is diagnostic only, NOT yet a repair policy.
# ============================================================

conservative_bridge_0910 = (
    isolated_bridge_candidates_0910[
        (
            isolated_bridge_candidates_0910[
                "cv_residual_um"
            ] <= 4.0
        )
        &
        (
            isolated_bridge_candidates_0910[
                "max_step_um"
            ] <= 8.0
        )
        &
        (
            isolated_bridge_candidates_0910[
                "step_balance"
            ] <= 0.50
        )
        &
        (
            isolated_bridge_candidates_0910[
                "turn_angle_deg"
            ] <= 60.0
        )
        &
        (
            isolated_bridge_candidates_0910[
                "ambiguity_margin_um"
            ] >= 1.0
        )
    ]
    .copy()
)


conservative_counts_0910 = (
    conservative_bridge_0910
    .groupby(
        "prefix",
        as_index=False,
    )
    .agg(
        candidates=(
            "b_id",
            "size",
        ),

        datasets_with_candidate=(
            "dataset",
            "nunique",
        ),

        median_cv_residual_um=(
            "cv_residual_um",
            "median",
        ),

        median_pair_count=(
            "pair_count",
            "median",
        ),
    )
)


# ============================================================
# Potential endpoint conflicts
#
# If multiple isolated B nodes propose the same A or C, later
# repair cannot simply add all candidates independently.
# ============================================================

conflict_view = (
    conservative_bridge_0910
    .copy()
)

if len(conflict_view):

    a_use = (
        conflict_view
        .groupby(
            [
                "dataset",
                "a_id",
            ]
        )[
            "b_id"
        ]
        .transform(
            "nunique"
        )
    )

    c_use = (
        conflict_view
        .groupby(
            [
                "dataset",
                "c_id",
            ]
        )[
            "b_id"
        ]
        .transform(
            "nunique"
        )
    )

    conflict_view[
        "conflict_free"
    ] = (
        (a_use == 1)
        &
        (c_use == 1)
    )

    conflict_summary_0910 = (
        conflict_view
        .groupby(
            "prefix",
            as_index=False,
        )
        .agg(
            candidates=(
                "b_id",
                "size",
            ),

            conflict_free=(
                "conflict_free",
                "sum",
            ),

            conflict_free_rate=(
                "conflict_free",
                "mean",
            ),
        )
    )

else:

    conflict_summary_0910 = (
        pd.DataFrame()
    )


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.10 RESULT")
print("=" * 72)

print()
print("ISOLATED-NODE OPPORTUNITIES")

display(
    opportunity_0910
)

print()
print("BEST-PAIR GEOMETRY QUANTILES")

display(
    geometry_quantiles_0910
)

print()
print("RESIDUAL CENSUS")

display(
    residual_census_0910
)

print()
print(
    "CONSERVATIVE TWO-EDGE BRIDGE VIEW "
    "(diagnostic only)"
)

display(
    conservative_counts_0910
)

print()
print("ENDPOINT-CONFLICT AUDIT")

display(
    conflict_summary_0910
)

print()
print(
    "Total isolated nodes:",
    int(
        isolated_stats_0910[
            "isolated_nodes"
        ].sum()
    ),
)

print(
    "Isolated nodes with local A/C pair:",
    len(
        isolated_bridge_candidates_0910
    ),
)

print(
    "Conservative bridge candidates:",
    len(
        conservative_bridge_0910
    ),
)

print()
print(
    "No edge was added."
)

print(
    "No graph/file was modified."
)

print(
    "No GT was used."
)

print()
print(
    "09.10 isolated-node bridge census: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.11 Existing-Node Two-Edge Bridge Oracle Audit
# Purpose:
#   Use locked official prediction->GT node matching to measure
#   whether conservative isolated-node A->B->C bridge proposals
#   correspond to true GT lineage structure.
#
#   Primary oracle condition:
#       A, B, C are all officially matched
#       AND GT contains A*->B*
#       AND GT contains B*->C*
#
#   GT is used ONLY for Fold0 oracle analysis.
#   Candidate generation itself remains prediction-only.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 oracle analysis only
# Keep after running: YES — formal Stage-09 bridge evidence
# ============================================================

import numpy as np
import pandas as pd

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
)


# ============================================================
# Preconditions
# ============================================================

assert "conservative_bridge_0910" in globals()
assert "WARM2_0995" in globals()
assert "WARM2_09975" in globals()
assert "TRAIN_DIR" in globals()
assert "dataset_names" in globals()
assert "prefix_map" in globals()

assert "load_graph" in globals()
assert "apply_frozen_g2" in globals()
assert "apply_low_conf_component_pruning" in globals()
assert "node_coordinate_keys" in globals()

assert len(conservative_bridge_0910) == 6932


# ============================================================
# Frozen pre-cleanup state
# ============================================================

G2_P2_MIN = 0.80
CONF_COMPONENT_LIMIT = 9


# ============================================================
# Rebuild candidate policy table
#
# No oracle / GT-derived fields are allowed here.
# ============================================================

bridges = (
    conservative_bridge_0910[
        [
            "dataset",
            "prefix",
            "a_id",
            "b_id",
            "c_id",
            "t_mid",
            "cv_residual_um",
            "second_cv_residual_um",
            "ambiguity_margin_um",
            "max_step_um",
            "step_balance",
            "turn_angle_deg",
            "pair_count",
        ]
    ]
    .copy()
)

for col in [
    "a_id",
    "b_id",
    "c_id",
]:
    bridges[col] = (
        bridges[col]
        .astype(int)
    )


# ============================================================
# Conflict-free flag
#
# A cannot feed multiple isolated B candidates.
# C cannot receive multiple isolated B candidates.
# ============================================================

a_uses = (
    bridges
    .groupby(
        [
            "dataset",
            "a_id",
        ]
    )["b_id"]
    .transform("nunique")
)

c_uses = (
    bridges
    .groupby(
        [
            "dataset",
            "c_id",
        ]
    )["b_id"]
    .transform("nunique")
)

bridges[
    "conflict_free"
] = (
    (a_uses == 1)
    &
    (c_uses == 1)
)

bridges[
    "unique_pair"
] = (
    bridges[
        "pair_count"
    ] == 1
)

bridges[
    "conflict_free_unique"
] = (
    bridges[
        "conflict_free"
    ]
    &
    bridges[
        "unique_pair"
    ]
)


# ============================================================
# Official matching
# ============================================================

oracle_parts = []

print(
    "=== 09.11 TWO-EDGE BRIDGE ORACLE AUDIT ==="
)


for i, name in enumerate(
    dataset_names,
    start=1,
):

    candidate_df = (
        bridges[
            bridges["dataset"]
            == name
        ]
        .copy()
    )

    if not len(candidate_df):
        continue


    # --------------------------------------------------------
    # Fresh pre-cleanup baseline-v8 graph
    # --------------------------------------------------------

    pred_g = load_graph(
        WARM2_0995
        / f"{name}.geff"
    )

    high_g = load_graph(
        WARM2_09975
        / f"{name}.geff"
    )

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    scale = np.asarray(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    apply_frozen_g2(
        pred_g,
        scale,
    )

    high_keys = (
        node_coordinate_keys(
            high_g
        )
    )

    apply_low_conf_component_pruning(
        pred_g,
        high_keys,
        CONF_COMPONENT_LIMIT,
    )


    # --------------------------------------------------------
    # Locked official node matching
    # --------------------------------------------------------

    official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )

    matched = (
        pred_g
        .node_attrs(
            unpack=True
        )
        .select(
            [
                "node_id",
                "match_node_id",
                "match_score",
            ]
        )
        .to_pandas()
    )

    match_map = dict(
        zip(
            matched[
                "node_id"
            ].astype(int),
            matched[
                "match_node_id"
            ].astype(int),
        )
    )


    # --------------------------------------------------------
    # Prediction -> GT node IDs
    # --------------------------------------------------------

    for role in [
        "a",
        "b",
        "c",
    ]:

        candidate_df[
            f"gt_{role}_id"
        ] = [
            match_map.get(
                int(node_id),
                -1,
            )
            for node_id in candidate_df[
                f"{role}_id"
            ]
        ]


    # --------------------------------------------------------
    # Observable structure
    # --------------------------------------------------------

    candidate_df[
        "a_observable"
    ] = (
        candidate_df[
            "gt_a_id"
        ] >= 0
    )

    candidate_df[
        "b_observable"
    ] = (
        candidate_df[
            "gt_b_id"
        ] >= 0
    )

    candidate_df[
        "c_observable"
    ] = (
        candidate_df[
            "gt_c_id"
        ] >= 0
    )

    candidate_df[
        "left_observable"
    ] = (
        candidate_df[
            "a_observable"
        ]
        &
        candidate_df[
            "b_observable"
        ]
    )

    candidate_df[
        "right_observable"
    ] = (
        candidate_df[
            "b_observable"
        ]
        &
        candidate_df[
            "c_observable"
        ]
    )

    candidate_df[
        "bridge_observable"
    ] = (
        candidate_df[
            "a_observable"
        ]
        &
        candidate_df[
            "b_observable"
        ]
        &
        candidate_df[
            "c_observable"
        ]
    )


    # --------------------------------------------------------
    # GT edge truth
    # --------------------------------------------------------

    left_truth = []
    right_truth = []

    for row in (
        candidate_df.itertuples()
    ):

        if row.left_observable:

            left_truth.append(
                bool(
                    gt_g.has_edge(
                        int(
                            row.gt_a_id
                        ),
                        int(
                            row.gt_b_id
                        ),
                    )
                )
            )

        else:

            left_truth.append(
                False
            )


        if row.right_observable:

            right_truth.append(
                bool(
                    gt_g.has_edge(
                        int(
                            row.gt_b_id
                        ),
                        int(
                            row.gt_c_id
                        ),
                    )
                )
            )

        else:

            right_truth.append(
                False
            )


    candidate_df[
        "left_true"
    ] = left_truth

    candidate_df[
        "right_true"
    ] = right_truth

    candidate_df[
        "bridge_true"
    ] = (
        candidate_df[
            "bridge_observable"
        ]
        &
        candidate_df[
            "left_true"
        ]
        &
        candidate_df[
            "right_true"
        ]
    )


    oracle_parts.append(
        candidate_df
    )


    if (
        i == 1
        or i % 10 == 0
        or i == len(dataset_names)
    ):
        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


oracle_bridges_0911 = (
    pd.concat(
        oracle_parts,
        ignore_index=True,
    )
)

assert (
    len(oracle_bridges_0911)
    == 6932
)


# ============================================================
# Scope definitions
# ============================================================

SCOPES_0911 = {
    "CONSERVATIVE_ALL":
        np.ones(
            len(
                oracle_bridges_0911
            ),
            dtype=bool,
        ),

    "CONFLICT_FREE":
        oracle_bridges_0911[
            "conflict_free"
        ].to_numpy(
            dtype=bool
        ),

    "UNIQUE_PAIR":
        oracle_bridges_0911[
            "unique_pair"
        ].to_numpy(
            dtype=bool
        ),

    "CONFLICT_FREE_UNIQUE":
        oracle_bridges_0911[
            "conflict_free_unique"
        ].to_numpy(
            dtype=bool
        ),
}


# ============================================================
# Summary
# ============================================================

def summarize_bridge_scope(
    df,
    mask,
    scope,
    prefix="ALL",
):

    sub = df.loc[
        mask
    ].copy()

    if prefix != "ALL":
        sub = sub[
            sub["prefix"]
            == prefix
        ]

    n = len(sub)

    a_obs = int(
        sub[
            "a_observable"
        ].sum()
    )

    b_obs = int(
        sub[
            "b_observable"
        ].sum()
    )

    c_obs = int(
        sub[
            "c_observable"
        ].sum()
    )

    left_obs = int(
        sub[
            "left_observable"
        ].sum()
    )

    right_obs = int(
        sub[
            "right_observable"
        ].sum()
    )

    bridge_obs = int(
        sub[
            "bridge_observable"
        ].sum()
    )

    left_true = int(
        sub[
            "left_true"
        ].sum()
    )

    right_true = int(
        sub[
            "right_true"
        ].sum()
    )

    bridge_true = int(
        sub[
            "bridge_true"
        ].sum()
    )


    return {
        "scope":
            scope,

        "prefix":
            prefix,

        "candidates":
            n,

        "A_observable":
            a_obs,

        "B_observable":
            b_obs,

        "C_observable":
            c_obs,

        "full_bridge_observable":
            bridge_obs,

        "full_observable_rate":
            (
                bridge_obs / n
                if n
                else np.nan
            ),

        "left_observable_edges":
            left_obs,

        "left_true_edges":
            left_true,

        "left_precision":
            (
                left_true
                / left_obs
                if left_obs
                else np.nan
            ),

        "right_observable_edges":
            right_obs,

        "right_true_edges":
            right_true,

        "right_precision":
            (
                right_true
                / right_obs
                if right_obs
                else np.nan
            ),

        "true_full_bridges":
            bridge_true,

        "full_bridge_precision":
            (
                bridge_true
                / bridge_obs
                if bridge_obs
                else np.nan
            ),
    }


summary_rows_0911 = []

for (
    scope,
    mask
) in SCOPES_0911.items():

    summary_rows_0911.append(
        summarize_bridge_scope(
            oracle_bridges_0911,
            mask,
            scope,
            "ALL",
        )
    )

    for prefix in [
        "44b6",
        "6bba",
    ]:

        summary_rows_0911.append(
            summarize_bridge_scope(
                oracle_bridges_0911,
                mask,
                scope,
                prefix,
            )
        )


oracle_summary_0911 = (
    pd.DataFrame(
        summary_rows_0911
    )
)


# ============================================================
# Observable full bridges — geometry comparison
#
# Compare TRUE vs FALSE only among fully observable candidates.
# This is diagnostic; no new threshold is selected here.
# ============================================================

obs = (
    oracle_bridges_0911[
        oracle_bridges_0911[
            "bridge_observable"
        ]
    ]
    .copy()
)

geometry_rows_0911 = []

for truth_label, sub in [
    (
        "TRUE_BRIDGE",
        obs[
            obs["bridge_true"]
        ],
    ),
    (
        "FALSE_BRIDGE",
        obs[
            ~obs["bridge_true"]
        ],
    ),
]:

    if not len(sub):
        continue

    geometry_rows_0911.append({
        "class":
            truth_label,

        "n":
            len(sub),

        "median_residual_um":
            float(
                sub[
                    "cv_residual_um"
                ].median()
            ),

        "median_max_step_um":
            float(
                sub[
                    "max_step_um"
                ].median()
            ),

        "median_step_balance":
            float(
                sub[
                    "step_balance"
                ].median()
            ),

        "median_turn_angle_deg":
            float(
                sub[
                    "turn_angle_deg"
                ].median()
            ),

        "median_ambiguity_margin_um":
            float(
                sub[
                    "ambiguity_margin_um"
                ].median()
            ),

        "unique_pair_rate":
            float(
                sub[
                    "unique_pair"
                ].mean()
            ),
    })


geometry_summary_0911 = (
    pd.DataFrame(
        geometry_rows_0911
    )
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.11 RESULT")
print("=" * 72)

print()
print("BRIDGE ORACLE SUMMARY")

display(
    oracle_summary_0911
)

print()
print(
    "FULLY OBSERVABLE TRUE/FALSE GEOMETRY"
)

display(
    geometry_summary_0911
)

print()
print(
    "Total conservative candidates:",
    len(
        oracle_bridges_0911
    ),
)

print(
    "Conflict-free candidates:",
    int(
        oracle_bridges_0911[
            "conflict_free"
        ].sum()
    ),
)

print(
    "Unique-pair candidates:",
    int(
        oracle_bridges_0911[
            "unique_pair"
        ].sum()
    ),
)

print(
    "Conflict-free + unique:",
    int(
        oracle_bridges_0911[
            "conflict_free_unique"
        ].sum()
    ),
)

print()
print(
    "No edge was added."
)

print(
    "No graph/file was modified."
)

print(
    "GT matching was used only for oracle analysis."
)

print()
print(
    "09.11 two-edge bridge oracle audit: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.12 Two-Edge Bridge Repair Ablation on Baseline-v8
# Purpose:
#   Test conservative conflict-free isolated-node A->B->C
#   two-edge bridge recovery ON TOP OF the frozen baseline-v8
#   single-edge repair policy.
#
# Baseline-v8:
#   Warm2 det=.995
#   -> G2 p2=.80
#   -> confidence pruning K=9
#   -> mutual-conservative single-edge repair, residual <= 4 um
#   -> frozen cleanup
#
# Candidate:
#   Warm2 det=.995
#   -> G2 p2=.80
#   -> confidence pruning K=9
#   -> frozen single-edge repair, residual <= 4 um
#   -> conservative conflict-free two-edge A->B->C repair
#   -> frozen cleanup
#
# Two-edge candidate requirements:
#   B is an existing isolated node
#   residual <= 4 um
#   max step <= 8 um
#   step balance <= .50
#   turn angle <= 60 deg
#   ambiguity margin >= 1 um
#   A/C endpoint usage conflict-free
#
# Interaction rule:
#   If frozen single-edge repair already consumes A, B, or C
#   such that A->B->C is no longer degree-valid, skip bridge.
#
# Changes files: NO — in-memory graphs only
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only; repair policy uses NO GT
# Keep after running: YES — formal Stage-09 bridge ablation
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "conservative_bridge_0910" in globals()
assert "build_mutual_policy_0908" in globals()
assert "prepare_precleanup_graph" in globals()
assert "add_available_repairs_0907" in globals()
assert "apply_frozen_component_cleanup" in globals()
assert "evaluate_graph_0906" in globals()
assert "recovered_edge_dist" in globals()

FROZEN_BASELINE_V8_SCORE = 0.769647


# ============================================================
# Frozen single-edge baseline-v8 policy
# ============================================================

single_policy_0912 = (
    build_mutual_policy_0908(
        4.0
    )
)

assert len(single_policy_0912) == 3945


# ============================================================
# Prediction-only two-edge bridge policy
#
# Recompute conflict-free flag from candidate table only.
# No oracle columns are used.
# ============================================================

two_policy_0912 = (
    conservative_bridge_0910[
        [
            "dataset",
            "prefix",
            "a_id",
            "b_id",
            "c_id",
            "cv_residual_um",
            "max_step_um",
            "step_balance",
            "turn_angle_deg",
            "ambiguity_margin_um",
            "pair_count",
        ]
    ]
    .copy()
)

for col in [
    "a_id",
    "b_id",
    "c_id",
]:
    two_policy_0912[col] = (
        two_policy_0912[col]
        .astype(int)
    )


a_usage = (
    two_policy_0912
    .groupby(
        [
            "dataset",
            "a_id",
        ]
    )["b_id"]
    .transform("nunique")
)

c_usage = (
    two_policy_0912
    .groupby(
        [
            "dataset",
            "c_id",
        ]
    )["b_id"]
    .transform("nunique")
)

two_policy_0912[
    "conflict_free"
] = (
    (a_usage == 1)
    &
    (c_usage == 1)
)

two_policy_0912 = (
    two_policy_0912[
        two_policy_0912[
            "conflict_free"
        ]
    ]
    .copy()
)

assert len(two_policy_0912) == 6468


# ============================================================
# Oracle leakage guard
# ============================================================

FORBIDDEN_0912 = {
    "gt_a_id",
    "gt_b_id",
    "gt_c_id",
    "bridge_true",
    "bridge_observable",
    "left_true",
    "right_true",
}

assert not (
    FORBIDDEN_0912
    & set(two_policy_0912.columns)
)


# ============================================================
# Two-edge bridge insertion helper
# ============================================================

def add_two_edge_bridges_0912(
    graph,
    candidate_bridges,
):
    """
    Add A->B and B->C only when the bridge remains structurally
    valid after frozen single-edge repair.

    Required current topology:
        out_degree(A) = 0
        in_degree(B)  = 0
        out_degree(B) = 0
        in_degree(C)  = 0

    No forced rewiring.
    """

    node_table = (
        graph
        .node_attrs(
            unpack=True
        )
        .to_pandas()
        .set_index("node_id")
    )

    stats = {
        "requested_bridges":
            len(candidate_bridges),

        "added_bridges":
            0,

        "edges_added":
            0,

        "missing_endpoint":
            0,

        "degree_conflict":
            0,

        "existing_edge_conflict":
            0,
    }


    for row in candidate_bridges.itertuples(
        index=False
    ):

        a_id = int(row.a_id)
        b_id = int(row.b_id)
        c_id = int(row.c_id)


        # ----------------------------------------
        # All three existing nodes must survive
        # G2 + K9 and single-edge repair stage.
        # ----------------------------------------

        if not (
            graph.has_node(a_id)
            and graph.has_node(b_id)
            and graph.has_node(c_id)
        ):
            stats[
                "missing_endpoint"
            ] += 1
            continue


        # ----------------------------------------
        # Do not duplicate edges.
        # ----------------------------------------

        if (
            graph.has_edge(
                a_id,
                b_id,
            )
            or graph.has_edge(
                b_id,
                c_id,
            )
        ):
            stats[
                "existing_edge_conflict"
            ] += 1
            continue


        # ----------------------------------------
        # Strict topology gate after interaction
        # with frozen single-edge repair.
        # ----------------------------------------

        if not (
            graph.out_degree(a_id) == 0
            and graph.in_degree(b_id) == 0
            and graph.out_degree(b_id) == 0
            and graph.in_degree(c_id) == 0
        ):
            stats[
                "degree_conflict"
            ] += 1
            continue


        # ----------------------------------------
        # Temporal consistency
        # ----------------------------------------

        ta = int(
            node_table.loc[
                a_id,
                "t",
            ]
        )

        tb = int(
            node_table.loc[
                b_id,
                "t",
            ]
        )

        tc = int(
            node_table.loc[
                c_id,
                "t",
            ]
        )

        assert (
            tb == ta + 1
            and tc == tb + 1
        )


        # ----------------------------------------
        # Verified model-grid edge_dist
        # ----------------------------------------

        dist_ab = (
            recovered_edge_dist(
                node_table,
                a_id,
                b_id,
            )
        )

        dist_bc = (
            recovered_edge_dist(
                node_table,
                b_id,
                c_id,
            )
        )


        graph.add_edge(
            a_id,
            b_id,
            {
                "edge_dist":
                    dist_ab,

                "edge_prob":
                    0.0,
            },
        )

        graph.add_edge(
            b_id,
            c_id,
            {
                "edge_dist":
                    dist_bc,

                "edge_prob":
                    0.0,
            },
        )


        stats[
            "added_bridges"
        ] += 1

        stats[
            "edges_added"
        ] += 2


    return stats


# ============================================================
# Fold0 evaluation
# ============================================================

baseline_v8_records_0912 = []
two_edge_records_0912 = []

rows_0912 = []

single_edges_added_total = 0

two_requested_total = 0
two_added_total = 0
two_edges_added_total = 0

two_missing_total = 0
two_degree_conflict_total = 0
two_existing_conflict_total = 0

baseline_cleanup_removed_total = 0
two_cleanup_removed_total = 0


print(
    "=== 09.12 TWO-EDGE BRIDGE ABLATION ==="
)


for i, name in enumerate(
    dataset_names,
    start=1,
):

    single_candidates = (
        single_policy_0912[
            single_policy_0912[
                "dataset"
            ] == name
        ]
    )

    bridge_candidates = (
        two_policy_0912[
            two_policy_0912[
                "dataset"
            ] == name
        ]
    )


    # ========================================================
    # A. Frozen baseline-v8
    # ========================================================

    base_g, scale = (
        prepare_precleanup_graph(
            name
        )
    )

    single_stats_base = (
        add_available_repairs_0907(
            base_g,
            single_candidates,
        )
    )

    assert (
        single_stats_base["added"]
        == len(single_candidates)
    )

    base_cleanup = (
        apply_frozen_component_cleanup(
            base_g,
            scale,
        )
    )

    baseline_cleanup_removed_total += int(
        base_cleanup[
            "nodes_removed"
        ]
    )

    gt_base = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    base_m = evaluate_graph_0906(
        base_g,
        gt_base,
        scale,
        name,
    )

    baseline_v8_records_0912.append(
        base_m
    )


    # ========================================================
    # B. baseline-v8 + two-edge bridge
    # ========================================================

    test_g, test_scale = (
        prepare_precleanup_graph(
            name
        )
    )

    assert np.allclose(
        scale,
        test_scale,
    )


    # Frozen single-edge repair first.
    single_stats = (
        add_available_repairs_0907(
            test_g,
            single_candidates,
        )
    )

    assert (
        single_stats["added"]
        == len(single_candidates)
    )

    single_edges_added_total += (
        single_stats["added"]
    )


    # Then conflict-free two-edge bridge repair.
    two_stats = (
        add_two_edge_bridges_0912(
            test_g,
            bridge_candidates,
        )
    )

    two_requested_total += (
        two_stats[
            "requested_bridges"
        ]
    )

    two_added_total += (
        two_stats[
            "added_bridges"
        ]
    )

    two_edges_added_total += (
        two_stats[
            "edges_added"
        ]
    )

    two_missing_total += (
        two_stats[
            "missing_endpoint"
        ]
    )

    two_degree_conflict_total += (
        two_stats[
            "degree_conflict"
        ]
    )

    two_existing_conflict_total += (
        two_stats[
            "existing_edge_conflict"
        ]
    )


    # Frozen cleanup after BOTH recovery stages.
    test_cleanup = (
        apply_frozen_component_cleanup(
            test_g,
            test_scale,
        )
    )

    two_cleanup_removed_total += int(
        test_cleanup[
            "nodes_removed"
        ]
    )


    gt_test = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    test_m = evaluate_graph_0906(
        test_g,
        gt_test,
        test_scale,
        name,
    )

    two_edge_records_0912.append(
        test_m
    )


    # ========================================================
    # Paired evidence
    # ========================================================

    rows_0912.append({
        "dataset":
            name,

        "prefix":
            prefix_map[name],

        "two_requested":
            two_stats[
                "requested_bridges"
            ],

        "two_added":
            two_stats[
                "added_bridges"
            ],

        "degree_conflict":
            two_stats[
                "degree_conflict"
            ],

        "existing_edge_conflict":
            two_stats[
                "existing_edge_conflict"
            ],

        "delta_edge_jaccard":
            (
                test_m[
                    "edge_jaccard"
                ]
                - base_m[
                    "edge_jaccard"
                ]
            ),

        "delta_adj_edge_jaccard":
            (
                test_m[
                    "adj_edge_jaccard"
                ]
                - base_m[
                    "adj_edge_jaccard"
                ]
            ),

        "delta_recall":
            (
                test_m[
                    "node_recall"
                ]
                - base_m[
                    "node_recall"
                ]
            ),

        "delta_node_ratio":
            (
                test_m[
                    "total_node_ratio"
                ]
                - base_m[
                    "total_node_ratio"
                ]
            ),
    })


    if (
        i == 1
        or i % 10 == 0
        or i == len(dataset_names)
    ):
        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


# ============================================================
# Aggregate
# ============================================================

base_s_0912 = summarise(
    baseline_v8_records_0912
)

two_s_0912 = summarise(
    two_edge_records_0912
)


def summary_row_0912(
    policy,
    summary,
    records,
):

    return {
        "policy":
            policy,

        "edge_jaccard":
            summary[
                "edge_jaccard"
            ],

        "adj_edge_jaccard":
            summary[
                "adj_edge_jaccard"
            ],

        "division_jaccard":
            summary[
                "division_jaccard"
            ],

        "node_recall":
            summary[
                "node_recall"
            ],

        "median_node_ratio":
            float(
                np.median([
                    m[
                        "total_node_ratio"
                    ]
                    for m in records
                ])
            ),

        "score":
            summary[
                "score"
            ],
    }


overall_0912 = pd.DataFrame([
    summary_row_0912(
        "baseline_v8_single_edge",
        base_s_0912,
        baseline_v8_records_0912,
    ),

    summary_row_0912(
        "baseline_v8_plus_two_edge",
        two_s_0912,
        two_edge_records_0912,
    ),
])


# ============================================================
# Reproduction gate
# ============================================================

base_score_0912 = float(
    base_s_0912[
        "score"
    ]
)

two_score_0912 = float(
    two_s_0912[
        "score"
    ]
)

assert abs(
    base_score_0912
    - FROZEN_BASELINE_V8_SCORE
) < 5e-6, (
    f"baseline-v8 reproduction failed: "
    f"{base_score_0912:.6f}"
)


# ============================================================
# Prefix aggregate
# ============================================================

prefix_rows_0912 = []

for prefix in [
    "44b6",
    "6bba",
]:

    idx = [
        j
        for j, name in enumerate(
            dataset_names
        )
        if prefix_map[name]
        == prefix
    ]

    variants = {
        "baseline_v8": [
            baseline_v8_records_0912[j]
            for j in idx
        ],

        "plus_two_edge": [
            two_edge_records_0912[j]
            for j in idx
        ],
    }

    for policy, recs in (
        variants.items()
    ):

        s = summarise(
            recs
        )

        prefix_rows_0912.append({
            "prefix":
                prefix,

            "policy":
                policy,

            "edge_jaccard":
                s[
                    "edge_jaccard"
                ],

            "adj_edge_jaccard":
                s[
                    "adj_edge_jaccard"
                ],

            "division_jaccard":
                s[
                    "division_jaccard"
                ],

            "node_recall":
                s[
                    "node_recall"
                ],

            "score":
                s[
                    "score"
                ],
        })


prefix_0912 = pd.DataFrame(
    prefix_rows_0912
)


# ============================================================
# Paired robustness
# ============================================================

paired_0912 = pd.DataFrame(
    rows_0912
)

paired_summary_rows_0912 = []

for prefix in [
    "ALL",
    "44b6",
    "6bba",
]:

    sub = (
        paired_0912
        if prefix == "ALL"
        else paired_0912[
            paired_0912[
                "prefix"
            ] == prefix
        ]
    )

    paired_summary_rows_0912.append({
        "prefix":
            prefix,

        "datasets":
            len(sub),

        "bridges_requested":
            int(
                sub[
                    "two_requested"
                ].sum()
            ),

        "bridges_added":
            int(
                sub[
                    "two_added"
                ].sum()
            ),

        "edgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "adjEdgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "mean_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                sub[
                    "delta_recall"
                ].mean()
            ),

        "mean_delta_node_ratio":
            float(
                sub[
                    "delta_node_ratio"
                ].mean()
            ),
    })


paired_summary_0912 = (
    pd.DataFrame(
        paired_summary_rows_0912
    )
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.12 RESULT")
print("=" * 72)

print()
print("OVERALL")

display(
    overall_0912
)

print(
    "Two-edge - baseline-v8 score:",
    f"{two_score_0912 - base_score_0912:+.6f}",
)

print()
print("BY PREFIX")

display(
    prefix_0912
)

print()
print("PAIRED ROBUSTNESS")

display(
    paired_summary_0912
)

print()
print("INTERACTION / CLEANUP COUNTS")

print(
    "Two-edge bridges requested:",
    two_requested_total,
)

print(
    "Two-edge bridges successfully added:",
    two_added_total,
)

print(
    "Actual two-edge associations added:",
    two_edges_added_total,
)

print(
    "Missing endpoint skips:",
    two_missing_total,
)

print(
    "Degree-conflict skips:",
    two_degree_conflict_total,
)

print(
    "Existing-edge conflict skips:",
    two_existing_conflict_total,
)

print()
print(
    "baseline-v8 cleanup nodes removed:",
    baseline_cleanup_removed_total,
)

print(
    "plus-two-edge cleanup nodes removed:",
    two_cleanup_removed_total,
)

print(
    "Additional nodes rescued:",
    (
        baseline_cleanup_removed_total
        - two_cleanup_removed_total
    ),
)

print()
print(
    "baseline-v8 reproduction:",
    f"{base_score_0912:.6f}",
)

print()
print(
    "No GEFF file was modified."
)

print(
    "Two-edge policy used NO GT information."
)

print()
print(
    "09.12 two-edge bridge ablation: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.13 Strict Two-Edge Bridge Robustness
# Purpose:
#   Test a stricter two-edge bridge policy that requires:
#
#       conservative geometry
#       endpoint conflict-free
#       pair_count == 1
#
#   Compare against:
#       baseline-v8
#       current conflict-free two-edge policy from 09.12
#
#   Frozen single-edge repair r<=4 um always runs first.
#
# Changes files: NO — in-memory graphs only
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only; repair policy uses NO GT
# Keep after running: YES — formal Stage-09 robustness evidence
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "conservative_bridge_0910" in globals()
assert "single_policy_0912" in globals()
assert "baseline_v8_records_0912" in globals()
assert "two_edge_records_0912" in globals()

assert "prepare_precleanup_graph" in globals()
assert "add_available_repairs_0907" in globals()
assert "add_two_edge_bridges_0912" in globals()
assert "apply_frozen_component_cleanup" in globals()
assert "evaluate_graph_0906" in globals()


# ============================================================
# Build prediction-only strict bridge policy
# ============================================================

strict_policy_0913 = (
    conservative_bridge_0910[
        [
            "dataset",
            "prefix",
            "a_id",
            "b_id",
            "c_id",
            "pair_count",
        ]
    ]
    .copy()
)

for col in [
    "a_id",
    "b_id",
    "c_id",
]:
    strict_policy_0913[col] = (
        strict_policy_0913[col]
        .astype(int)
    )


# ------------------------------------------------------------
# Same endpoint conflict-free definition as 09.12
# ------------------------------------------------------------

a_usage = (
    strict_policy_0913
    .groupby(
        [
            "dataset",
            "a_id",
        ]
    )["b_id"]
    .transform("nunique")
)

c_usage = (
    strict_policy_0913
    .groupby(
        [
            "dataset",
            "c_id",
        ]
    )["b_id"]
    .transform("nunique")
)

strict_policy_0913[
    "conflict_free"
] = (
    (a_usage == 1)
    &
    (c_usage == 1)
)


strict_policy_0913 = (
    strict_policy_0913[
        strict_policy_0913[
            "conflict_free"
        ]
        &
        (
            strict_policy_0913[
                "pair_count"
            ] == 1
        )
    ]
    .copy()
)


assert len(strict_policy_0913) == 2289, (
    f"Expected 2289 strict bridges; "
    f"got {len(strict_policy_0913)}"
)


# ============================================================
# Leakage guard
# ============================================================

FORBIDDEN = {
    "gt_a_id",
    "gt_b_id",
    "gt_c_id",
    "bridge_true",
    "bridge_observable",
}

assert not (
    FORBIDDEN
    & set(strict_policy_0913.columns)
)


# ============================================================
# Evaluate strict policy
# ============================================================

strict_records_0913 = []
rows_0913 = []

requested_total = 0
added_total = 0
edges_added_total = 0
degree_conflict_total = 0
existing_conflict_total = 0
cleanup_removed_total = 0


print(
    "=== 09.13 STRICT TWO-EDGE ROBUSTNESS ==="
)


for i, name in enumerate(
    dataset_names
):

    # --------------------------------------------------------
    # Fresh G2 + K9 graph
    # --------------------------------------------------------

    g, scale = (
        prepare_precleanup_graph(
            name
        )
    )


    # --------------------------------------------------------
    # Frozen baseline-v8 single-edge repair first
    # --------------------------------------------------------

    single_candidates = (
        single_policy_0912[
            single_policy_0912[
                "dataset"
            ] == name
        ]
    )

    single_stats = (
        add_available_repairs_0907(
            g,
            single_candidates,
        )
    )

    assert (
        single_stats["added"]
        == len(single_candidates)
    )


    # --------------------------------------------------------
    # Strict two-edge bridge
    # --------------------------------------------------------

    bridge_candidates = (
        strict_policy_0913[
            strict_policy_0913[
                "dataset"
            ] == name
        ]
    )

    stats = (
        add_two_edge_bridges_0912(
            g,
            bridge_candidates,
        )
    )

    requested_total += (
        stats["requested_bridges"]
    )

    added_total += (
        stats["added_bridges"]
    )

    edges_added_total += (
        stats["edges_added"]
    )

    degree_conflict_total += (
        stats["degree_conflict"]
    )

    existing_conflict_total += (
        stats[
            "existing_edge_conflict"
        ]
    )


    # --------------------------------------------------------
    # Frozen cleanup
    # --------------------------------------------------------

    cleanup = (
        apply_frozen_component_cleanup(
            g,
            scale,
        )
    )

    cleanup_removed_total += int(
        cleanup[
            "nodes_removed"
        ]
    )


    # --------------------------------------------------------
    # Locked metric
    # --------------------------------------------------------

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    m = evaluate_graph_0906(
        g,
        gt_g,
        scale,
        name,
    )

    strict_records_0913.append(
        m
    )


    # --------------------------------------------------------
    # Compare with the exact baseline-v8 record from 09.12
    # --------------------------------------------------------

    base_m = (
        baseline_v8_records_0912[i]
    )

    rows_0913.append({
        "dataset":
            name,

        "prefix":
            prefix_map[name],

        "bridges_requested":
            stats[
                "requested_bridges"
            ],

        "bridges_added":
            stats[
                "added_bridges"
            ],

        "delta_edge_jaccard":
            (
                m["edge_jaccard"]
                - base_m["edge_jaccard"]
            ),

        "delta_adj_edge_jaccard":
            (
                m["adj_edge_jaccard"]
                - base_m[
                    "adj_edge_jaccard"
                ]
            ),

        "delta_recall":
            (
                m["node_recall"]
                - base_m["node_recall"]
            ),

        "delta_node_ratio":
            (
                m["total_node_ratio"]
                - base_m[
                    "total_node_ratio"
                ]
            ),
    })


# ============================================================
# Aggregate
# ============================================================

strict_s = summarise(
    strict_records_0913
)

base_s = summarise(
    baseline_v8_records_0912
)

current_s = summarise(
    two_edge_records_0912
)


def make_row(
    policy,
    s,
    records,
):

    return {
        "policy":
            policy,

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    }


overall_0913 = pd.DataFrame([
    make_row(
        "baseline_v8",
        base_s,
        baseline_v8_records_0912,
    ),

    make_row(
        "two_edge_conflict_free",
        current_s,
        two_edge_records_0912,
    ),

    make_row(
        "two_edge_conflict_free_unique",
        strict_s,
        strict_records_0913,
    ),
])


# ============================================================
# Prefix aggregate
# ============================================================

prefix_rows = []

for prefix in [
    "44b6",
    "6bba",
]:

    idx = [
        j
        for j, name in enumerate(
            dataset_names
        )
        if prefix_map[name]
        == prefix
    ]

    variants = {
        "baseline_v8": [
            baseline_v8_records_0912[j]
            for j in idx
        ],

        "conflict_free": [
            two_edge_records_0912[j]
            for j in idx
        ],

        "conflict_free_unique": [
            strict_records_0913[j]
            for j in idx
        ],
    }

    for policy, recs in (
        variants.items()
    ):

        s = summarise(
            recs
        )

        prefix_rows.append({
            "prefix":
                prefix,

            "policy":
                policy,

            "edge_jaccard":
                s["edge_jaccard"],

            "adj_edge_jaccard":
                s["adj_edge_jaccard"],

            "node_recall":
                s["node_recall"],

            "score":
                s["score"],
        })


prefix_0913 = pd.DataFrame(
    prefix_rows
)


# ============================================================
# Paired robustness
# ============================================================

paired = pd.DataFrame(
    rows_0913
)

paired_rows = []

for prefix in [
    "ALL",
    "44b6",
    "6bba",
]:

    sub = (
        paired
        if prefix == "ALL"
        else paired[
            paired["prefix"]
            == prefix
        ]
    )

    paired_rows.append({
        "prefix":
            prefix,

        "datasets":
            len(sub),

        "bridges_requested":
            int(
                sub[
                    "bridges_requested"
                ].sum()
            ),

        "bridges_added":
            int(
                sub[
                    "bridges_added"
                ].sum()
            ),

        "edgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "adjEdgeJ_improved_rate":
            float(
                (
                    sub[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "mean_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                sub[
                    "delta_recall"
                ].mean()
            ),

        "mean_delta_node_ratio":
            float(
                sub[
                    "delta_node_ratio"
                ].mean()
            ),
    })


paired_summary_0913 = (
    pd.DataFrame(
        paired_rows
    )
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.13 RESULT")
print("=" * 72)

print()
print("OVERALL")

display(
    overall_0913
)

print()
print("BY PREFIX")

display(
    prefix_0913
)

print()
print(
    "STRICT POLICY PAIRED ROBUSTNESS"
)

display(
    paired_summary_0913
)

print()
print("COUNTS")

print(
    "Strict bridges requested:",
    requested_total,
)

print(
    "Strict bridges added:",
    added_total,
)

print(
    "Associations added:",
    edges_added_total,
)

print(
    "Degree-conflict skips:",
    degree_conflict_total,
)

print(
    "Existing-edge conflict skips:",
    existing_conflict_total,
)

print(
    "Nodes rescued vs baseline-v8:",
    84803
    - cleanup_removed_total,
)

print()
print(
    "Strict score - baseline-v8:",
    f"{float(strict_s['score']) - 0.769647:+.6f}",
)

print(
    "Strict score - current two-edge:",
    f"{float(strict_s['score']) - 0.770799:+.6f}",
)

print()
print(
    "09.13 strict two-edge robustness: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.14 Strict Two-Edge Residual Robustness
# Purpose:
#   Calibrate ONLY the CV residual threshold for the successful
#   strict two-edge bridge policy.
#
#   Compare:
#       residual <= 2.0 um
#       residual <= 3.0 um
#       residual <= 4.0 um  [current 09.13 policy]
#
# Strict bridge policy:
#   - existing isolated B only
#   - conservative geometry
#   - pair_count == 1
#   - endpoint conflict-free
#
# Important:
#   conflict-free is recomputed separately INSIDE each residual
#   threshold before applying pair_count == 1.
#
# Frozen pipeline:
#   Warm2
#   -> G2 p2=.80
#   -> K9
#   -> frozen single-edge repair r<=4 um
#   -> strict two-edge repair
#   -> frozen cleanup
#   -> locked metric
#
# Changes files: NO — in-memory graphs only
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only; repair policy uses NO GT
# Keep after running: YES — formal Stage-09 robustness evidence
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "isolated_bridge_candidates_0910" in globals()
assert "single_policy_0912" in globals()

assert "prepare_precleanup_graph" in globals()
assert "add_available_repairs_0907" in globals()
assert "add_two_edge_bridges_0912" in globals()
assert "apply_frozen_component_cleanup" in globals()
assert "evaluate_graph_0906" in globals()

assert "baseline_v8_records_0912" in globals()

RADIUS_VALUES_0914 = [
    2.0,
    3.0,
    4.0,
]


# ============================================================
# Prediction-only strict-policy builder
# ============================================================

def build_strict_two_edge_policy_0914(
    residual_um,
):

    p = (
        isolated_bridge_candidates_0910[
            [
                "dataset",
                "prefix",
                "a_id",
                "b_id",
                "c_id",
                "cv_residual_um",
                "max_step_um",
                "step_balance",
                "turn_angle_deg",
                "ambiguity_margin_um",
                "pair_count",
            ]
        ]
        .copy()
    )

    for col in [
        "a_id",
        "b_id",
        "c_id",
    ]:
        p[col] = p[col].astype(int)


    # --------------------------------------------------------
    # Geometry first.
    #
    # Only residual_um changes across 09.14.
    # --------------------------------------------------------

    p = p[
        (
            p["cv_residual_um"]
            <= residual_um
        )
        &
        (
            p["max_step_um"]
            <= 8.0
        )
        &
        (
            p["step_balance"]
            <= 0.50
        )
        &
        (
            p["turn_angle_deg"]
            <= 60.0
        )
        &
        (
            p["ambiguity_margin_um"]
            >= 1.0
        )
    ].copy()


    # --------------------------------------------------------
    # Recompute endpoint conflicts INSIDE this radius.
    #
    # This preserves the exact 09.13 rule at r=4 while allowing
    # conflicts to disappear naturally for stricter radii.
    # --------------------------------------------------------

    a_usage = (
        p
        .groupby(
            [
                "dataset",
                "a_id",
            ]
        )["b_id"]
        .transform("nunique")
    )

    c_usage = (
        p
        .groupby(
            [
                "dataset",
                "c_id",
            ]
        )["b_id"]
        .transform("nunique")
    )

    p[
        "conflict_free"
    ] = (
        (a_usage == 1)
        &
        (c_usage == 1)
    )


    # --------------------------------------------------------
    # Strict unique-pair policy
    # --------------------------------------------------------

    p = p[
        p["conflict_free"]
        &
        (
            p["pair_count"]
            == 1
        )
    ].copy()

    return p


# ============================================================
# Reproduction gate — current r=4 policy
# ============================================================

policy_r4_0914 = (
    build_strict_two_edge_policy_0914(
        4.0
    )
)

assert len(policy_r4_0914) == 2289, (
    "09.13 policy reproduction failed: "
    f"{len(policy_r4_0914)} != 2289"
)


# ============================================================
# Evaluate each radius
# ============================================================

result_rows_0914 = []
prefix_rows_0914 = []
paired_rows_0914 = []

records_by_radius_0914 = {}


print(
    "=== 09.14 STRICT TWO-EDGE RESIDUAL ROBUSTNESS ==="
)


for radius in RADIUS_VALUES_0914:

    policy = (
        build_strict_two_edge_policy_0914(
            radius
        )
    )

    records = []
    dataset_rows = []

    requested_total = 0
    added_total = 0
    edge_total = 0

    degree_conflict_total = 0
    existing_conflict_total = 0

    cleanup_removed_total = 0


    for i, name in enumerate(
        dataset_names
    ):

        # ====================================================
        # Fresh frozen pre-cleanup graph
        # ====================================================

        g, scale = (
            prepare_precleanup_graph(
                name
            )
        )


        # ====================================================
        # Frozen baseline-v8 single-edge repair
        # ====================================================

        single_candidates = (
            single_policy_0912[
                single_policy_0912[
                    "dataset"
                ] == name
            ]
        )

        single_stats = (
            add_available_repairs_0907(
                g,
                single_candidates,
            )
        )

        assert (
            single_stats["added"]
            == len(single_candidates)
        )


        # ====================================================
        # Strict two-edge repair for current radius
        # ====================================================

        bridge_candidates = (
            policy[
                policy[
                    "dataset"
                ] == name
            ]
        )

        stats = (
            add_two_edge_bridges_0912(
                g,
                bridge_candidates,
            )
        )

        requested_total += (
            stats[
                "requested_bridges"
            ]
        )

        added_total += (
            stats[
                "added_bridges"
            ]
        )

        edge_total += (
            stats[
                "edges_added"
            ]
        )

        degree_conflict_total += (
            stats[
                "degree_conflict"
            ]
        )

        existing_conflict_total += (
            stats[
                "existing_edge_conflict"
            ]
        )


        # ====================================================
        # Frozen cleanup
        # ====================================================

        cleanup = (
            apply_frozen_component_cleanup(
                g,
                scale,
            )
        )

        cleanup_removed_total += int(
            cleanup[
                "nodes_removed"
            ]
        )


        # ====================================================
        # Locked metric
        # ====================================================

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )

        m = evaluate_graph_0906(
            g,
            gt_g,
            scale,
            name,
        )

        records.append(
            m
        )


        # ====================================================
        # Paired evidence vs baseline-v8
        # ====================================================

        base_m = (
            baseline_v8_records_0912[
                i
            ]
        )

        dataset_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "radius_um":
                radius,

            "bridges_requested":
                stats[
                    "requested_bridges"
                ],

            "bridges_added":
                stats[
                    "added_bridges"
                ],

            "delta_edge_jaccard":
                (
                    m["edge_jaccard"]
                    - base_m[
                        "edge_jaccard"
                    ]
                ),

            "delta_adj_edge_jaccard":
                (
                    m[
                        "adj_edge_jaccard"
                    ]
                    - base_m[
                        "adj_edge_jaccard"
                    ]
                ),

            "delta_recall":
                (
                    m["node_recall"]
                    - base_m[
                        "node_recall"
                    ]
                ),

            "delta_node_ratio":
                (
                    m[
                        "total_node_ratio"
                    ]
                    - base_m[
                        "total_node_ratio"
                    ]
                ),
        })


    # ========================================================
    # Overall
    # ========================================================

    records_by_radius_0914[
        radius
    ] = records

    s = summarise(
        records
    )

    result_rows_0914.append({
        "radius_um":
            radius,

        "policy_candidates":
            len(policy),

        "bridges_added":
            added_total,

        "associations_added":
            edge_total,

        "edge_jaccard":
            s[
                "edge_jaccard"
            ],

        "adj_edge_jaccard":
            s[
                "adj_edge_jaccard"
            ],

        "division_jaccard":
            s[
                "division_jaccard"
            ],

        "node_recall":
            s[
                "node_recall"
            ],

        "median_node_ratio":
            float(
                np.median([
                    m[
                        "total_node_ratio"
                    ]
                    for m in records
                ])
            ),

        "score":
            s["score"],

        "nodes_rescued":
            (
                84803
                - cleanup_removed_total
            ),

        "degree_conflict_skips":
            degree_conflict_total,

        "existing_edge_conflict_skips":
            existing_conflict_total,
    })


    # ========================================================
    # Prefix + paired robustness
    # ========================================================

    df = pd.DataFrame(
        dataset_rows
    )

    for prefix in [
        "44b6",
        "6bba",
    ]:

        idx = [
            j
            for j, name in enumerate(
                dataset_names
            )
            if prefix_map[name]
            == prefix
        ]

        recs = [
            records[j]
            for j in idx
        ]

        ps = summarise(
            recs
        )

        sub = df[
            df["prefix"]
            == prefix
        ]

        prefix_rows_0914.append({
            "radius_um":
                radius,

            "prefix":
                prefix,

            "policy_candidates":
                int(
                    (
                        policy[
                            "prefix"
                        ] == prefix
                    ).sum()
                ),

            "bridges_added":
                int(
                    sub[
                        "bridges_added"
                    ].sum()
                ),

            "edge_jaccard":
                ps[
                    "edge_jaccard"
                ],

            "adj_edge_jaccard":
                ps[
                    "adj_edge_jaccard"
                ],

            "node_recall":
                ps[
                    "node_recall"
                ],

            "score":
                ps[
                    "score"
                ],

            "edgeJ_improved_rate":
                float(
                    (
                        sub[
                            "delta_edge_jaccard"
                        ] > 0
                    ).mean()
                ),

            "adjEdgeJ_improved_rate":
                float(
                    (
                        sub[
                            "delta_adj_edge_jaccard"
                        ] > 0
                    ).mean()
                ),

            "median_delta_edgeJ":
                float(
                    sub[
                        "delta_edge_jaccard"
                    ].median()
                ),

            "median_delta_adjEdgeJ":
                float(
                    sub[
                        "delta_adj_edge_jaccard"
                    ].median()
                ),

            "mean_delta_recall":
                float(
                    sub[
                        "delta_recall"
                    ].mean()
                ),
        })


    # ALL paired row
    paired_rows_0914.append({
        "radius_um":
            radius,

        "datasets":
            len(df),

        "edgeJ_improved_rate":
            float(
                (
                    df[
                        "delta_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "adjEdgeJ_improved_rate":
            float(
                (
                    df[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "mean_delta_edgeJ":
            float(
                df[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                df[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_adjEdgeJ":
            float(
                df[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                df[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                df[
                    "delta_recall"
                ].mean()
            ),

        "mean_delta_node_ratio":
            float(
                df[
                    "delta_node_ratio"
                ].mean()
            ),
    })


    print(
        f"radius={radius:.1f} um"
        f" | candidates={len(policy)}"
        f" | added={added_total}"
        f" | score={s['score']:.6f}"
    )


# ============================================================
# Final tables
# ============================================================

overall_0914 = (
    pd.DataFrame(
        result_rows_0914
    )
    .sort_values(
        "radius_um"
    )
    .reset_index(
        drop=True
    )
)

prefix_0914 = (
    pd.DataFrame(
        prefix_rows_0914
    )
    .sort_values(
        [
            "prefix",
            "radius_um",
        ]
    )
    .reset_index(
        drop=True
    )
)

paired_0914 = (
    pd.DataFrame(
        paired_rows_0914
    )
    .sort_values(
        "radius_um"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# Reproduce 09.13 r=4 result
# ============================================================

r4 = (
    overall_0914[
        overall_0914[
            "radius_um"
        ] == 4.0
    ]
    .iloc[0]
)

assert abs(
    float(r4["score"])
    - 0.770921
) < 5e-6, (
    "09.13 r=4 score reproduction failed."
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.14 RESULT")
print("=" * 72)

print()
print("OVERALL")
display(
    overall_0914
)

print()
print("BY PREFIX / ROBUSTNESS")
display(
    prefix_0914
)

print()
print("ALL-DATASET PAIRED ROBUSTNESS")
display(
    paired_0914
)

best = (
    overall_0914
    .sort_values(
        "score",
        ascending=False,
    )
    .iloc[0]
)

print()
print(
    "Best radius:",
    f"{best['radius_um']:.1f} um",
)

print(
    "Best score:",
    f"{best['score']:.6f}",
)

print(
    "Delta vs baseline-v8:",
    f"{best['score'] - 0.769647:+.6f}",
)

print()
print(
    "No GEFF file was modified."
)

print(
    "Repair policies used NO GT information."
)

print()
print(
    "09.14 strict two-edge residual robustness: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.15 Repair-Stage Priority Ablation
# Purpose:
#   Determine priority when the two validated Stage-09 repair
#   mechanisms compete for the same endpoints.
#
# CURRENT:
#   G2 -> K9
#   -> frozen single-edge repair
#   -> strict two-edge repair
#   -> cleanup
#
# REVERSE:
#   G2 -> K9
#   -> strict two-edge repair
#   -> remaining structurally-valid single-edge repair
#   -> cleanup
#
# Frozen geometry:
#   single-edge residual <= 4 um
#
#   strict two-edge:
#       residual <= 4 um
#       max step <= 8 um
#       balance <= .50
#       turn <= 60 deg
#       ambiguity >= 1 um
#       pair_count == 1
#       endpoint conflict-free
#
# Changes files: NO — in-memory graphs only
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only; repair policies use NO GT
# Keep after running: YES — formal Stage-09 interaction evidence
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "single_policy_0912" in globals()
assert "build_strict_two_edge_policy_0914" in globals()

assert "prepare_precleanup_graph" in globals()
assert "add_available_repairs_0907" in globals()
assert "add_two_edge_bridges_0912" in globals()
assert "apply_frozen_component_cleanup" in globals()
assert "evaluate_graph_0906" in globals()

assert "records_by_radius_0914" in globals()
assert "baseline_v8_records_0912" in globals()


# ============================================================
# Frozen policies
# ============================================================

single_policy_0915 = (
    single_policy_0912
    .copy()
)

strict_two_policy_0915 = (
    build_strict_two_edge_policy_0914(
        4.0
    )
)

assert len(single_policy_0915) == 3945
assert len(strict_two_policy_0915) == 2289


# ============================================================
# Current single -> two result from 09.14
# ============================================================

current_records_0915 = (
    records_by_radius_0914[
        4.0
    ]
)

current_s_0915 = summarise(
    current_records_0915
)

assert abs(
    float(
        current_s_0915["score"]
    )
    - 0.770921
) < 5e-6


# ============================================================
# Reverse order:
# two-edge -> surviving single-edge
# ============================================================

reverse_records_0915 = []
dataset_rows_0915 = []

two_requested_total = 0
two_added_total = 0

single_requested_total = 0
single_added_total = 0
single_degree_conflict_total = 0
single_existing_total = 0
single_missing_total = 0

cleanup_removed_total = 0


print(
    "=== 09.15 REPAIR-STAGE PRIORITY ABLATION ==="
)


for i, name in enumerate(
    dataset_names
):

    # --------------------------------------------------------
    # Fresh G2 + K9 graph
    # --------------------------------------------------------

    g, scale = (
        prepare_precleanup_graph(
            name
        )
    )


    # ========================================================
    # 1. STRICT TWO-EDGE FIRST
    # ========================================================

    two_candidates = (
        strict_two_policy_0915[
            strict_two_policy_0915[
                "dataset"
            ] == name
        ]
    )

    two_stats = (
        add_two_edge_bridges_0912(
            g,
            two_candidates,
        )
    )

    two_requested_total += (
        two_stats[
            "requested_bridges"
        ]
    )

    two_added_total += (
        two_stats[
            "added_bridges"
        ]
    )


    # ========================================================
    # 2. THEN REMAINING VALID SINGLE-EDGE REPAIRS
    # ========================================================

    single_candidates = (
        single_policy_0915[
            single_policy_0915[
                "dataset"
            ] == name
        ]
    )

    single_stats = (
        add_available_repairs_0907(
            g,
            single_candidates,
        )
    )

    single_requested_total += (
        single_stats[
            "candidate_edges"
        ]
    )

    single_added_total += (
        single_stats[
            "added"
        ]
    )

    single_degree_conflict_total += (
        single_stats[
            "degree_conflict"
        ]
    )

    single_existing_total += (
        single_stats[
            "already_present"
        ]
    )

    single_missing_total += (
        single_stats[
            "missing_endpoint"
        ]
    )


    # ========================================================
    # 3. Frozen cleanup
    # ========================================================

    cleanup = (
        apply_frozen_component_cleanup(
            g,
            scale,
        )
    )

    cleanup_removed_total += int(
        cleanup[
            "nodes_removed"
        ]
    )


    # ========================================================
    # 4. Locked metric
    # ========================================================

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    m = evaluate_graph_0906(
        g,
        gt_g,
        scale,
        name,
    )

    reverse_records_0915.append(
        m
    )


    # ========================================================
    # Paired comparisons
    # ========================================================

    current_m = (
        current_records_0915[i]
    )

    base_m = (
        baseline_v8_records_0912[i]
    )

    dataset_rows_0915.append({
        "dataset":
            name,

        "prefix":
            prefix_map[name],

        "two_requested":
            two_stats[
                "requested_bridges"
            ],

        "two_added":
            two_stats[
                "added_bridges"
            ],

        "single_requested":
            single_stats[
                "candidate_edges"
            ],

        "single_added":
            single_stats[
                "added"
            ],

        # Reverse vs current ordering
        "delta_reverse_vs_current_edgeJ":
            (
                m["edge_jaccard"]
                - current_m[
                    "edge_jaccard"
                ]
            ),

        "delta_reverse_vs_current_adjEdgeJ":
            (
                m[
                    "adj_edge_jaccard"
                ]
                - current_m[
                    "adj_edge_jaccard"
                ]
            ),

        "delta_reverse_vs_current_recall":
            (
                m["node_recall"]
                - current_m[
                    "node_recall"
                ]
            ),

        # Reverse vs baseline-v8
        "delta_reverse_vs_base_edgeJ":
            (
                m["edge_jaccard"]
                - base_m[
                    "edge_jaccard"
                ]
            ),

        "delta_reverse_vs_base_adjEdgeJ":
            (
                m[
                    "adj_edge_jaccard"
                ]
                - base_m[
                    "adj_edge_jaccard"
                ]
            ),

        "delta_reverse_vs_base_recall":
            (
                m["node_recall"]
                - base_m[
                    "node_recall"
                ]
            ),
    })


    if (
        i == 0
        or (i + 1) % 10 == 0
        or i + 1 == len(dataset_names)
    ):
        print(
            f"Processed {i + 1:02d}/"
            f"{len(dataset_names)}"
        )


# ============================================================
# Overall
# ============================================================

reverse_s_0915 = summarise(
    reverse_records_0915
)

base_s_0915 = summarise(
    baseline_v8_records_0912
)


def make_summary_row_0915(
    policy,
    s,
    records,
):

    return {
        "policy":
            policy,

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    }


overall_0915 = pd.DataFrame([
    make_summary_row_0915(
        "baseline_v8",
        base_s_0915,
        baseline_v8_records_0912,
    ),

    make_summary_row_0915(
        "single_then_two",
        current_s_0915,
        current_records_0915,
    ),

    make_summary_row_0915(
        "two_then_single",
        reverse_s_0915,
        reverse_records_0915,
    ),
])


# ============================================================
# Prefix aggregates
# ============================================================

prefix_rows_0915 = []

for prefix in [
    "44b6",
    "6bba",
]:

    idx = [
        j
        for j, name in enumerate(
            dataset_names
        )
        if prefix_map[name]
        == prefix
    ]

    variants = {
        "baseline_v8": [
            baseline_v8_records_0912[j]
            for j in idx
        ],

        "single_then_two": [
            current_records_0915[j]
            for j in idx
        ],

        "two_then_single": [
            reverse_records_0915[j]
            for j in idx
        ],
    }

    for policy, records in variants.items():

        s = summarise(
            records
        )

        prefix_rows_0915.append({
            "prefix":
                prefix,

            "policy":
                policy,

            "edge_jaccard":
                s[
                    "edge_jaccard"
                ],

            "adj_edge_jaccard":
                s[
                    "adj_edge_jaccard"
                ],

            "node_recall":
                s[
                    "node_recall"
                ],

            "score":
                s[
                    "score"
                ],
        })


prefix_0915 = pd.DataFrame(
    prefix_rows_0915
)


# ============================================================
# Paired ordering robustness
# ============================================================

paired_df_0915 = pd.DataFrame(
    dataset_rows_0915
)

paired_rows = []

for prefix in [
    "ALL",
    "44b6",
    "6bba",
]:

    sub = (
        paired_df_0915
        if prefix == "ALL"
        else paired_df_0915[
            paired_df_0915[
                "prefix"
            ] == prefix
        ]
    )

    paired_rows.append({
        "prefix":
            prefix,

        "datasets":
            len(sub),

        "reverse_better_edgeJ_rate":
            float(
                (
                    sub[
                        "delta_reverse_vs_current_edgeJ"
                    ] > 0
                ).mean()
            ),

        "reverse_better_adjEdgeJ_rate":
            float(
                (
                    sub[
                        "delta_reverse_vs_current_adjEdgeJ"
                    ] > 0
                ).mean()
            ),

        "mean_reverse_vs_current_edgeJ":
            float(
                sub[
                    "delta_reverse_vs_current_edgeJ"
                ].mean()
            ),

        "median_reverse_vs_current_edgeJ":
            float(
                sub[
                    "delta_reverse_vs_current_edgeJ"
                ].median()
            ),

        "mean_reverse_vs_current_adjEdgeJ":
            float(
                sub[
                    "delta_reverse_vs_current_adjEdgeJ"
                ].mean()
            ),

        "median_reverse_vs_current_adjEdgeJ":
            float(
                sub[
                    "delta_reverse_vs_current_adjEdgeJ"
                ].median()
            ),

        "mean_reverse_vs_current_recall":
            float(
                sub[
                    "delta_reverse_vs_current_recall"
                ].mean()
            ),
    })


paired_summary_0915 = (
    pd.DataFrame(
        paired_rows
    )
)


# ============================================================
# Result
# ============================================================

current_score = float(
    current_s_0915[
        "score"
    ]
)

reverse_score = float(
    reverse_s_0915[
        "score"
    ]
)


print()
print("=" * 72)
print("09.15 RESULT")
print("=" * 72)

print()
print("OVERALL")
display(
    overall_0915
)

print()
print("BY PREFIX")
display(
    prefix_0915
)

print()
print("ORDERING PAIRED ROBUSTNESS")
display(
    paired_summary_0915
)

print()
print("REVERSE-ORDER INTERACTION COUNTS")

print(
    "Two-edge requested:",
    two_requested_total,
)

print(
    "Two-edge added:",
    two_added_total,
)

print(
    "Single-edge requested:",
    single_requested_total,
)

print(
    "Single-edge added:",
    single_added_total,
)

print(
    "Single degree-conflict skips:",
    single_degree_conflict_total,
)

print(
    "Single existing-edge skips:",
    single_existing_total,
)

print(
    "Single missing-endpoint skips:",
    single_missing_total,
)

print(
    "Cleanup nodes removed:",
    cleanup_removed_total,
)

print()
print(
    "Reverse - current score:",
    f"{reverse_score - current_score:+.6f}",
)

print(
    "Reverse - baseline-v8 score:",
    f"{reverse_score - 0.769647:+.6f}",
)

print()
print(
    "No GEFF file was modified."
)

print(
    "Repair policies used NO GT information."
)

print()
print(
    "09.15 repair priority ablation: COMPLETE"
)

baseline-v9

Checkpoint:
Warm2
31d209bfde1fd5bbf3b2a661cdb4a25bb72922b832d315ee5a75cc32f44656c1

Detection:
det = .995
high = .9975

Confidence pruning:
K = 9

Division:
p2 = .80
daughter separation >= 6 µm
angle >= 120°
max parent-daughter <= 8 µm
balance <= .50

Single-edge recovery:
mutual support
residual <= 4 µm
max step <= 8 µm
balance <= .50
turn <= 60°
ambiguity >= 1 µm

Strict two-edge recovery:
existing isolated B
residual <= 4 µm
max step <= 8 µm
balance <= .50
turn <= 60°
ambiguity >= 1 µm
pair_count == 1
endpoint conflict-free

Order:
G2
→ K9
→ single-edge
→ strict two-edge
→ cleanup

Cleanup:
remove singleton
remove 2-node component if displacement > 3.5 µm

Fold0:
Edge Jaccard       0.774393
Adjusted Edge J    0.767075
Division Jaccard   0.038462
Recall             0.956612
Score              0.770921

In [ ]:
# ============================================================
# RUN — 09.16 Synthetic Missing-Node Gap Census
# Purpose:
#   Audit one-frame gaps where the intermediate cell detection
#   B(t+1) may be completely missing:
#
#       P(t-1) -> A(t)    [missing B(t+1)]    C(t+2) -> D(t+3)
#
#   Require motion context on BOTH sides.
#
#   Left prediction:
#       B_left = 2A - P
#       C_pred = 3A - 2P
#
#   Right prediction:
#       B_right = 2C - D
#       A_pred = 3C - 2D
#
#   This Cell ONLY performs prediction-only census.
#   It does NOT create synthetic nodes or edges.
#
# Pipeline state:
#   Warm2 -> G2 -> K9
#   -> frozen single-edge recovery
#   -> frozen strict two-edge recovery
#   -> [THIS AUDIT]
#   -> cleanup later
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: YES — Stage-09 synthetic-gap census evidence
# ============================================================

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from tracking_cellmot.io import open_dataset


# ============================================================
# Preconditions
# ============================================================

assert "prepare_precleanup_graph" in globals()

assert "single_policy_0915" in globals()
assert "strict_two_policy_0915" in globals()

assert "add_available_repairs_0907" in globals()
assert "add_two_edge_bridges_0912" in globals()

assert "dataset_names" in globals()
assert "prefix_map" in globals()
assert "TRAIN_DIR" in globals()


# ============================================================
# Diagnostic search parameters
#
# SEARCH radius is intentionally broad.
# It is NOT a frozen synthetic-node policy.
# ============================================================

SEARCH_RESIDUAL_UM_0916 = 8.0


# ============================================================
# Geometry helper
# ============================================================

def angle_between_deg_0916(
    v1,
    v2,
):

    n1 = float(
        np.linalg.norm(v1)
    )

    n2 = float(
        np.linalg.norm(v2)
    )

    if (
        n1 <= 1e-12
        or n2 <= 1e-12
    ):
        return np.nan

    cosine = float(
        np.dot(v1, v2)
        / (n1 * n2)
    )

    cosine = np.clip(
        cosine,
        -1.0,
        1.0,
    )

    return float(
        np.degrees(
            np.arccos(cosine)
        )
    )


# ============================================================
# Build frozen baseline-v9 PRE-CLEANUP graph
# ============================================================

def prepare_v9_precleanup_0916(
    name,
):

    g, scale = (
        prepare_precleanup_graph(
            name
        )
    )


    # --------------------------------------------------------
    # Frozen single-edge repair
    # --------------------------------------------------------

    single_candidates = (
        single_policy_0915[
            single_policy_0915[
                "dataset"
            ] == name
        ]
    )

    single_stats = (
        add_available_repairs_0907(
            g,
            single_candidates,
        )
    )

    assert (
        single_stats["added"]
        == len(single_candidates)
    )


    # --------------------------------------------------------
    # Frozen strict two-edge repair
    # --------------------------------------------------------

    two_candidates = (
        strict_two_policy_0915[
            strict_two_policy_0915[
                "dataset"
            ] == name
        ]
    )

    add_two_edge_bridges_0912(
        g,
        two_candidates,
    )

    return g, scale


# ============================================================
# One-dataset census
# ============================================================

def synthetic_gap_census_one_0916(
    name,
):

    g, scale = (
        prepare_v9_precleanup_0916(
            name
        )
    )

    scale = np.asarray(
        scale,
        dtype=float,
    )


    # --------------------------------------------------------
    # Node table
    # --------------------------------------------------------

    nodes = (
        g.node_attrs(
            unpack=True
        )
        .to_pandas()
    )

    node_ids = (
        nodes["node_id"]
        .to_numpy(
            dtype=int
        )
    )

    t_arr = (
        nodes["t"]
        .to_numpy(
            dtype=int
        )
    )

    pos = np.column_stack([
        nodes["z"].to_numpy(
            dtype=float
        ) * scale[0],

        nodes["y"].to_numpy(
            dtype=float
        ) * scale[1],

        nodes["x"].to_numpy(
            dtype=float
        ) * scale[2],
    ])


    id_to_idx = {
        int(node_id): i
        for i, node_id in enumerate(
            node_ids
        )
    }


    in_deg = np.asarray(
        g.in_degree(
            node_ids.tolist()
        ),
        dtype=int,
    )

    out_deg = np.asarray(
        g.out_degree(
            node_ids.tolist()
        ),
        dtype=int,
    )


    t_min = int(
        t_arr.min()
    )

    t_max = int(
        t_arr.max()
    )


    # ========================================================
    # Eligible left anchors A:
    #
    # P -> A
    # A has exactly one predecessor
    # A has no successor
    # ========================================================

    left_mask = (
        (in_deg == 1)
        &
        (out_deg == 0)
        &
        (t_arr >= t_min + 1)
        &
        (t_arr <= t_max - 3)
    )


    # ========================================================
    # Eligible right anchors C:
    #
    # C -> D
    # C has no predecessor
    # C has exactly one successor
    # ========================================================

    right_mask = (
        (in_deg == 0)
        &
        (out_deg == 1)
        &
        (t_arr >= t_min + 2)
        &
        (t_arr <= t_max - 1)
    )


    left_idx = np.flatnonzero(
        left_mask
    )

    right_idx = np.flatnonzero(
        right_mask
    )


    # ========================================================
    # Right anchors grouped by frame
    # ========================================================

    right_by_t = {}

    for t in np.unique(
        t_arr[right_idx]
    ):

        idx = right_idx[
            t_arr[right_idx]
            == t
        ]

        if len(idx):

            right_by_t[int(t)] = (
                cKDTree(
                    pos[idx]
                ),
                idx,
            )


    # ========================================================
    # Existing-node occupancy trees at missing frame t+1
    # ========================================================

    frame_trees = {}

    for t in np.unique(
        t_arr
    ):

        idx = np.flatnonzero(
            t_arr == t
        )

        if len(idx):

            frame_trees[int(t)] = (
                cKDTree(
                    pos[idx]
                ),
                idx,
            )


    # ========================================================
    # Generate broad A...C candidates
    # ========================================================

    rows = []


    for ai in left_idx:

        a_id = int(
            node_ids[ai]
        )

        ta = int(
            t_arr[ai]
        )

        tc = ta + 2

        if tc not in right_by_t:
            continue


        # ----------------------------------------------------
        # Unique predecessor P
        # ----------------------------------------------------

        preds = g.predecessors(
            a_id
        )

        if len(preds) != 1:
            continue

        p_id = int(
            preds[0]
        )

        if p_id not in id_to_idx:
            continue

        pi = id_to_idx[
            p_id
        ]

        if int(
            t_arr[pi]
        ) != ta - 1:
            continue


        # ----------------------------------------------------
        # Left constant-velocity predictions
        # ----------------------------------------------------

        v_left = (
            pos[ai]
            - pos[pi]
        )

        b_left = (
            pos[ai]
            + v_left
        )

        c_pred = (
            pos[ai]
            + 2.0 * v_left
        )


        # ----------------------------------------------------
        # Search C(t+2) near left prediction
        # ----------------------------------------------------

        c_tree, c_indices = (
            right_by_t[tc]
        )

        local_c = (
            c_tree.query_ball_point(
                c_pred,
                r=SEARCH_RESIDUAL_UM_0916,
            )
        )

        if not local_c:
            continue


        for local_index in local_c:

            ci = c_indices[
                int(local_index)
            ]

            c_id = int(
                node_ids[ci]
            )


            # ------------------------------------------------
            # Unique successor D
            # ------------------------------------------------

            succs = g.successors(
                c_id
            )

            if len(succs) != 1:
                continue

            d_id = int(
                succs[0]
            )

            if d_id not in id_to_idx:
                continue

            di = id_to_idx[
                d_id
            ]

            if int(
                t_arr[di]
            ) != tc + 1:
                continue


            # ------------------------------------------------
            # Right motion prediction
            # ------------------------------------------------

            v_right = (
                pos[di]
                - pos[ci]
            )

            b_right = (
                pos[ci]
                - v_right
            )

            a_pred = (
                pos[ci]
                - 2.0 * v_right
            )


            # ------------------------------------------------
            # Residuals
            # ------------------------------------------------

            left_residual = float(
                np.linalg.norm(
                    pos[ci]
                    - c_pred
                )
            )

            right_residual = float(
                np.linalg.norm(
                    pos[ai]
                    - a_pred
                )
            )

            b_agreement = float(
                np.linalg.norm(
                    b_left
                    - b_right
                )
            )


            # ------------------------------------------------
            # Three estimates of missing B
            # ------------------------------------------------

            b_midpoint = (
                0.5
                * (
                    pos[ai]
                    + pos[ci]
                )
            )

            b_consensus = (
                b_left
                + b_right
                + b_midpoint
            ) / 3.0


            midpoint_left_error = float(
                np.linalg.norm(
                    b_left
                    - b_midpoint
                )
            )

            midpoint_right_error = float(
                np.linalg.norm(
                    b_right
                    - b_midpoint
                )
            )

            midpoint_consistency = max(
                midpoint_left_error,
                midpoint_right_error,
            )


            # ------------------------------------------------
            # Motion geometry
            # ------------------------------------------------

            left_step = float(
                np.linalg.norm(
                    v_left
                )
            )

            right_step = float(
                np.linalg.norm(
                    v_right
                )
            )

            synthetic_step_ab = float(
                np.linalg.norm(
                    b_consensus
                    - pos[ai]
                )
            )

            synthetic_step_bc = float(
                np.linalg.norm(
                    pos[ci]
                    - b_consensus
                )
            )

            max_step = max(
                left_step,
                right_step,
                synthetic_step_ab,
                synthetic_step_bc,
            )

            min_step = min(
                left_step,
                right_step,
                synthetic_step_ab,
                synthetic_step_bc,
            )

            step_balance = float(
                (
                    max_step
                    - min_step
                )
                / max(
                    max_step,
                    1e-12,
                )
            )

            velocity_angle = (
                angle_between_deg_0916(
                    v_left,
                    v_right,
                )
            )


            # ------------------------------------------------
            # Is there already a detection near predicted B?
            #
            # This is crucial before ever synthesizing a node.
            # ------------------------------------------------

            missing_t = ta + 1

            nearest_existing_um = np.inf
            nearest_existing_id = -1

            if missing_t in frame_trees:

                b_tree, b_indices = (
                    frame_trees[
                        missing_t
                    ]
                )

                dist, local_b = (
                    b_tree.query(
                        b_consensus,
                        k=1,
                    )
                )

                nearest_existing_um = float(
                    dist
                )

                nearest_existing_id = int(
                    node_ids[
                        b_indices[
                            int(local_b)
                        ]
                    ]
                )


            rows.append({
                "dataset":
                    name,

                "prefix":
                    prefix_map[name],

                "p_id":
                    p_id,

                "a_id":
                    a_id,

                "c_id":
                    c_id,

                "d_id":
                    d_id,

                "t_missing":
                    missing_t,

                "left_residual_um":
                    left_residual,

                "right_residual_um":
                    right_residual,

                "max_directional_residual_um":
                    max(
                        left_residual,
                        right_residual,
                    ),

                "b_agreement_um":
                    b_agreement,

                "midpoint_consistency_um":
                    midpoint_consistency,

                "left_step_um":
                    left_step,

                "right_step_um":
                    right_step,

                "synthetic_ab_um":
                    synthetic_step_ab,

                "synthetic_bc_um":
                    synthetic_step_bc,

                "max_step_um":
                    max_step,

                "step_balance":
                    step_balance,

                "velocity_angle_deg":
                    velocity_angle,

                "nearest_existing_midpoint_um":
                    nearest_existing_um,

                "nearest_existing_midpoint_id":
                    nearest_existing_id,

                "b_z_um":
                    float(
                        b_consensus[0]
                    ),

                "b_y_um":
                    float(
                        b_consensus[1]
                    ),

                "b_x_um":
                    float(
                        b_consensus[2]
                    ),
            })


    return pd.DataFrame(
        rows
    )


# ============================================================
# Full Fold0 census
# ============================================================

parts_0916 = []


print(
    "=== 09.16 SYNTHETIC MISSING-NODE GAP CENSUS ==="
)


for i, name in enumerate(
    dataset_names,
    start=1,
):

    df = (
        synthetic_gap_census_one_0916(
            name
        )
    )

    if len(df):
        parts_0916.append(
            df
        )

    if (
        i == 1
        or i % 10 == 0
        or i == len(dataset_names)
    ):

        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


assert parts_0916

synthetic_gap_candidates_0916 = (
    pd.concat(
        parts_0916,
        ignore_index=True,
    )
)


# ============================================================
# Mutual-best A <-> C support
#
# LEFT:
#   for each A choose minimum left residual C
#
# RIGHT:
#   for each C choose minimum right residual A
#
# Require both choices to agree.
# ============================================================

cand = (
    synthetic_gap_candidates_0916
    .copy()
)


left_best_idx = (
    cand
    .groupby(
        [
            "dataset",
            "a_id",
        ]
    )[
        "left_residual_um"
    ]
    .idxmin()
)


right_best_idx = (
    cand
    .groupby(
        [
            "dataset",
            "c_id",
        ]
    )[
        "right_residual_um"
    ]
    .idxmin()
)


cand[
    "left_best"
] = False

cand.loc[
    left_best_idx,
    "left_best",
] = True


cand[
    "right_best"
] = False

cand.loc[
    right_best_idx,
    "right_best",
] = True


cand[
    "mutual_best"
] = (
    cand["left_best"]
    &
    cand["right_best"]
)


synthetic_gap_candidates_0916 = (
    cand
)


# ============================================================
# Geometry summary
# ============================================================

mutual_0916 = (
    cand[
        cand["mutual_best"]
    ]
    .copy()
)


geometry_quantiles_0916 = (
    mutual_0916
    .groupby(
        "prefix"
    )
    [
        [
            "max_directional_residual_um",
            "b_agreement_um",
            "midpoint_consistency_um",
            "max_step_um",
            "step_balance",
            "velocity_angle_deg",
            "nearest_existing_midpoint_um",
        ]
    ]
    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .round(4)
)


# ============================================================
# Diagnostic geometry gates
#
# Not a frozen policy.
# ============================================================

strong_geometry_0916 = (
    mutual_0916[
        (
            mutual_0916[
                "max_directional_residual_um"
            ] <= 4.0
        )
        &
        (
            mutual_0916[
                "b_agreement_um"
            ] <= 4.0
        )
        &
        (
            mutual_0916[
                "max_step_um"
            ] <= 8.0
        )
        &
        (
            mutual_0916[
                "step_balance"
            ] <= 0.50
        )
        &
        (
            mutual_0916[
                "velocity_angle_deg"
            ] <= 60.0
        )
    ]
    .copy()
)


# ============================================================
# Existing-node exclusion census
#
# Do NOT choose a threshold yet.
# ============================================================

occupancy_rows_0916 = []

for prefix in [
    "44b6",
    "6bba",
]:

    sub = (
        strong_geometry_0916[
            strong_geometry_0916[
                "prefix"
            ] == prefix
        ]
    )

    for empty_radius in [
        0.0,
        2.0,
        3.0,
        4.0,
        5.0,
    ]:

        kept = sub[
            sub[
                "nearest_existing_midpoint_um"
            ] > empty_radius
        ]

        occupancy_rows_0916.append({
            "prefix":
                prefix,

            "empty_radius_um":
                empty_radius,

            "strong_geometry":
                len(sub),

            "remaining":
                len(kept),

            "remaining_fraction":
                (
                    len(kept)
                    / len(sub)
                    if len(sub)
                    else np.nan
                ),
        })


occupancy_census_0916 = (
    pd.DataFrame(
        occupancy_rows_0916
    )
)


# ============================================================
# Counts
# ============================================================

count_rows_0916 = []

for prefix in [
    "44b6",
    "6bba",
]:

    all_sub = cand[
        cand["prefix"]
        == prefix
    ]

    mutual_sub = mutual_0916[
        mutual_0916["prefix"]
        == prefix
    ]

    strong_sub = (
        strong_geometry_0916[
            strong_geometry_0916[
                "prefix"
            ] == prefix
        ]
    )

    count_rows_0916.append({
        "prefix":
            prefix,

        "broad_pairs":
            len(all_sub),

        "mutual_best_pairs":
            len(mutual_sub),

        "strong_geometry_pairs":
            len(strong_sub),

        "strong_empty_gt_2um":
            int(
                (
                    strong_sub[
                        "nearest_existing_midpoint_um"
                    ] > 2.0
                ).sum()
            ),

        "strong_empty_gt_4um":
            int(
                (
                    strong_sub[
                        "nearest_existing_midpoint_um"
                    ] > 4.0
                ).sum()
            ),
    })


counts_0916 = pd.DataFrame(
    count_rows_0916
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.16 RESULT")
print("=" * 72)

print()
print("CANDIDATE COUNTS")

display(
    counts_0916
)

print()
print("MUTUAL-BEST GEOMETRY QUANTILES")

display(
    geometry_quantiles_0916
)

print()
print("MIDPOINT OCCUPANCY CENSUS")

display(
    occupancy_census_0916
)

print()
print(
    "Broad candidate pairs:",
    len(
        synthetic_gap_candidates_0916
    ),
)

print(
    "Mutual-best pairs:",
    len(
        mutual_0916
    ),
)

print(
    "Strong-geometry pairs:",
    len(
        strong_geometry_0916
    ),
)

print()
print(
    "No synthetic node was created."
)

print(
    "No edge was added."
)

print(
    "No graph/file was modified."
)

print(
    "No GT was used."
)

print()
print(
    "09.16 synthetic missing-node census: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.17 Synthetic Missing-Node Oracle Audit
# Purpose:
#   Evaluate whether Stage-09 synthetic-gap candidates have
#   positive GT evidence for a truly missing intermediate cell:
#
#       prediction:
#           P -> A    [missing B]    C -> D
#
#       GT positive evidence:
#           A* -> B* -> C*
#
#   Additional safety checks:
#       - Is GT middle node B* already matched by an existing
#         prediction node?  If YES, synthesizing B would risk
#         duplicating an existing detection.
#
#       - How far is the proposed synthetic B coordinate from B*?
#
#       - Is proposed B within the official 7 um matching radius?
#
# IMPORTANT:
#   GT is sparse. Absence of an A*->B*->C* path is NOT
#   automatically classified as false.
#
#   We classify:
#       POSITIVE:
#           unique GT A*->B*->C* path exists
#
#       EXPLICIT_NEGATIVE:
#           both GT sides contain explicit t+1 adjacency but
#           there is no common intermediate node
#
#       UNRESOLVED:
#           insufficient GT annotation
#
# No synthetic node is created.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 oracle analysis only
# Keep after running: YES — formal Stage-09 synthetic-gap evidence
# ============================================================

import numpy as np
import pandas as pd

from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
)


# ============================================================
# Preconditions
# ============================================================

assert "strong_geometry_0916" in globals()
assert "prepare_v9_precleanup_0916" in globals()

assert "dataset_names" in globals()
assert "prefix_map" in globals()
assert "TRAIN_DIR" in globals()
assert "load_graph" in globals()

assert len(strong_geometry_0916) == 1067


# ============================================================
# Prediction-only candidate table
# ============================================================

candidates_0917 = (
    strong_geometry_0916[
        [
            "dataset",
            "prefix",
            "p_id",
            "a_id",
            "c_id",
            "d_id",
            "t_missing",

            "max_directional_residual_um",
            "b_agreement_um",
            "midpoint_consistency_um",
            "max_step_um",
            "step_balance",
            "velocity_angle_deg",

            "nearest_existing_midpoint_um",

            "b_z_um",
            "b_y_um",
            "b_x_um",
        ]
    ]
    .copy()
)

for col in [
    "p_id",
    "a_id",
    "c_id",
    "d_id",
    "t_missing",
]:
    candidates_0917[col] = (
        candidates_0917[col]
        .astype(int)
    )


# ============================================================
# Oracle audit
# ============================================================

oracle_parts_0917 = []


print(
    "=== 09.17 SYNTHETIC MISSING-NODE ORACLE AUDIT ==="
)


for i, name in enumerate(
    dataset_names,
    start=1,
):

    sub = (
        candidates_0917[
            candidates_0917[
                "dataset"
            ] == name
        ]
        .copy()
    )

    if not len(sub):
        continue


    # --------------------------------------------------------
    # Frozen baseline-v9 PRE-cleanup prediction graph
    # --------------------------------------------------------

    pred_g, scale = (
        prepare_v9_precleanup_0916(
            name
        )
    )

    scale = np.asarray(
        scale,
        dtype=float,
    )


    # --------------------------------------------------------
    # Fresh GT
    # --------------------------------------------------------

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )


    # --------------------------------------------------------
    # Locked official node matching
    #
    # Mutates pred_g by adding match_node_id / match_score.
    # --------------------------------------------------------

    official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )


    # ========================================================
    # Prediction -> GT map
    # ========================================================

    pred_matches = (
        pred_g
        .node_attrs(
            unpack=True
        )
        .to_pandas()
    )

    assert (
        "match_node_id"
        in pred_matches.columns
    )

    match_map = dict(
        zip(
            pred_matches[
                "node_id"
            ].astype(int),
            pred_matches[
                "match_node_id"
            ].astype(int),
        )
    )


    # All GT nodes already claimed by an existing prediction.
    matched_gt_ids = set(
        pred_matches.loc[
            pred_matches[
                "match_node_id"
            ] >= 0,
            "match_node_id",
        ]
        .astype(int)
        .tolist()
    )


    # ========================================================
    # GT node table
    # ========================================================

    gt_nodes = (
        gt_g
        .node_attrs(
            unpack=True
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    # ========================================================
    # Prediction endpoint matching
    # ========================================================

    for role in [
        "p",
        "a",
        "c",
        "d",
    ]:

        sub[
            f"gt_{role}_id"
        ] = [
            int(
                match_map.get(
                    int(node_id),
                    -1,
                )
            )
            for node_id in sub[
                f"{role}_id"
            ]
        ]


    # ========================================================
    # Candidate-level GT audit
    # ========================================================

    result_rows = []


    for row in sub.itertuples(
        index=False
    ):

        gt_p = int(
            row.gt_p_id
        )

        gt_a = int(
            row.gt_a_id
        )

        gt_c = int(
            row.gt_c_id
        )

        gt_d = int(
            row.gt_d_id
        )

        t_missing = int(
            row.t_missing
        )


        # ----------------------------------------------------
        # Endpoint observability
        # ----------------------------------------------------

        ac_observable = (
            gt_a >= 0
            and gt_c >= 0
        )

        full_context_observable = (
            gt_p >= 0
            and gt_a >= 0
            and gt_c >= 0
            and gt_d >= 0
        )


        # Defaults
        gt_middle_count = 0
        gt_middle_id = -1

        lineage_positive = False
        explicit_negative = False

        context_true = False

        middle_already_matched = False

        synthetic_error_um = np.nan
        synthetic_within_7um = False


        # ----------------------------------------------------
        # Check P*->A* and C*->D* when fully observable.
        #
        # This is supplementary motion-context evidence.
        # ----------------------------------------------------

        if full_context_observable:

            context_true = bool(
                gt_g.has_edge(
                    gt_p,
                    gt_a,
                )
                and
                gt_g.has_edge(
                    gt_c,
                    gt_d,
                )
            )


        # ----------------------------------------------------
        # A* ... C* oracle
        # ----------------------------------------------------

        if ac_observable:

            # Explicit t+1 GT successors of A.
            a_successors = [
                int(x)
                for x in gt_g.successors(
                    gt_a
                )
            ]

            a_successors_t1 = [
                x
                for x in a_successors
                if (
                    x in gt_nodes.index
                    and int(
                        gt_nodes.loc[
                            x,
                            "t",
                        ]
                    ) == t_missing
                )
            ]


            # Explicit t+1 GT predecessors of C.
            c_predecessors = [
                int(x)
                for x in gt_g.predecessors(
                    gt_c
                )
            ]

            c_predecessors_t1 = [
                x
                for x in c_predecessors
                if (
                    x in gt_nodes.index
                    and int(
                        gt_nodes.loc[
                            x,
                            "t",
                        ]
                    ) == t_missing
                )
            ]


            # Common intermediate nodes:
            #
            #     A* -> B* -> C*
            #
            middle_ids = sorted(
                set(
                    a_successors_t1
                )
                &
                set(
                    c_predecessors_t1
                )
            )

            gt_middle_count = len(
                middle_ids
            )


            # ------------------------------------------------
            # Positive evidence requires UNIQUE intermediate B.
            # ------------------------------------------------

            if gt_middle_count == 1:

                gt_middle_id = int(
                    middle_ids[0]
                )

                lineage_positive = True


                # --------------------------------------------
                # Is true B* already represented by another
                # prediction node?
                # --------------------------------------------

                middle_already_matched = (
                    gt_middle_id
                    in matched_gt_ids
                )


                # --------------------------------------------
                # Synthetic coordinate error
                # --------------------------------------------

                gt_b = gt_nodes.loc[
                    gt_middle_id
                ]

                gt_b_um = np.array(
                    [
                        float(
                            gt_b["z"]
                        ) * scale[0],

                        float(
                            gt_b["y"]
                        ) * scale[1],

                        float(
                            gt_b["x"]
                        ) * scale[2],
                    ],
                    dtype=float,
                )

                pred_b_um = np.array(
                    [
                        float(
                            row.b_z_um
                        ),

                        float(
                            row.b_y_um
                        ),

                        float(
                            row.b_x_um
                        ),
                    ],
                    dtype=float,
                )

                synthetic_error_um = float(
                    np.linalg.norm(
                        pred_b_um
                        - gt_b_um
                    )
                )

                synthetic_within_7um = (
                    synthetic_error_um
                    <= 7.0
                )


            # ------------------------------------------------
            # Explicit negative:
            #
            # Both A and C have explicit t+1 GT adjacency,
            # but those annotations do NOT share a middle node.
            #
            # This avoids treating missing GT annotation as a
            # negative example.
            # ------------------------------------------------

            elif (
                len(
                    a_successors_t1
                ) > 0
                and
                len(
                    c_predecessors_t1
                ) > 0
            ):

                explicit_negative = True


        # ====================================================
        # Synthetic-node oracle classification
        # ====================================================

        synthetic_positive = bool(
            lineage_positive
            and not middle_already_matched
            and synthetic_within_7um
        )


        # A candidate is a resolved negative for synthesis if:
        #
        # 1. GT explicitly contradicts the A...C lineage, OR
        #
        # 2. correct B* already has an existing matched
        #    prediction (duplicate-node risk), OR
        #
        # 3. B* is genuinely missing but proposed coordinate
        #    falls outside official 7 um matching radius.
        #
        synthetic_negative = bool(
            explicit_negative
            or (
                lineage_positive
                and (
                    middle_already_matched
                    or not synthetic_within_7um
                )
            )
        )


        result_rows.append({
            **row._asdict(),

            "ac_observable":
                ac_observable,

            "full_context_observable":
                full_context_observable,

            "context_true":
                context_true,

            "gt_middle_count":
                gt_middle_count,

            "gt_middle_id":
                gt_middle_id,

            "lineage_positive":
                lineage_positive,

            "explicit_negative":
                explicit_negative,

            "middle_already_matched":
                middle_already_matched,

            "synthetic_error_um":
                synthetic_error_um,

            "synthetic_within_7um":
                synthetic_within_7um,

            "synthetic_positive":
                synthetic_positive,

            "synthetic_negative":
                synthetic_negative,
        })


    oracle_parts_0917.append(
        pd.DataFrame(
            result_rows
        )
    )


    if (
        i == 1
        or i % 10 == 0
        or i == len(
            dataset_names
        )
    ):

        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


oracle_synthetic_0917 = (
    pd.concat(
        oracle_parts_0917,
        ignore_index=True,
    )
)

assert (
    len(
        oracle_synthetic_0917
    )
    == 1067
)


# ============================================================
# Predeclared occupancy scopes
#
# These use prediction-only nearest-existing-node distance.
# No threshold is selected here.
# ============================================================

SCOPES_0917 = {
    "STRONG_ALL":
        np.ones(
            len(
                oracle_synthetic_0917
            ),
            dtype=bool,
        ),

    "EMPTY_GT_2UM":
        (
            oracle_synthetic_0917[
                "nearest_existing_midpoint_um"
            ] > 2.0
        ).to_numpy(),

    "EMPTY_GT_4UM":
        (
            oracle_synthetic_0917[
                "nearest_existing_midpoint_um"
            ] > 4.0
        ).to_numpy(),

    "EMPTY_GT_5UM":
        (
            oracle_synthetic_0917[
                "nearest_existing_midpoint_um"
            ] > 5.0
        ).to_numpy(),
}


# ============================================================
# Scope summary
# ============================================================

def summarize_scope_0917(
    df,
    mask,
    scope,
    prefix,
):

    sub = df.loc[
        mask
    ].copy()

    if prefix != "ALL":

        sub = sub[
            sub["prefix"]
            == prefix
        ]

    n = len(sub)

    ac_obs = int(
        sub[
            "ac_observable"
        ].sum()
    )

    lineage_pos = int(
        sub[
            "lineage_positive"
        ].sum()
    )

    explicit_neg = int(
        sub[
            "explicit_negative"
        ].sum()
    )

    duplicate_middle = int(
        (
            sub[
                "lineage_positive"
            ]
            &
            sub[
                "middle_already_matched"
            ]
        ).sum()
    )

    true_missing_middle = int(
        (
            sub[
                "lineage_positive"
            ]
            &
            ~sub[
                "middle_already_matched"
            ]
        ).sum()
    )

    synth_pos = int(
        sub[
            "synthetic_positive"
        ].sum()
    )

    synth_neg = int(
        sub[
            "synthetic_negative"
        ].sum()
    )

    resolved = (
        synth_pos
        + synth_neg
    )


    missing_errors = sub.loc[
        (
            sub[
                "lineage_positive"
            ]
            &
            ~sub[
                "middle_already_matched"
            ]
        ),
        "synthetic_error_um",
    ].dropna()


    context_obs = int(
        sub[
            "full_context_observable"
        ].sum()
    )

    context_true = int(
        (
            sub[
                "full_context_observable"
            ]
            &
            sub[
                "context_true"
            ]
        ).sum()
    )


    return {
        "scope":
            scope,

        "prefix":
            prefix,

        "candidates":
            n,

        "AC_observable":
            ac_obs,

        "lineage_positive":
            lineage_pos,

        "explicit_negative":
            explicit_neg,

        "lineage_resolved_precision":
            (
                lineage_pos
                / (
                    lineage_pos
                    + explicit_neg
                )
                if (
                    lineage_pos
                    + explicit_neg
                ) > 0
                else np.nan
            ),

        "positive_middle_already_matched":
            duplicate_middle,

        "positive_middle_missing":
            true_missing_middle,

        "synthetic_positive":
            synth_pos,

        "synthetic_negative":
            synth_neg,

        "synthetic_resolved_precision":
            (
                synth_pos
                / resolved
                if resolved > 0
                else np.nan
            ),

        "median_missing_B_error_um":
            (
                float(
                    missing_errors.median()
                )
                if len(
                    missing_errors
                )
                else np.nan
            ),

        "missing_B_within_7_rate":
            (
                float(
                    (
                        missing_errors
                        <= 7.0
                    ).mean()
                )
                if len(
                    missing_errors
                )
                else np.nan
            ),

        "full_context_observable":
            context_obs,

        "full_context_true":
            context_true,

        "context_precision":
            (
                context_true
                / context_obs
                if context_obs
                else np.nan
            ),
    }


summary_rows_0917 = []

for (
    scope,
    mask
) in SCOPES_0917.items():

    for prefix in [
        "ALL",
        "44b6",
        "6bba",
    ]:

        summary_rows_0917.append(
            summarize_scope_0917(
                oracle_synthetic_0917,
                mask,
                scope,
                prefix,
            )
        )


oracle_summary_0917 = (
    pd.DataFrame(
        summary_rows_0917
    )
)


# ============================================================
# Positive missing-middle position-error quantiles
# ============================================================

position_rows_0917 = []

for scope, mask in (
    SCOPES_0917.items()
):

    sub = (
        oracle_synthetic_0917.loc[
            mask
        ]
        .copy()
    )

    sub = sub[
        sub[
            "lineage_positive"
        ]
        &
        ~sub[
            "middle_already_matched"
        ]
    ]

    for prefix in [
        "ALL",
        "44b6",
        "6bba",
    ]:

        q = (
            sub
            if prefix == "ALL"
            else sub[
                sub["prefix"]
                == prefix
            ]
        )

        errors = (
            q[
                "synthetic_error_um"
            ]
            .dropna()
        )

        if not len(errors):
            continue

        position_rows_0917.append({
            "scope":
                scope,

            "prefix":
                prefix,

            "n_missing_middle":
                len(errors),

            "p25_error_um":
                float(
                    errors.quantile(
                        0.25
                    )
                ),

            "median_error_um":
                float(
                    errors.median()
                ),

            "p75_error_um":
                float(
                    errors.quantile(
                        0.75
                    )
                ),

            "p90_error_um":
                float(
                    errors.quantile(
                        0.90
                    )
                ),

            "within_4um_rate":
                float(
                    (
                        errors <= 4.0
                    ).mean()
                ),

            "within_7um_rate":
                float(
                    (
                        errors <= 7.0
                    ).mean()
                ),
        })


position_error_0917 = (
    pd.DataFrame(
        position_rows_0917
    )
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.17 RESULT")
print("=" * 72)

print()
print("SYNTHETIC ORACLE SUMMARY")

display(
    oracle_summary_0917
)

print()
print(
    "POSITIVE MISSING-B POSITION ERROR"
)

display(
    position_error_0917
)

print()
print("GLOBAL COUNTS")

print(
    "Strong candidates:",
    len(
        oracle_synthetic_0917
    ),
)

print(
    "A/C observable:",
    int(
        oracle_synthetic_0917[
            "ac_observable"
        ].sum()
    ),
)

print(
    "Positive GT A->B->C paths:",
    int(
        oracle_synthetic_0917[
            "lineage_positive"
        ].sum()
    ),
)

print(
    "Explicit GT negatives:",
    int(
        oracle_synthetic_0917[
            "explicit_negative"
        ].sum()
    ),
)

print(
    "Positive paths where B already has matched prediction:",
    int(
        (
            oracle_synthetic_0917[
                "lineage_positive"
            ]
            &
            oracle_synthetic_0917[
                "middle_already_matched"
            ]
        ).sum()
    ),
)

print(
    "Oracle-supported synthetic positives:",
    int(
        oracle_synthetic_0917[
            "synthetic_positive"
        ].sum()
    ),
)

print()
print(
    "No synthetic node was created."
)

print(
    "No edge was added."
)

print(
    "No graph/file was modified."
)

print(
    "GT was used only for oracle analysis."
)

print()
print(
    "09.17 synthetic missing-node oracle audit: COMPLETE"
)

In [ ]:
# ============================================================
# PATCH — 09.18 Restore Missing nearest_existing_midpoint_id
# Purpose:
#   Restore the prediction-only nearest-existing-node ID that
#   was present in 09.16 but accidentally omitted when 09.17
#   constructed candidates_0917.
#
#   This is a schema repair only.
#   It does NOT recompute candidates or alter oracle results.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: NO — DELETE after 09.18 succeeds
# ============================================================

import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "strong_geometry_0916" in globals()
assert "oracle_synthetic_0917" in globals()

assert (
    "nearest_existing_midpoint_id"
    in strong_geometry_0916.columns
)

assert (
    "nearest_existing_midpoint_id"
    not in oracle_synthetic_0917.columns
)


# ============================================================
# Exact candidate identity
# ============================================================

MERGE_KEYS_0918_PATCH = [
    "dataset",
    "p_id",
    "a_id",
    "c_id",
    "d_id",
    "t_missing",
]


source_0918_patch = (
    strong_geometry_0916[
        MERGE_KEYS_0918_PATCH
        + [
            "nearest_existing_midpoint_id",
        ]
    ]
    .copy()
)


# Every Stage-09.17 candidate should map to exactly one
# Stage-09.16 geometry candidate.
assert not source_0918_patch.duplicated(
    MERGE_KEYS_0918_PATCH
).any(), (
    "09.16 candidate identity is not unique."
)


before_n = len(
    oracle_synthetic_0917
)


oracle_synthetic_0917 = (
    oracle_synthetic_0917
    .merge(
        source_0918_patch,
        on=MERGE_KEYS_0918_PATCH,
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# Verification
# ============================================================

assert len(
    oracle_synthetic_0917
) == before_n

assert (
    "nearest_existing_midpoint_id"
    in oracle_synthetic_0917.columns
)

assert oracle_synthetic_0917[
    "nearest_existing_midpoint_id"
].notna().all()

oracle_synthetic_0917[
    "nearest_existing_midpoint_id"
] = (
    oracle_synthetic_0917[
        "nearest_existing_midpoint_id"
    ]
    .astype(int)
)


print(
    "=== 09.18 PATCH RESULT ==="
)

print(
    "Rows preserved:",
    len(
        oracle_synthetic_0917
    ),
)

print(
    "nearest_existing_midpoint_id restored:",
    True,
)

print(
    "Missing restored IDs:",
    int(
        oracle_synthetic_0917[
            "nearest_existing_midpoint_id"
        ]
        .isna()
        .sum()
    ),
)

print(
    "09.18 schema patch: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.18 Existing-Middle Positive Topology Audit
# Purpose:
#   Inspect the oracle-positive synthetic-gap cases where the
#   true GT middle node B* is ALREADY matched by an existing
#   prediction node.
#
#   Goal:
#       determine why baseline-v9 did not already recover
#
#           A -> B -> C
#
#   despite B being detected.
#
#   This audit classifies the matched prediction B by:
#       - current in/out degree
#       - whether A->B or B->C already exists
#       - current predecessors / successors
#       - distance to motion-consensus midpoint
#       - A-B / B-C physical distances
#       - component relationship
#       - high-confidence detection membership
#
#   GT is used ONLY to identify the 9 known positive cases and
#   locate the prediction node matched to B*.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — oracle diagnostic only
# Keep after running: YES — Stage-09 reassignment evidence
# ============================================================

import numpy as np
import pandas as pd

from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
)


# ============================================================
# Preconditions
# ============================================================

assert "oracle_synthetic_0917" in globals()
assert "prepare_v9_precleanup_0916" in globals()
assert "WARM2_09975" in globals()
assert "node_coordinate_keys" in globals()
assert "component_sets" in globals()
assert "load_graph" in globals()
assert "TRAIN_DIR" in globals()


# ============================================================
# Select only oracle-positive cases where B* is already
# represented by an existing prediction node.
# ============================================================

duplicate_positive_0918 = (
    oracle_synthetic_0917[
        oracle_synthetic_0917[
            "lineage_positive"
        ]
        &
        oracle_synthetic_0917[
            "middle_already_matched"
        ]
    ]
    .copy()
)

assert len(
    duplicate_positive_0918
) == 9


# ============================================================
# Helper
# ============================================================

def topology_label_0918(
    indeg,
    outdeg,
):

    if indeg == 0 and outdeg == 0:
        return "ISOLATED"

    if indeg == 0 and outdeg == 1:
        return "TRACK_START"

    if indeg == 1 and outdeg == 0:
        return "TRACK_END"

    if indeg == 1 and outdeg == 1:
        return "INTERNAL"

    if outdeg > 1:
        return "BRANCH"

    if indeg > 1:
        return "MERGE"

    return f"OTHER_{indeg}_{outdeg}"


# ============================================================
# Audit
# ============================================================

audit_rows_0918 = []

print(
    "=== 09.18 EXISTING-MIDDLE TOPOLOGY AUDIT ==="
)


for name in (
    duplicate_positive_0918[
        "dataset"
    ].unique()
):

    case_df = (
        duplicate_positive_0918[
            duplicate_positive_0918[
                "dataset"
            ] == name
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Frozen baseline-v9 pre-cleanup graph
    # --------------------------------------------------------

    pred_g, scale = (
        prepare_v9_precleanup_0916(
            name
        )
    )

    scale = np.asarray(
        scale,
        dtype=float,
    )


    # --------------------------------------------------------
    # Fresh GT + official matching
    # --------------------------------------------------------

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )

    official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )


    # --------------------------------------------------------
    # Prediction node table
    # --------------------------------------------------------

    pred_nodes = (
        pred_g
        .node_attrs(
            unpack=True
        )
        .to_pandas()
        .set_index(
            "node_id"
        )
    )


    # --------------------------------------------------------
    # GT -> prediction inverse match map
    # --------------------------------------------------------

    matched_rows = (
        pred_nodes[
            pred_nodes[
                "match_node_id"
            ] >= 0
        ]
        .reset_index()
    )

    gt_to_pred = dict(
        zip(
            matched_rows[
                "match_node_id"
            ].astype(int),
            matched_rows[
                "node_id"
            ].astype(int),
        )
    )


    # --------------------------------------------------------
    # High-confidence node keys
    # --------------------------------------------------------

    high_g = load_graph(
        WARM2_09975
        / f"{name}.geff"
    )

    high_keys = (
        node_coordinate_keys(
            high_g
        )
    )


    # --------------------------------------------------------
    # Component lookup
    # --------------------------------------------------------

    component_lookup = {}

    comps = component_sets(
        pred_g
    )

    for comp_idx, comp in enumerate(
        comps
    ):

        comp = list(
            map(
                int,
                comp,
            )
        )

        comp_size = len(
            comp
        )

        for node_id in comp:

            component_lookup[
                node_id
            ] = (
                comp_idx,
                comp_size,
            )


    # --------------------------------------------------------
    # Candidate cases
    # --------------------------------------------------------

    for row in (
        case_df.itertuples(
            index=False
        )
    ):

        a_id = int(
            row.a_id
        )

        c_id = int(
            row.c_id
        )

        gt_b_id = int(
            row.gt_middle_id
        )


        assert (
            gt_b_id
            in gt_to_pred
        )

        b_id = int(
            gt_to_pred[
                gt_b_id
            ]
        )


        # ====================================================
        # Coordinates
        # ====================================================

        a = pred_nodes.loc[
            a_id
        ]

        b = pred_nodes.loc[
            b_id
        ]

        c = pred_nodes.loc[
            c_id
        ]


        a_um = np.array(
            [
                float(a["z"]) * scale[0],
                float(a["y"]) * scale[1],
                float(a["x"]) * scale[2],
            ]
        )

        b_um = np.array(
            [
                float(b["z"]) * scale[0],
                float(b["y"]) * scale[1],
                float(b["x"]) * scale[2],
            ]
        )

        c_um = np.array(
            [
                float(c["z"]) * scale[0],
                float(c["y"]) * scale[1],
                float(c["x"]) * scale[2],
            ]
        )

        consensus_um = np.array(
            [
                float(
                    row.b_z_um
                ),
                float(
                    row.b_y_um
                ),
                float(
                    row.b_x_um
                ),
            ]
        )


        dist_consensus_b = float(
            np.linalg.norm(
                b_um
                - consensus_um
            )
        )

        dist_ab = float(
            np.linalg.norm(
                b_um
                - a_um
            )
        )

        dist_bc = float(
            np.linalg.norm(
                c_um
                - b_um
            )
        )


        # ====================================================
        # Current topology
        # ====================================================

        a_in = int(
            pred_g.in_degree(
                a_id
            )
        )

        a_out = int(
            pred_g.out_degree(
                a_id
            )
        )

        b_in = int(
            pred_g.in_degree(
                b_id
            )
        )

        b_out = int(
            pred_g.out_degree(
                b_id
            )
        )

        c_in = int(
            pred_g.in_degree(
                c_id
            )
        )

        c_out = int(
            pred_g.out_degree(
                c_id
            )
        )


        has_ab = bool(
            pred_g.has_edge(
                a_id,
                b_id,
            )
        )

        has_bc = bool(
            pred_g.has_edge(
                b_id,
                c_id,
            )
        )


        preds_b = list(
            map(
                int,
                pred_g.predecessors(
                    b_id
                ),
            )
        )

        succs_b = list(
            map(
                int,
                pred_g.successors(
                    b_id
                ),
            )
        )


        # ====================================================
        # Component information
        # ====================================================

        a_comp, a_size = (
            component_lookup[
                a_id
            ]
        )

        b_comp, b_size = (
            component_lookup[
                b_id
            ]
        )

        c_comp, c_size = (
            component_lookup[
                c_id
            ]
        )


        # ====================================================
        # High-confidence membership
        #
        # Reuse exact coordinate-key convention from Stage07/08.
        # ====================================================

        b_key = (
            int(
                b["t"]
            ),
            float(
                b["z"]
            ),
            float(
                b["y"]
            ),
            float(
                b["x"]
            ),
        )

        b_high_conf = (
            b_key
            in high_keys
        )


        # ====================================================
        # Reassignment-relevant state
        # ====================================================

        free_for_ab = (
            a_out == 0
            and b_in == 0
        )

        free_for_bc = (
            b_out == 0
            and c_in == 0
        )

        direct_bridge_possible = (
            free_for_ab
            and free_for_bc
        )


        audit_rows_0918.append({
            "dataset":
                name,

            "prefix":
                row.prefix,

            "a_id":
                a_id,

            "b_pred_id":
                b_id,

            "c_id":
                c_id,

            "gt_b_id":
                gt_b_id,

            "t_missing":
                int(
                    row.t_missing
                ),

            "b_topology":
                topology_label_0918(
                    b_in,
                    b_out,
                ),

            "A_degree":
                f"{a_in}/{a_out}",

            "B_degree":
                f"{b_in}/{b_out}",

            "C_degree":
                f"{c_in}/{c_out}",

            "has_A_to_B":
                has_ab,

            "has_B_to_C":
                has_bc,

            "B_predecessors":
                tuple(
                    preds_b
                ),

            "B_successors":
                tuple(
                    succs_b
                ),

            "free_for_A_to_B":
                free_for_ab,

            "free_for_B_to_C":
                free_for_bc,

            "direct_bridge_possible":
                direct_bridge_possible,

            "B_high_conf":
                b_high_conf,

            "consensus_to_B_um":
                dist_consensus_b,

            "A_to_B_um":
                dist_ab,

            "B_to_C_um":
                dist_bc,

            "A_component_size":
                a_size,

            "B_component_size":
                b_size,

            "C_component_size":
                c_size,

            "A_B_same_component":
                a_comp == b_comp,

            "B_C_same_component":
                b_comp == c_comp,

            "A_C_same_component":
                a_comp == c_comp,

            "nearest_existing_midpoint_um":
                float(
                    row.nearest_existing_midpoint_um
                ),

            "nearest_existing_is_B":
                (
                    int(
                        row.nearest_existing_midpoint_id
                    )
                    == b_id
                ),
        })


existing_middle_audit_0918 = (
    pd.DataFrame(
        audit_rows_0918
    )
)

assert len(
    existing_middle_audit_0918
) == 9


# ============================================================
# Topology summary
# ============================================================

topology_summary_0918 = (
    existing_middle_audit_0918
    .groupby(
        "b_topology",
        as_index=False,
    )
    .agg(
        cases=(
            "b_pred_id",
            "size",
        ),

        high_conf=(
            "B_high_conf",
            "sum",
        ),

        already_has_AB=(
            "has_A_to_B",
            "sum",
        ),

        already_has_BC=(
            "has_B_to_C",
            "sum",
        ),

        direct_bridge_possible=(
            "direct_bridge_possible",
            "sum",
        ),

        median_consensus_to_B_um=(
            "consensus_to_B_um",
            "median",
        ),

        median_AB_um=(
            "A_to_B_um",
            "median",
        ),

        median_BC_um=(
            "B_to_C_um",
            "median",
        ),
    )
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.18 RESULT")
print("=" * 72)

print()
print("MATCHED-MIDDLE TOPOLOGY SUMMARY")

display(
    topology_summary_0918
)

print()
print("CASE DETAILS")

display(
    existing_middle_audit_0918[
        [
            "dataset",
            "prefix",
            "a_id",
            "b_pred_id",
            "c_id",
            "b_topology",
            "A_degree",
            "B_degree",
            "C_degree",
            "has_A_to_B",
            "has_B_to_C",
            "free_for_A_to_B",
            "free_for_B_to_C",
            "direct_bridge_possible",
            "B_high_conf",
            "consensus_to_B_um",
            "A_to_B_um",
            "B_to_C_um",
            "B_component_size",
            "nearest_existing_midpoint_um",
            "nearest_existing_is_B",
        ]
    ]
)

print()
print("GLOBAL COUNTS")

print(
    "Cases:",
    len(
        existing_middle_audit_0918
    ),
)

print(
    "B high-confidence:",
    int(
        existing_middle_audit_0918[
            "B_high_conf"
        ].sum()
    ),
)

print(
    "Already has A->B:",
    int(
        existing_middle_audit_0918[
            "has_A_to_B"
        ].sum()
    ),
)

print(
    "Already has B->C:",
    int(
        existing_middle_audit_0918[
            "has_B_to_C"
        ].sum()
    ),
)

print(
    "Direct A->B->C insertion possible:",
    int(
        existing_middle_audit_0918[
            "direct_bridge_possible"
        ].sum()
    ),
)

print(
    "Nearest midpoint detection is true B:",
    int(
        existing_middle_audit_0918[
            "nearest_existing_is_B"
        ].sum()
    ),
)

print()
print(
    "No graph/file was modified."
)

print(
    "GT was used only for oracle diagnosis."
)

print()
print(
    "09.18 existing-middle topology audit: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.19 Isolated True-Middle Miss Attribution Audit
# Purpose:
#   Explain why the 5 oracle-confirmed cases with:
#
#       A(t) -> B(t+1) -> C(t+2)
#
#   where B is:
#       - already detected
#       - high-confidence
#       - isolated
#       - structurally free for A->B->C
#
#   were nevertheless NOT recovered by the frozen strict
#   two-edge bridge policy.
#
#   For each true A-B-C triplet, reconstruct the exact Stage-09.10
#   local A/C candidate pool around B and measure:
#
#       - whether true A and C were eligible
#       - true pair rank by CV residual
#       - which pair was selected as best
#       - true/best geometry
#       - ambiguity
#       - pair_count
#       - conservative-gate status
#       - conflict-free status
#       - strict-policy membership
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — only to select the 5 known true cases;
#          all candidate/ranking measurements are prediction-only
# Keep after running: YES — Stage-09 candidate-generation evidence
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "existing_middle_audit_0918" in globals()
assert "isolated_bridge_candidates_0910" in globals()
assert "conservative_bridge_0910" in globals()
assert "strict_two_policy_0915" in globals()

assert "prepare_precleanup_graph" in globals()
assert "turn_angle_deg_0910" in globals()
assert "step_balance_0910" in globals()


# ============================================================
# Select the five safe-but-missed true bridges
# ============================================================

truth_cases_0919 = (
    existing_middle_audit_0918[
        (
            existing_middle_audit_0918[
                "b_topology"
            ] == "ISOLATED"
        )
        &
        (
            existing_middle_audit_0918[
                "direct_bridge_possible"
            ]
        )
    ]
    .copy()
)

assert len(truth_cases_0919) == 5


# ============================================================
# Reconstruct conflict-free status of the r=4 conservative pool
# ============================================================

cons_0919 = (
    conservative_bridge_0910
    .copy()
)

for col in [
    "a_id",
    "b_id",
    "c_id",
]:
    cons_0919[col] = (
        cons_0919[col]
        .astype(int)
    )


a_usage_0919 = (
    cons_0919
    .groupby(
        [
            "dataset",
            "a_id",
        ]
    )["b_id"]
    .transform("nunique")
)

c_usage_0919 = (
    cons_0919
    .groupby(
        [
            "dataset",
            "c_id",
        ]
    )["b_id"]
    .transform("nunique")
)

cons_0919[
    "conflict_free"
] = (
    (a_usage_0919 == 1)
    &
    (c_usage_0919 == 1)
)


conflict_map_0919 = {
    (
        str(row.dataset),
        int(row.a_id),
        int(row.b_id),
        int(row.c_id),
    ): bool(row.conflict_free)

    for row in cons_0919.itertuples(
        index=False
    )
}


strict_keys_0919 = {
    (
        str(row.dataset),
        int(row.a_id),
        int(row.b_id),
        int(row.c_id),
    )
    for row in strict_two_policy_0915.itertuples(
        index=False
    )
}


# ============================================================
# Exact local-pair reconstruction
# ============================================================

def audit_true_bridge_0919(
    case,
):

    name = str(
        case.dataset
    )

    true_a_id = int(
        case.a_id
    )

    b_id = int(
        case.b_pred_id
    )

    true_c_id = int(
        case.c_id
    )


    # --------------------------------------------------------
    # Exact 09.10 state:
    # G2 + K9, BEFORE Stage-09 gap repairs / cleanup.
    # --------------------------------------------------------

    g, scale = (
        prepare_precleanup_graph(
            name
        )
    )

    scale = np.asarray(
        scale,
        dtype=float,
    )


    nodes = (
        g.node_attrs(
            unpack=True
        )
        .to_pandas()
        .set_index("node_id")
    )


    assert true_a_id in nodes.index
    assert b_id in nodes.index
    assert true_c_id in nodes.index


    def pos_um(node_id):

        r = nodes.loc[
            node_id
        ]

        return np.array(
            [
                float(r["z"]) * scale[0],
                float(r["y"]) * scale[1],
                float(r["x"]) * scale[2],
            ],
            dtype=float,
        )


    a_true = pos_um(
        true_a_id
    )

    b = pos_um(
        b_id
    )

    c_true = pos_um(
        true_c_id
    )


    tb = int(
        nodes.loc[
            b_id,
            "t",
        ]
    )


    # --------------------------------------------------------
    # B must really have been isolated in the Stage-09.10 state
    # --------------------------------------------------------

    b_in_pre = int(
        g.in_degree(
            b_id
        )
    )

    b_out_pre = int(
        g.out_degree(
            b_id
        )
    )


    # --------------------------------------------------------
    # Eligible A endpoints at t-1:
    # out_degree == 0 and within 8 um of B
    # --------------------------------------------------------

    frame_a = (
        nodes[
            nodes["t"]
            == tb - 1
        ]
        .index
        .astype(int)
        .tolist()
    )

    eligible_a = []

    for a_id in frame_a:

        if (
            int(
                g.out_degree(
                    a_id
                )
            )
            != 0
        ):
            continue

        d = float(
            np.linalg.norm(
                pos_um(a_id)
                - b
            )
        )

        if d <= 8.0:

            eligible_a.append(
                a_id
            )


    # --------------------------------------------------------
    # Eligible C endpoints at t+1:
    # in_degree == 0 and within 8 um of B
    # --------------------------------------------------------

    frame_c = (
        nodes[
            nodes["t"]
            == tb + 1
        ]
        .index
        .astype(int)
        .tolist()
    )

    eligible_c = []

    for c_id in frame_c:

        if (
            int(
                g.in_degree(
                    c_id
                )
            )
            != 0
        ):
            continue

        d = float(
            np.linalg.norm(
                pos_um(c_id)
                - b
            )
        )

        if d <= 8.0:

            eligible_c.append(
                c_id
            )


    true_a_eligible = (
        true_a_id
        in eligible_a
    )

    true_c_eligible = (
        true_c_id
        in eligible_c
    )


    # --------------------------------------------------------
    # Enumerate exactly the A/C pair pool used conceptually
    # by 09.10.
    # --------------------------------------------------------

    pair_rows = []

    for a_id in eligible_a:

        a = pos_um(
            a_id
        )

        d_ab = float(
            np.linalg.norm(
                b - a
            )
        )

        for c_id in eligible_c:

            c = pos_um(
                c_id
            )

            d_bc = float(
                np.linalg.norm(
                    c - b
                )
            )

            residual = float(
                np.linalg.norm(
                    a + c - 2.0 * b
                )
            )

            max_step = max(
                d_ab,
                d_bc,
            )

            balance = (
                step_balance_0910(
                    d_ab,
                    d_bc,
                )
            )

            angle = (
                turn_angle_deg_0910(
                    a,
                    b,
                    c,
                )
            )

            pair_rows.append({
                "a_id":
                    int(a_id),

                "c_id":
                    int(c_id),

                "residual_um":
                    residual,

                "d_ab_um":
                    d_ab,

                "d_bc_um":
                    d_bc,

                "max_step_um":
                    max_step,

                "step_balance":
                    balance,

                "turn_angle_deg":
                    angle,
            })


    pairs = pd.DataFrame(
        pair_rows
    )

    assert len(pairs) > 0


    pairs = (
        pairs
        .sort_values(
            [
                "residual_um",
                "max_step_um",
                "a_id",
                "c_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    pairs[
        "rank"
    ] = (
        np.arange(
            len(pairs)
        )
        + 1
    )


    best = pairs.iloc[
        0
    ]


    second_residual = (
        float(
            pairs.iloc[
                1
            ][
                "residual_um"
            ]
        )
        if len(pairs) > 1
        else np.inf
    )


    best_ambiguity = (
        second_residual
        - float(
            best[
                "residual_um"
            ]
        )
    )


    # --------------------------------------------------------
    # Locate TRUE A/C pair in pool
    # --------------------------------------------------------

    true_pair = pairs[
        (
            pairs[
                "a_id"
            ] == true_a_id
        )
        &
        (
            pairs[
                "c_id"
            ] == true_c_id
        )
    ]


    true_pair_present = (
        len(true_pair)
        == 1
    )


    if true_pair_present:

        true_row = (
            true_pair.iloc[
                0
            ]
        )

        true_rank = int(
            true_row[
                "rank"
            ]
        )

        true_residual = float(
            true_row[
                "residual_um"
            ]
        )

        true_max_step = float(
            true_row[
                "max_step_um"
            ]
        )

        true_balance = float(
            true_row[
                "step_balance"
            ]
        )

        true_angle = float(
            true_row[
                "turn_angle_deg"
            ]
        )

    else:

        true_rank = -1
        true_residual = np.nan
        true_max_step = np.nan
        true_balance = np.nan
        true_angle = np.nan


    # --------------------------------------------------------
    # Reproduction check against retained 09.10 best row for B
    # --------------------------------------------------------

    retained = (
        isolated_bridge_candidates_0910[
            (
                isolated_bridge_candidates_0910[
                    "dataset"
                ] == name
            )
            &
            (
                isolated_bridge_candidates_0910[
                    "b_id"
                ].astype(int)
                == b_id
            )
        ]
    )


    retained_present = (
        len(retained)
        == 1
    )


    if retained_present:

        retained_row = (
            retained.iloc[
                0
            ]
        )

        retained_matches_reconstruction = (
            int(
                retained_row[
                    "a_id"
                ]
            )
            == int(
                best[
                    "a_id"
                ]
            )
            and
            int(
                retained_row[
                    "c_id"
                ]
            )
            == int(
                best[
                    "c_id"
                ]
            )
            and
            np.isclose(
                float(
                    retained_row[
                        "cv_residual_um"
                    ]
                ),
                float(
                    best[
                        "residual_um"
                    ]
                ),
                atol=1e-8,
            )
        )

    else:

        retained_matches_reconstruction = False


    # --------------------------------------------------------
    # Frozen conservative gates apply to SELECTED BEST pair.
    # --------------------------------------------------------

    best_residual = float(
        best[
            "residual_um"
        ]
    )

    best_max_step = float(
        best[
            "max_step_um"
        ]
    )

    best_balance = float(
        best[
            "step_balance"
        ]
    )

    best_angle = float(
        best[
            "turn_angle_deg"
        ]
    )


    pass_residual = (
        best_residual
        <= 4.0
    )

    pass_step = (
        best_max_step
        <= 8.0
    )

    pass_balance = (
        best_balance
        <= 0.50
    )

    pass_angle = (
        best_angle
        <= 60.0
    )

    pass_ambiguity = (
        best_ambiguity
        >= 1.0
    )

    selected_conservative = (
        pass_residual
        and pass_step
        and pass_balance
        and pass_angle
        and pass_ambiguity
    )


    best_key = (
        name,
        int(
            best[
                "a_id"
            ]
        ),
        b_id,
        int(
            best[
                "c_id"
            ]
        ),
    )


    conflict_free = (
        conflict_map_0919.get(
            best_key,
            False,
        )
        if selected_conservative
        else False
    )

    unique_pair = (
        len(pairs)
        == 1
    )

    strict_member = (
        best_key
        in strict_keys_0919
    )


    # --------------------------------------------------------
    # Primary miss attribution
    # --------------------------------------------------------

    reasons = []

    if not true_a_eligible:
        reasons.append(
            "TRUE_A_NOT_ELIGIBLE"
        )

    if not true_c_eligible:
        reasons.append(
            "TRUE_C_NOT_ELIGIBLE"
        )

    if true_pair_present and true_rank != 1:
        reasons.append(
            "TRUE_PAIR_NOT_BEST"
        )

    if not pass_residual:
        reasons.append(
            "BEST_RESIDUAL_GT_4"
        )

    if not pass_step:
        reasons.append(
            "BEST_MAX_STEP_GT_8"
        )

    if not pass_balance:
        reasons.append(
            "BEST_BALANCE_GT_0.5"
        )

    if not pass_angle:
        reasons.append(
            "BEST_ANGLE_GT_60"
        )

    if not pass_ambiguity:
        reasons.append(
            "BEST_AMBIGUITY_LT_1"
        )

    if (
        selected_conservative
        and not conflict_free
    ):
        reasons.append(
            "ENDPOINT_CONFLICT"
        )

    if (
        selected_conservative
        and conflict_free
        and not unique_pair
    ):
        reasons.append(
            "PAIR_COUNT_GT_1"
        )

    if strict_member:
        reasons.append(
            "UNEXPECTED_STRICT_MEMBER"
        )

    if not reasons:
        reasons.append(
            "UNATTRIBUTED"
        )


    return {
        "dataset":
            name,

        "prefix":
            case.prefix,

        "true_a_id":
            true_a_id,

        "b_id":
            b_id,

        "true_c_id":
            true_c_id,

        "B_pre_degree":
            f"{b_in_pre}/{b_out_pre}",

        "eligible_A_count":
            len(
                eligible_a
            ),

        "eligible_C_count":
            len(
                eligible_c
            ),

        "pair_count":
            len(
                pairs
            ),

        "true_A_eligible":
            true_a_eligible,

        "true_C_eligible":
            true_c_eligible,

        "true_pair_present":
            true_pair_present,

        "true_pair_rank":
            true_rank,

        "true_residual_um":
            true_residual,

        "true_max_step_um":
            true_max_step,

        "true_balance":
            true_balance,

        "true_angle_deg":
            true_angle,

        "best_a_id":
            int(
                best[
                    "a_id"
                ]
            ),

        "best_c_id":
            int(
                best[
                    "c_id"
                ]
            ),

        "best_is_true_pair":
            (
                int(
                    best[
                        "a_id"
                    ]
                ) == true_a_id
                and
                int(
                    best[
                        "c_id"
                    ]
                ) == true_c_id
            ),

        "best_residual_um":
            best_residual,

        "best_max_step_um":
            best_max_step,

        "best_balance":
            best_balance,

        "best_angle_deg":
            best_angle,

        "best_ambiguity_um":
            best_ambiguity,

        "pass_residual":
            pass_residual,

        "pass_step":
            pass_step,

        "pass_balance":
            pass_balance,

        "pass_angle":
            pass_angle,

        "pass_ambiguity":
            pass_ambiguity,

        "selected_conservative":
            selected_conservative,

        "conflict_free":
            conflict_free,

        "unique_pair":
            unique_pair,

        "strict_policy_member":
            strict_member,

        "retained_0910_present":
            retained_present,

        "retained_matches_reconstruction":
            retained_matches_reconstruction,

        "miss_reason":
            " | ".join(
                reasons
            ),
    }


# ============================================================
# Full five-case audit
# ============================================================

rows_0919 = []

print(
    "=== 09.19 ISOLATED TRUE-MIDDLE MISS ATTRIBUTION ==="
)

for case in (
    truth_cases_0919
    .itertuples(
        index=False
    )
):

    rows_0919.append(
        audit_true_bridge_0919(
            case
        )
    )


miss_audit_0919 = (
    pd.DataFrame(
        rows_0919
    )
)


# ============================================================
# Basic integrity
# ============================================================

assert len(
    miss_audit_0919
) == 5

assert (
    miss_audit_0919[
        "B_pre_degree"
    ] == "0/0"
).all()

assert (
    miss_audit_0919[
        "retained_matches_reconstruction"
    ]
).all()


# ============================================================
# Miss-reason summary
# ============================================================

reason_summary_0919 = (
    miss_audit_0919[
        "miss_reason"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "miss_reason"
    )
    .reset_index(
        name="cases"
    )
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.19 RESULT")
print("=" * 72)

print()
print("MISS REASON SUMMARY")

display(
    reason_summary_0919
)

print()
print("FIVE TRUE ISOLATED-B CASES")

display(
    miss_audit_0919[
        [
            "dataset",
            "prefix",

            "true_a_id",
            "b_id",
            "true_c_id",

            "eligible_A_count",
            "eligible_C_count",
            "pair_count",

            "true_pair_rank",
            "best_is_true_pair",

            "true_residual_um",
            "best_residual_um",

            "true_max_step_um",
            "true_balance",
            "true_angle_deg",

            "best_ambiguity_um",

            "pass_residual",
            "pass_balance",
            "pass_angle",
            "pass_ambiguity",

            "conflict_free",
            "unique_pair",
            "strict_policy_member",

            "miss_reason",
        ]
    ]
)

print()
print("GLOBAL COUNTS")

print(
    "True isolated-B cases:",
    len(
        miss_audit_0919
    ),
)

print(
    "True pair selected as best:",
    int(
        miss_audit_0919[
            "best_is_true_pair"
        ].sum()
    ),
)

print(
    "Pass conservative geometry:",
    int(
        miss_audit_0919[
            "selected_conservative"
        ].sum()
    ),
)

print(
    "Conflict-free:",
    int(
        miss_audit_0919[
            "conflict_free"
        ].sum()
    ),
)

print(
    "Unique local pair:",
    int(
        miss_audit_0919[
            "unique_pair"
        ].sum()
    ),
)

print(
    "Already in strict policy:",
    int(
        miss_audit_0919[
            "strict_policy_member"
        ].sum()
    ),
)

print()
print(
    "No graph/file was modified."
)

print(
    "GT was used only to identify the five known true cases."
)

print()
print(
    "09.19 isolated true-middle miss attribution: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.20 Strict Two-Edge Relaxed-Geometry Oracle Audit
# Purpose:
#   Determine whether the successful strict existing-node
#   two-edge bridge policy is unnecessarily restrictive in:
#
#       turn angle <= 60 deg
#       step balance <= 0.50
#
#   Predeclared prediction-only variants:
#
#       CURRENT:
#           angle <= 60, balance <= .50
#
#       ANGLE90:
#           angle <= 90, balance <= .50
#
#       BALANCE075:
#           angle <= 60, balance <= .75
#
#       BOTH_RELAXED:
#           angle <= 90, balance <= .75
#
#   Frozen for ALL variants:
#       residual <= 4 um
#       max step <= 8 um
#       ambiguity >= 1 um
#       pair_count == 1
#       endpoint conflict-free
#
#   GT is used only AFTER prediction-only policies are built,
#   to measure observable bridge precision.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 oracle analysis only
# Keep after running: YES — Stage-09 geometry evidence
# ============================================================

import numpy as np
import pandas as pd

from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
)


# ============================================================
# Preconditions
# ============================================================

assert "isolated_bridge_candidates_0910" in globals()
assert "prepare_precleanup_graph" in globals()
assert "truth_cases_0919" in globals()

assert "dataset_names" in globals()
assert "prefix_map" in globals()
assert "TRAIN_DIR" in globals()
assert "load_graph" in globals()


# ============================================================
# Predeclared variants
# ============================================================

VARIANTS_0920 = {
    "CURRENT_60_050": {
        "angle_max": 60.0,
        "balance_max": 0.50,
    },

    "ANGLE90_050": {
        "angle_max": 90.0,
        "balance_max": 0.50,
    },

    "ANGLE60_075": {
        "angle_max": 60.0,
        "balance_max": 0.75,
    },

    "ANGLE90_075": {
        "angle_max": 90.0,
        "balance_max": 0.75,
    },
}


# ============================================================
# Prediction-only policy builder
# ============================================================

def build_policy_0920(
    angle_max,
    balance_max,
):

    p = (
        isolated_bridge_candidates_0910[
            [
                "dataset",
                "prefix",
                "a_id",
                "b_id",
                "c_id",

                "cv_residual_um",
                "max_step_um",
                "step_balance",
                "turn_angle_deg",
                "ambiguity_margin_um",
                "pair_count",
            ]
        ]
        .copy()
    )

    for col in [
        "a_id",
        "b_id",
        "c_id",
    ]:
        p[col] = p[col].astype(int)


    # --------------------------------------------------------
    # Geometry first
    # --------------------------------------------------------

    p = p[
        (
            p["cv_residual_um"]
            <= 4.0
        )
        &
        (
            p["max_step_um"]
            <= 8.0
        )
        &
        (
            p["step_balance"]
            <= balance_max
        )
        &
        (
            p["turn_angle_deg"]
            <= angle_max
        )
        &
        (
            p["ambiguity_margin_um"]
            >= 1.0
        )
    ].copy()


    # --------------------------------------------------------
    # Recompute conflicts INSIDE each policy
    # --------------------------------------------------------

    a_usage = (
        p
        .groupby(
            [
                "dataset",
                "a_id",
            ]
        )["b_id"]
        .transform("nunique")
    )

    c_usage = (
        p
        .groupby(
            [
                "dataset",
                "c_id",
            ]
        )["b_id"]
        .transform("nunique")
    )

    p["conflict_free"] = (
        (a_usage == 1)
        &
        (c_usage == 1)
    )


    # --------------------------------------------------------
    # Keep strict unique-pair requirement
    # --------------------------------------------------------

    p = p[
        p["conflict_free"]
        &
        (
            p["pair_count"]
            == 1
        )
    ].copy()

    return p


# ============================================================
# Build all policies BEFORE any GT usage
# ============================================================

policies_0920 = {}

for policy_name, cfg in (
    VARIANTS_0920.items()
):

    policies_0920[
        policy_name
    ] = build_policy_0920(
        cfg["angle_max"],
        cfg["balance_max"],
    )


# Current policy must reproduce Stage 09.14.
assert (
    len(
        policies_0920[
            "CURRENT_60_050"
        ]
    )
    == 2289
), (
    "Current strict policy did not reproduce 2289 candidates."
)


# ============================================================
# Policy candidate counts
# ============================================================

policy_count_rows = []

for policy_name, p in (
    policies_0920.items()
):

    policy_count_rows.append({
        "policy":
            policy_name,

        "ALL":
            len(p),

        "44b6":
            int(
                (
                    p["prefix"]
                    == "44b6"
                ).sum()
            ),

        "6bba":
            int(
                (
                    p["prefix"]
                    == "6bba"
                ).sum()
            ),
    })


policy_counts_0920 = (
    pd.DataFrame(
        policy_count_rows
    )
)


# ============================================================
# Create union so official matching runs only once per dataset
# ============================================================

union_parts = []

for policy_name, p in (
    policies_0920.items()
):

    q = p[
        [
            "dataset",
            "prefix",
            "a_id",
            "b_id",
            "c_id",
        ]
    ].copy()

    q["policy"] = (
        policy_name
    )

    union_parts.append(q)


policy_membership_0920 = (
    pd.concat(
        union_parts,
        ignore_index=True,
    )
)


union_candidates_0920 = (
    policy_membership_0920[
        [
            "dataset",
            "prefix",
            "a_id",
            "b_id",
            "c_id",
        ]
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


# ============================================================
# Official observable-truth audit
# ============================================================

oracle_parts = []

print(
    "=== 09.20 RELAXED TWO-EDGE GEOMETRY ORACLE AUDIT ==="
)


for i, name in enumerate(
    dataset_names,
    start=1,
):

    sub = (
        union_candidates_0920[
            union_candidates_0920[
                "dataset"
            ] == name
        ]
        .copy()
    )

    if not len(sub):
        continue


    # --------------------------------------------------------
    # Exact candidate-generation state:
    # G2 + K9, before gap repair / cleanup
    # --------------------------------------------------------

    pred_g, scale = (
        prepare_precleanup_graph(
            name
        )
    )

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )


    # --------------------------------------------------------
    # Locked official matching
    # --------------------------------------------------------

    official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(
            np.asarray(
                scale,
                dtype=float,
            )
        ),
        max_distance=7.0,
    )


    matches = (
        pred_g
        .node_attrs(
            unpack=True
        )
        .to_pandas()
    )

    match_map = dict(
        zip(
            matches[
                "node_id"
            ].astype(int),
            matches[
                "match_node_id"
            ].astype(int),
        )
    )


    # --------------------------------------------------------
    # Prediction -> GT
    # --------------------------------------------------------

    for role in [
        "a",
        "b",
        "c",
    ]:

        sub[
            f"gt_{role}_id"
        ] = [
            int(
                match_map.get(
                    int(node_id),
                    -1,
                )
            )
            for node_id in sub[
                f"{role}_id"
            ]
        ]


    sub[
        "bridge_observable"
    ] = (
        (sub["gt_a_id"] >= 0)
        &
        (sub["gt_b_id"] >= 0)
        &
        (sub["gt_c_id"] >= 0)
    )


    # --------------------------------------------------------
    # Full GT A* -> B* -> C* truth
    # --------------------------------------------------------

    bridge_true = []

    left_true = []
    right_true = []

    for row in sub.itertuples(
        index=False
    ):

        if not row.bridge_observable:

            left_true.append(
                False
            )

            right_true.append(
                False
            )

            bridge_true.append(
                False
            )

            continue


        left = bool(
            gt_g.has_edge(
                int(row.gt_a_id),
                int(row.gt_b_id),
            )
        )

        right = bool(
            gt_g.has_edge(
                int(row.gt_b_id),
                int(row.gt_c_id),
            )
        )

        left_true.append(
            left
        )

        right_true.append(
            right
        )

        bridge_true.append(
            left and right
        )


    sub[
        "left_true"
    ] = left_true

    sub[
        "right_true"
    ] = right_true

    sub[
        "bridge_true"
    ] = bridge_true


    oracle_parts.append(
        sub
    )


    if (
        i == 1
        or i % 10 == 0
        or i == len(dataset_names)
    ):

        print(
            f"Processed {i:02d}/"
            f"{len(dataset_names)}"
        )


oracle_union_0920 = (
    pd.concat(
        oracle_parts,
        ignore_index=True,
    )
)


# ============================================================
# Attach truth back to every policy
# ============================================================

TRUTH_KEYS = [
    "dataset",
    "prefix",
    "a_id",
    "b_id",
    "c_id",
]


truth_table_0920 = (
    oracle_union_0920[
        TRUTH_KEYS
        + [
            "bridge_observable",
            "left_true",
            "right_true",
            "bridge_true",
        ]
    ]
    .drop_duplicates(
        TRUTH_KEYS
    )
)


policy_oracle_0920 = (
    policy_membership_0920
    .merge(
        truth_table_0920,
        on=TRUTH_KEYS,
        how="left",
        validate="many_to_one",
    )
)

assert policy_oracle_0920[
    "bridge_observable"
].notna().all()


# ============================================================
# Oracle summary
# ============================================================

summary_rows = []

for policy_name in (
    VARIANTS_0920.keys()
):

    for prefix in [
        "ALL",
        "44b6",
        "6bba",
    ]:

        sub = (
            policy_oracle_0920[
                policy_oracle_0920[
                    "policy"
                ] == policy_name
            ]
        )

        if prefix != "ALL":

            sub = sub[
                sub["prefix"]
                == prefix
            ]


        observable = sub[
            sub[
                "bridge_observable"
            ]
        ]

        n_obs = len(
            observable
        )

        n_true = int(
            observable[
                "bridge_true"
            ].sum()
        )

        n_false = (
            n_obs
            - n_true
        )


        summary_rows.append({
            "policy":
                policy_name,

            "prefix":
                prefix,

            "candidates":
                len(sub),

            "observable":
                n_obs,

            "true_bridges":
                n_true,

            "false_bridges":
                n_false,

            "observable_precision":
                (
                    n_true
                    / n_obs
                    if n_obs
                    else np.nan
                ),

            "left_true_rate":
                (
                    float(
                        observable[
                            "left_true"
                        ].mean()
                    )
                    if n_obs
                    else np.nan
                ),

            "right_true_rate":
                (
                    float(
                        observable[
                            "right_true"
                        ].mean()
                    )
                    if n_obs
                    else np.nan
                ),
        })


oracle_summary_0920 = (
    pd.DataFrame(
        summary_rows
    )
)


# ============================================================
# Incremental candidates relative to CURRENT
#
# Critical question:
# Are the NEW candidates introduced by relaxation still clean?
# ============================================================

current_keys = {
    (
        str(row.dataset),
        int(row.a_id),
        int(row.b_id),
        int(row.c_id),
    )
    for row in policies_0920[
        "CURRENT_60_050"
    ].itertuples(
        index=False
    )
}


incremental_rows = []

for policy_name in [
    "ANGLE90_050",
    "ANGLE60_075",
    "ANGLE90_075",
]:

    sub = (
        policy_oracle_0920[
            policy_oracle_0920[
                "policy"
            ] == policy_name
        ]
        .copy()
    )

    sub[
        "is_new_vs_current"
    ] = [
        (
            str(row.dataset),
            int(row.a_id),
            int(row.b_id),
            int(row.c_id),
        )
        not in current_keys

        for row in sub.itertuples(
            index=False
        )
    ]

    new = sub[
        sub[
            "is_new_vs_current"
        ]
    ]

    for prefix in [
        "ALL",
        "44b6",
        "6bba",
    ]:

        q = (
            new
            if prefix == "ALL"
            else new[
                new["prefix"]
                == prefix
            ]
        )

        obs = q[
            q[
                "bridge_observable"
            ]
        ]

        n_obs = len(
            obs
        )

        n_true = int(
            obs[
                "bridge_true"
            ].sum()
        )

        incremental_rows.append({
            "policy":
                policy_name,

            "prefix":
                prefix,

            "new_candidates":
                len(q),

            "new_observable":
                n_obs,

            "new_true":
                n_true,

            "new_false":
                n_obs
                - n_true,

            "new_observable_precision":
                (
                    n_true
                    / n_obs
                    if n_obs
                    else np.nan
                ),
        })


incremental_oracle_0920 = (
    pd.DataFrame(
        incremental_rows
    )
)


# ============================================================
# Coverage of the five known isolated true-middle cases
#
# Diagnostic only.
# Threshold decisions MUST NOT be made from these five alone.
# ============================================================

true_case_keys = {
    (
        str(row.dataset),
        int(row.a_id),
        int(row.b_pred_id),
        int(row.c_id),
    )
    for row in truth_cases_0919.itertuples(
        index=False
    )
}


coverage_rows = []

for policy_name, p in (
    policies_0920.items()
):

    keys = {
        (
            str(row.dataset),
            int(row.a_id),
            int(row.b_id),
            int(row.c_id),
        )
        for row in p.itertuples(
            index=False
        )
    }

    covered = (
        true_case_keys
        & keys
    )

    coverage_rows.append({
        "policy":
            policy_name,

        "known_true_cases_covered":
            len(covered),

        "out_of":
            len(
                true_case_keys
            ),
    })


known_true_coverage_0920 = (
    pd.DataFrame(
        coverage_rows
    )
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.20 RESULT")
print("=" * 72)

print()
print("POLICY CANDIDATE COUNTS")

display(
    policy_counts_0920
)

print()
print("FULL POLICY ORACLE SUMMARY")

display(
    oracle_summary_0920
)

print()
print(
    "INCREMENTAL CANDIDATES VS CURRENT"
)

display(
    incremental_oracle_0920
)

print()
print(
    "KNOWN FIVE TRUE-CASE COVERAGE "
    "(diagnostic only)"
)

display(
    known_true_coverage_0920
)

print()
print(
    "No edge was added."
)

print(
    "No graph/file was modified."
)

print(
    "GT was used only after prediction-only policies were frozen."
)

print()
print(
    "09.20 relaxed-geometry oracle audit: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 09.21 Relaxed Two-Edge Geometry Metric Ablation
# Purpose:
#   Evaluate the four PREDECLARED prediction-only strict
#   two-edge bridge policies from 09.20 on the locked Fold0
#   competition metric.
#
# Policies:
#
#   CURRENT_60_050
#       angle <= 60 deg
#       balance <= .50
#
#   ANGLE90_050
#       angle <= 90 deg
#       balance <= .50
#
#   ANGLE60_075
#       angle <= 60 deg
#       balance <= .75
#
#   ANGLE90_075
#       angle <= 90 deg
#       balance <= .75
#
# Frozen for ALL policies:
#   residual <= 4 um
#   max step <= 8 um
#   ambiguity >= 1 um
#   pair_count == 1
#   endpoint conflict-free
#
# Full pipeline:
#   Warm2
#   -> G2 p2=.80
#   -> K9
#   -> frozen single-edge repair r<=4 um
#   -> selected strict two-edge policy
#   -> frozen cleanup
#   -> locked official metric
#
# Changes files: NO — in-memory graphs only
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only; policies use NO GT
# Keep after running: YES — formal Stage-09 policy selection
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Preconditions
# ============================================================

assert "policies_0920" in globals()
assert "single_policy_0915" in globals()

assert "prepare_precleanup_graph" in globals()
assert "add_available_repairs_0907" in globals()
assert "add_two_edge_bridges_0912" in globals()
assert "apply_frozen_component_cleanup" in globals()
assert "evaluate_graph_0906" in globals()

assert "baseline_v8_records_0912" in globals()

EXPECTED_CURRENT_SCORE_0921 = 0.770921


# ============================================================
# Policy order
# ============================================================

POLICY_ORDER_0921 = [
    "CURRENT_60_050",
    "ANGLE90_050",
    "ANGLE60_075",
    "ANGLE90_075",
]


assert set(
    POLICY_ORDER_0921
) == set(
    policies_0920.keys()
)


# ============================================================
# Evaluate every policy
# ============================================================

records_by_policy_0921 = {}

overall_rows_0921 = []
prefix_rows_0921 = []
paired_rows_0921 = []
count_rows_0921 = []


print(
    "=== 09.21 RELAXED TWO-EDGE GEOMETRY METRIC ABLATION ==="
)


for policy_name in POLICY_ORDER_0921:

    policy = (
        policies_0920[
            policy_name
        ]
        .copy()
    )

    records = []
    dataset_rows = []

    bridges_requested_total = 0
    bridges_added_total = 0
    associations_added_total = 0

    degree_conflict_total = 0
    existing_conflict_total = 0
    missing_endpoint_total = 0

    cleanup_removed_total = 0


    # ========================================================
    # Fold0
    # ========================================================

    for i, name in enumerate(
        dataset_names
    ):

        # ----------------------------------------------------
        # Fresh G2 + K9 graph
        # ----------------------------------------------------

        g, scale = (
            prepare_precleanup_graph(
                name
            )
        )


        # ----------------------------------------------------
        # Frozen single-edge recovery FIRST
        # ----------------------------------------------------

        single_candidates = (
            single_policy_0915[
                single_policy_0915[
                    "dataset"
                ] == name
            ]
        )

        single_stats = (
            add_available_repairs_0907(
                g,
                single_candidates,
            )
        )

        assert (
            single_stats["added"]
            == len(single_candidates)
        )


        # ----------------------------------------------------
        # Selected two-edge policy
        # ----------------------------------------------------

        two_candidates = (
            policy[
                policy[
                    "dataset"
                ] == name
            ]
        )

        two_stats = (
            add_two_edge_bridges_0912(
                g,
                two_candidates,
            )
        )


        bridges_requested_total += (
            two_stats[
                "requested_bridges"
            ]
        )

        bridges_added_total += (
            two_stats[
                "added_bridges"
            ]
        )

        associations_added_total += (
            two_stats[
                "edges_added"
            ]
        )

        degree_conflict_total += (
            two_stats[
                "degree_conflict"
            ]
        )

        existing_conflict_total += (
            two_stats[
                "existing_edge_conflict"
            ]
        )

        missing_endpoint_total += (
            two_stats[
                "missing_endpoint"
            ]
        )


        # ----------------------------------------------------
        # Frozen cleanup
        # ----------------------------------------------------

        cleanup = (
            apply_frozen_component_cleanup(
                g,
                scale,
            )
        )

        cleanup_removed_total += int(
            cleanup[
                "nodes_removed"
            ]
        )


        # ----------------------------------------------------
        # Locked metric
        # ----------------------------------------------------

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )

        m = evaluate_graph_0906(
            g,
            gt_g,
            scale,
            name,
        )

        records.append(
            m
        )


        # ----------------------------------------------------
        # Paired evidence vs baseline-v8
        # ----------------------------------------------------

        base_m = (
            baseline_v8_records_0912[
                i
            ]
        )

        dataset_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "policy":
                policy_name,

            "bridges_requested":
                two_stats[
                    "requested_bridges"
                ],

            "bridges_added":
                two_stats[
                    "added_bridges"
                ],

            "delta_edge_jaccard":
                (
                    m[
                        "edge_jaccard"
                    ]
                    - base_m[
                        "edge_jaccard"
                    ]
                ),

            "delta_adj_edge_jaccard":
                (
                    m[
                        "adj_edge_jaccard"
                    ]
                    - base_m[
                        "adj_edge_jaccard"
                    ]
                ),

            "delta_recall":
                (
                    m[
                        "node_recall"
                    ]
                    - base_m[
                        "node_recall"
                    ]
                ),

            "delta_node_ratio":
                (
                    m[
                        "total_node_ratio"
                    ]
                    - base_m[
                        "total_node_ratio"
                    ]
                ),
        })


    # ========================================================
    # Aggregate
    # ========================================================

    records_by_policy_0921[
        policy_name
    ] = records

    s = summarise(
        records
    )

    df = pd.DataFrame(
        dataset_rows
    )


    overall_rows_0921.append({
        "policy":
            policy_name,

        "policy_candidates":
            len(policy),

        "bridges_added":
            bridges_added_total,

        "associations_added":
            associations_added_total,

        "edge_jaccard":
            s[
                "edge_jaccard"
            ],

        "adj_edge_jaccard":
            s[
                "adj_edge_jaccard"
            ],

        "division_jaccard":
            s[
                "division_jaccard"
            ],

        "node_recall":
            s[
                "node_recall"
            ],

        "median_node_ratio":
            float(
                np.median([
                    m[
                        "total_node_ratio"
                    ]
                    for m in records
                ])
            ),

        "score":
            s[
                "score"
            ],

        "nodes_rescued_vs_v8":
            (
                84803
                - cleanup_removed_total
            ),
    })


    # ========================================================
    # Prefix
    # ========================================================

    for prefix in [
        "44b6",
        "6bba",
    ]:

        idx = [
            j
            for j, name in enumerate(
                dataset_names
            )
            if prefix_map[name]
            == prefix
        ]

        recs = [
            records[j]
            for j in idx
        ]

        ps = summarise(
            recs
        )

        sub = df[
            df[
                "prefix"
            ] == prefix
        ]


        prefix_rows_0921.append({
            "policy":
                policy_name,

            "prefix":
                prefix,

            "policy_candidates":
                int(
                    (
                        policy[
                            "prefix"
                        ] == prefix
                    ).sum()
                ),

            "bridges_added":
                int(
                    sub[
                        "bridges_added"
                    ].sum()
                ),

            "edge_jaccard":
                ps[
                    "edge_jaccard"
                ],

            "adj_edge_jaccard":
                ps[
                    "adj_edge_jaccard"
                ],

            "node_recall":
                ps[
                    "node_recall"
                ],

            "score":
                ps[
                    "score"
                ],

            "edgeJ_improved_rate":
                float(
                    (
                        sub[
                            "delta_edge_jaccard"
                        ] > 0
                    ).mean()
                ),

            "adjEdgeJ_improved_rate":
                float(
                    (
                        sub[
                            "delta_adj_edge_jaccard"
                        ] > 0
                    ).mean()
                ),

            "mean_delta_edgeJ":
                float(
                    sub[
                        "delta_edge_jaccard"
                    ].mean()
                ),

            "median_delta_edgeJ":
                float(
                    sub[
                        "delta_edge_jaccard"
                    ].median()
                ),

            "mean_delta_adjEdgeJ":
                float(
                    sub[
                        "delta_adj_edge_jaccard"
                    ].mean()
                ),

            "median_delta_adjEdgeJ":
                float(
                    sub[
                        "delta_adj_edge_jaccard"
                    ].median()
                ),

            "mean_delta_recall":
                float(
                    sub[
                        "delta_recall"
                    ].mean()
                ),
        })


    # ========================================================
    # ALL paired robustness
    # ========================================================

    paired_rows_0921.append({
        "policy":
            policy_name,

        "datasets":
            len(df),

        "edgeJ_improved_rate":
            float(
                (
                    df[
                        "delta_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "adjEdgeJ_improved_rate":
            float(
                (
                    df[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "mean_delta_edgeJ":
            float(
                df[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                df[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_adjEdgeJ":
            float(
                df[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                df[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                df[
                    "delta_recall"
                ].mean()
            ),

        "mean_delta_node_ratio":
            float(
                df[
                    "delta_node_ratio"
                ].mean()
            ),
    })


    # ========================================================
    # Interaction counts
    # ========================================================

    count_rows_0921.append({
        "policy":
            policy_name,

        "requested":
            bridges_requested_total,

        "added":
            bridges_added_total,

        "degree_conflict":
            degree_conflict_total,

        "existing_edge_conflict":
            existing_conflict_total,

        "missing_endpoint":
            missing_endpoint_total,

        "cleanup_nodes_removed":
            cleanup_removed_total,
    })


    print(
        f"{policy_name}"
        f" | candidates={len(policy)}"
        f" | added={bridges_added_total}"
        f" | score={s['score']:.6f}"
    )


# ============================================================
# Tables
# ============================================================

overall_0921 = (
    pd.DataFrame(
        overall_rows_0921
    )
)

prefix_0921 = (
    pd.DataFrame(
        prefix_rows_0921
    )
)

paired_0921 = (
    pd.DataFrame(
        paired_rows_0921
    )
)

counts_0921 = (
    pd.DataFrame(
        count_rows_0921
    )
)


# ============================================================
# Current-policy reproduction gate
# ============================================================

current_row = (
    overall_0921[
        overall_0921[
            "policy"
        ] == "CURRENT_60_050"
    ]
    .iloc[0]
)

assert abs(
    float(
        current_row[
            "score"
        ]
    )
    - EXPECTED_CURRENT_SCORE_0921
) < 5e-6, (
    "Current baseline-v9 reproduction failed."
)


# ============================================================
# Delta vs CURRENT
# ============================================================

current_score = float(
    current_row[
        "score"
    ]
)

current_edge = float(
    current_row[
        "edge_jaccard"
    ]
)

current_adj = float(
    current_row[
        "adj_edge_jaccard"
    ]
)


overall_0921[
    "delta_score_vs_current"
] = (
    overall_0921[
        "score"
    ]
    - current_score
)

overall_0921[
    "delta_edgeJ_vs_current"
] = (
    overall_0921[
        "edge_jaccard"
    ]
    - current_edge
)

overall_0921[
    "delta_adjEdgeJ_vs_current"
] = (
    overall_0921[
        "adj_edge_jaccard"
    ]
    - current_adj
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("09.21 RESULT")
print("=" * 72)

print()
print("OVERALL POLICY COMPARISON")

display(
    overall_0921[
        [
            "policy",
            "policy_candidates",
            "bridges_added",
            "associations_added",

            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "score",

            "delta_edgeJ_vs_current",
            "delta_adjEdgeJ_vs_current",
            "delta_score_vs_current",

            "nodes_rescued_vs_v8",
        ]
    ]
)

print()
print("BY PREFIX / ROBUSTNESS")

display(
    prefix_0921
)

print()
print("ALL-DATASET PAIRED ROBUSTNESS")

display(
    paired_0921
)

print()
print("INTERACTION COUNTS")

display(
    counts_0921
)


best = (
    overall_0921
    .sort_values(
        "score",
        ascending=False,
    )
    .iloc[0]
)

print()
print(
    "Best policy:",
    best["policy"],
)

print(
    "Best score:",
    f"{best['score']:.6f}",
)

print(
    "Delta vs current baseline-v9:",
    f"{best['delta_score_vs_current']:+.6f}",
)

print()
print(
    "No GEFF file was modified."
)

print(
    "All candidate policies were frozen before GT evaluation."
)

print()
print(
    "09.21 relaxed two-edge metric ablation: COMPLETE"
)

## Stage 09 Final Decision — baseline-v9 Frozen

Existing-node gap recovery is complete.

### Frozen repair order
G2 → K9 → single-edge recovery → strict two-edge recovery → cleanup.

### Single-edge recovery
- mutual support
- residual ≤ 4.0 µm
- max step ≤ 8.0 µm
- step balance ≤ 0.50
- turn angle ≤ 60°
- ambiguity margin ≥ 1.0 µm

### Strict two-edge recovery
- existing isolated middle node
- residual ≤ 4.0 µm
- max step ≤ 8.0 µm
- step balance ≤ 0.50
- turn angle ≤ 60°
- ambiguity margin ≥ 1.0 µm
- pair_count == 1
- endpoint conflict-free

### Rejected alternatives
- residual 5 µm single-edge: micro-gain only
- reverse repair ordering: no improvement
- relaxed angle ≤ 90°: worse
- relaxed balance ≤ 0.75: +0.000055 Fold0 only, weaker paired robustness
- synthetic-node insertion: insufficient evidence; most observable middle cells already have predictions

### Frozen Fold0 score
**0.770921**